# Generador de datos de Chulupi Lang

Este notebook fabrica **pares de oraciones** para pre-entrenar un traductor español–aymara.

Cada par tiene tres versiones de la misma idea:

| Versión | Qué es | Ejemplo |
|---|---|---|
| **Modo B** | Oración ficticia con sufijos y palabras gramaticales **reales** del aymara | `Nayaxa tisqthwa.` |
| **Modo A** | La misma oración con sufijos y palabras gramaticales **inventados** | `Muqixu tisqphyi.` |
| **Español** | La traducción | "Yo voy." |

Las raíces (la parte de la palabra que lleva el significado, como `tisqa` "ir") son inventadas en los dos modos.

**La regla de oro.** La especificación `chulupi_lang_spec.md` (versión 1.1.2) manda. Cada regla tiene un código `CH-xxx`. El código de este notebook cita esos códigos en comentarios, para que puedas buscar la regla en la especificación.

### Palabras que vas a ver

| Término | Qué significa |
|---|---|
| **raíz** | Parte de la palabra con el significado: `tisqa` "ir", `chimula` "casa". |
| **sufijo** | Trozo que se pega al final de la raíz y añade información: `-naka` = plural. |
| **morfema** | Cualquier pieza con significado propio: una raíz o un sufijo. |
| **id de morfema** | Nombre corto de un sufijo en la especificación: `PL` (plural), `P1` (1.ª persona), `AFF` (afirmativo). La sección 3 tiene la tabla completa. |
| **grafema** | Una "letra" del alfabeto, aunque se escriba con varios caracteres: `ch'` es un solo grafema. |
| **glosa** | Explicación pieza a pieza de una palabra: `tisq-th-wa` = ir-1>3.SIMP-AFF. |
| **idea** | Descripción en datos (JSON) de lo que dice la oración: verbo, sujeto, tiempo… |
| **par** | Una idea con sus tres versiones: B, A y español. |

### Cómo está organizado

| Sección | Qué hace |
|---|---|
| 0 | Introducción y parámetros |
| 1 | Lee la configuración de la especificación |
| 2 | Alfabeto y reglas de sonidos (fonotáctica) |
| 3 | Motor que construye palabras |
| 4 | Pruebas de referencia del motor |
| 5 | Léxico: lee las palabras del diccionario y revisa su formato |
| 6 | Raíces inventadas: las fabrica y las filtra contra palabras reales del aymara |
| 7 | Español: conjugación, artículos y la oración española |
| 8 | Oraciones aymaras (modos B y A) y los 40 casos de prueba |
| 9 | Ideas: revisarlas (R1–R22) y sortearlas con los pesos |
| 10 | Generación: todas las pruebas, las ideas, la cobertura mínima y los pares |
| 11 | Verificación (§13), archivos de salida e informe |
| 12 | Cómo usar los resultados (y qué hacer si algo falla) |

**Para usarlo:** cambia los valores de la celda "Parámetros" y luego usa *Entorno de ejecución → Ejecutar todo* (Colab) o *Run All* (VS Code).

## 0. Introducción y parámetros

**Qué hace esta sección.**
1. Detecta si el notebook corre en Google Colab o en tu computadora.
2. Instala lo que falte. Solo hace falta PyYAML, que lee los bloques de configuración de la especificación.
3. Reúne en una celda todo lo que puedes cambiar (los **parámetros**).
4. Busca los archivos de datos.

**Reglas que implementa.** Ninguna regla lingüística. Prepara el entorno.

**Qué entra.** Los archivos de la carpeta `datos/`: la especificación, `regresion_ch710.json` y `casos_prueba.json` son obligatorios. `lexico_sin_raices.csv` y `palabras_reales.txt` son opcionales.

**Qué sale.** La función `ruta()`, que dice dónde está cada archivo, y la carpeta de salida `CARPETA_SALIDA`.

**Ejemplo.** Al final, la carpeta de salida tendrá pares como este:
- B: `Nayaxa tisqthwa.`
- A: `Muqixu tisqphyi.`
- Español: "Yo voy."

In [ ]:
# 0.1 · Entorno: ¿Colab o local? ¿Falta algún paquete?
import sys
import importlib.util

EN_COLAB = "google.colab" in sys.modules   # Colab carga este módulo al arrancar

# Colab ya trae pandas, matplotlib y PyYAML. En una computadora pueden faltar.
# PyYAML lee los bloques ```yaml de la especificación (sección 1).
PAQUETES = {"yaml": "pyyaml", "pandas": "pandas", "matplotlib": "matplotlib"}
faltan = [pip_nombre for modulo, pip_nombre in PAQUETES.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    %pip install -q {" ".join(faltan)}

print("Entorno:", "Google Colab" if EN_COLAB else "local (VS Code o Jupyter)")
print("Python", sys.version.split()[0])

### Parámetros

Esta es la **única celda que necesitas cambiar**.

| Parámetro | Qué controla | Valores |
|---|---|---|
| `N_PARES` | Cuántos pares se generan | Número entero. 100.000 tarda unos minutos (ver la sección 10). |
| `VERSION_LEXICO` | Versión del léxico. Súbela cada vez que cambie una raíz (CH-309). | Texto, por ejemplo `"1.0"` |
| `SEMILLA` | Semilla del azar. La misma semilla da siempre el mismo archivo. | Número entero |
| `MODO_FILTRO` | Cómo se filtran las raíces que se parecen a palabras reales (CH-306). Decidido: `"corpus"` (P-06) | `"guia"` o `"corpus"` |
| `GENERAR` | `True`: genera el conjunto completo (secciones 10 y 11). `False`: solo pruebas, raíces y una muestra de 30 pares | `True` o `False` |
| `CARPETA_SALIDA` | Dónde se guardan los resultados | Nombre de carpeta |
| `FUENTE_COLAB` | Solo en Colab: de dónde salen los datos | `"github"`, `"drive"` o `"subir"` |
| `REPO_GITHUB`, `RAMA_GITHUB` | Solo con `"github"`: el repositorio del equipo y la rama que se descarga | Dirección y nombre de rama |
| `CARPETA_DRIVE` | Solo en Colab con `"drive"`: tu carpeta del proyecto en Google Drive | Ruta |

**Una semilla** es un número que fija el azar. Con la misma semilla, el programa "sortea" lo mismo cada vez.

In [ ]:
# 0.2 · Parámetros: cambia aquí lo que necesites.
N_PARES = 360_000         # pares pedidos; V27 quita los duplicados (~16 %) y quedan unos 300.000 (ver sección 10)
GENERAR = True            # True: genera el conjunto (secciones 10 y 11); False: solo pruebas y muestra
VERSION_LEXICO = "1.0"    # súbela si cambia alguna raíz (CH-309); va en cada par (CH-1301)
SEMILLA = 20261126        # semilla de la generación de pares (las raíces usan otra, fija: CH-309)
PROB_RECHAZO_DV = 0.15    # DC-68 / V30: una idea con regla [DERIVADA] se descarta con esta probabilidad
MAX_PROP_POR_DV = 0.14    # DC-68 / V30: ninguna regla [DERIVADA] supera esta proporción de las ideas
MODO_FILTRO = "corpus"    # "guia" o "corpus" (CH-306); decidido "corpus" (P-06)
CARPETA_SALIDA = "salida" # carpeta de resultados (en Colab con Drive, dentro de CARPETA_DRIVE)

# Solo para Google Colab
FUENTE_COLAB = "github"   # "github": descarga el repositorio del equipo; "drive": Google Drive; "subir": eliges los archivos
REPO_GITHUB = "https://github.com/Danilepez/Synthetic_Dataset.git"   # solo con "github"
RAMA_GITHUB = "dev/Camilo"                                             # solo con "github": rama que se descarga
CARPETA_DRIVE = "/content/drive/MyDrive/chulupi"                     # solo con "drive"

# Comprobaciones: un valor mal escrito se detecta aquí y no a mitad de la generación.
if not (isinstance(N_PARES, int) and N_PARES > 0):
    raise ValueError("N_PARES debe ser un número entero mayor que 0.")
if not isinstance(SEMILLA, int):
    raise ValueError("SEMILLA debe ser un número entero.")
if not (0 <= PROB_RECHAZO_DV < 1 and 0 < MAX_PROP_POR_DV <= 1):
    raise ValueError("PROB_RECHAZO_DV va de 0 a 1 (sin el 1) y MAX_PROP_POR_DV de 0 a 1.")
if not isinstance(GENERAR, bool):
    raise ValueError("GENERAR debe ser True o False.")
if MODO_FILTRO not in ("guia", "corpus"):
    raise ValueError('MODO_FILTRO debe ser "guia" o "corpus" (CH-306).')
if not (isinstance(VERSION_LEXICO, str) and VERSION_LEXICO):
    raise ValueError("VERSION_LEXICO debe ser un texto, por ejemplo \"1.0\".")
if FUENTE_COLAB not in ("github", "drive", "subir"):
    raise ValueError('FUENTE_COLAB debe ser "github", "drive" o "subir".')
print("Parámetros correctos.")

### Archivos de datos

El notebook necesita seis archivos: la especificación, los dos archivos de casos de prueba, el léxico, la lista de palabras reales y la tabla de conjugaciones. Los busca así:

- **En Colab con `"github"`** (lo más fácil): descarga el repositorio del equipo (`REPO_GITHUB`, rama `RAMA_GITHUB`) y usa sus archivos. No hay que subir nada. Los resultados quedan en Colab y **se borran al cerrar la sesión**: descárgalos al terminar.
- **En Colab con `"drive"`:** se monta Google Drive. Colab te pedirá permiso la primera vez. Los datos se leen de `CARPETA_DRIVE` y los resultados se guardan en `CARPETA_DRIVE/salida/`, así no se pierden.
- **En Colab con `"subir"`:** aparece un botón para elegir los archivos de tu computadora. Los resultados también se borran al cerrar la sesión.
- **En local (VS Code):** se usa la carpeta del notebook.

**Dos formas de ordenar los archivos** (el notebook reconoce las dos):

| Archivo | Repositorio del equipo | Carpeta simple |
|---|---|---|
| `chulupi_lang_spec.md` | `docs/specification/` | `datos/` |
| `casos_prueba.json`, `regresion_ch710.json` | `tests/` | `datos/` |
| `lexico_sin_raices.csv`, `palabras_reales.txt`, `conjugaciones.csv` | `data/lexicon/` | `datos/` |
| Léxico con raíces (`lexico.csv` y su versión) | `data/lexicon/lexico_con_raices_vX.csv` | `salida/` |

El repositorio se reconoce porque tiene las carpetas `docs/specification/` y `data/lexicon/`.

In [ ]:
# 0.3 · Ubicar los archivos de datos y la carpeta de salida.
import subprocess
from pathlib import Path

NOMBRE_SPEC = "chulupi_lang_spec.md"
OBLIGATORIOS = [NOMBRE_SPEC, "regresion_ch710.json", "casos_prueba.json"]
OPCIONALES = ["lexico_sin_raices.csv", "palabras_reales.txt", "conjugaciones.csv", "realismo.yaml"]
# Dónde está cada archivo dentro del repositorio del equipo (Synthetic_Dataset).
EN_REPO = {NOMBRE_SPEC: f"docs/specification/{NOMBRE_SPEC}",
           "regresion_ch710.json": "tests/regresion_ch710.json", "casos_prueba.json": "tests/casos_prueba.json",
           "lexico_sin_raices.csv": "data/lexicon/lexico_sin_raices.csv",
           "palabras_reales.txt": "data/lexicon/palabras_reales.txt",
           "conjugaciones.csv": "data/lexicon/conjugaciones.csv",
           "realismo.yaml": "data/lexicon/realismo.yaml",                                  # R23 (dev/Camilo)
           "lexico.csv": f"data/lexicon/lexico_con_raices_v{VERSION_LEXICO}.csv",           # CH-309: uno por versión
           "lexico_version.json": f"data/lexicon/lexico_con_raices_v{VERSION_LEXICO}.json"}


def es_repo(carpeta):
    """Dice si una carpeta es el repositorio del equipo (tiene docs/specification y data/lexicon).

    Parámetros:
        carpeta: ruta (Path).
    Devuelve:
        True o False.
    Reglas: ninguna (preparación).
    Ejemplo:
        >>> es_repo(Path("/content/Synthetic_Dataset"))
        True
    """
    return (carpeta / "docs" / "specification").is_dir() and (carpeta / "data" / "lexicon").is_dir()


RAIZ_REPO = None
if EN_COLAB and FUENTE_COLAB == "github":
    RAIZ_REPO = Path("/content") / Path(REPO_GITHUB).stem
    if not RAIZ_REPO.exists():                        # si ya se descargó en esta sesión, se reutiliza
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", RAMA_GITHUB, REPO_GITHUB, str(RAIZ_REPO)],
                       check=True)
    CARPETA_DATOS, BASE_SALIDA = RAIZ_REPO, RAIZ_REPO
elif EN_COLAB and FUENTE_COLAB == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_SALIDA = Path(CARPETA_DRIVE)
    RAIZ_REPO = BASE_SALIDA if es_repo(BASE_SALIDA) else None
    CARPETA_DATOS = BASE_SALIDA if RAIZ_REPO else BASE_SALIDA / "datos"
elif EN_COLAB:
    from google.colab import files
    CARPETA_DATOS = Path.cwd() / "datos"
    CARPETA_DATOS.mkdir(exist_ok=True)
    print("Elige estos archivos de tu computadora:", ", ".join(OBLIGATORIOS), "y, si los tienes,", ", ".join(OPCIONALES) + ".")
    for nombre, contenido in files.upload().items():
        (CARPETA_DATOS / nombre).write_bytes(contenido)
    BASE_SALIDA = Path.cwd()
else:
    # En local, el notebook se abre desde la carpeta del proyecto o desde notebooks/ del repositorio.
    RAIZ_REPO = next((c for c in (Path.cwd(), Path.cwd().parent) if es_repo(c)), None)
    candidatas = [Path.cwd() / "datos", Path.cwd(), Path.cwd().parent / "datos"]
    CARPETA_DATOS = RAIZ_REPO or next((c for c in candidatas if (c / NOMBRE_SPEC).exists()), candidatas[0])
    BASE_SALIDA = RAIZ_REPO or CARPETA_DATOS.parent

CARPETA_SALIDA = Path(CARPETA_SALIDA) if Path(CARPETA_SALIDA).is_absolute() else BASE_SALIDA / CARPETA_SALIDA
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)


def ruta(nombre):
    """Ruta de un archivo de datos.

    Parámetros:
        nombre: nombre del archivo, por ejemplo "casos_prueba.json" o "lexico.csv".
    Devuelve:
        En el repositorio del equipo, su lugar según EN_REPO. Si no, datos/<nombre>
        (salida/<nombre> para el léxico con raíces y su versión).
    Reglas: ninguna (preparación).
    Ejemplo:
        >>> ruta("casos_prueba.json").name
        'casos_prueba.json'
    """
    if RAIZ_REPO:
        return RAIZ_REPO / EN_REPO[nombre]
    return (CARPETA_SALIDA if nombre in ("lexico.csv", "lexico_version.json") else CARPETA_DATOS) / nombre


RUTA_SPEC = ruta(NOMBRE_SPEC)
faltan = [str(ruta(n)) for n in OBLIGATORIOS if not ruta(n).exists()]
if faltan:
    raise FileNotFoundError(f"Faltan archivos: {', '.join(faltan)}. Revisa la sección 'Archivos de datos' de arriba.")

print("Organización:  ", f"repositorio del equipo ({RAIZ_REPO})" if RAIZ_REPO else f"carpeta simple ({CARPETA_DATOS})")
print("Especificación:", RUTA_SPEC)
print("Salida:        ", CARPETA_SALIDA)
for n in OPCIONALES:
    print(f"{n}: {'sí está' if ruta(n).exists() else 'no está (se usará la alternativa de la especificación)'}")

## 1. Configuración

**Qué hace esta sección.** La especificación trae su configuración en bloques de texto marcados como ` ```yaml `. YAML es un formato de texto para datos (listas y tablas). Esta sección:
1. busca todos esos bloques en la especificación;
2. los lee y los guarda en el diccionario `CONFIG`;
3. comprueba que estén todos.

Todo el código que sigue lee sus tablas de `CONFIG`, no de valores escritos a mano. Así, si el equipo cambia la especificación (por ejemplo, el orden de las palabras en la v1.2), basta con volver a ejecutar el notebook.

**Reglas que implementa.** Carga los datos de CH-101 (alfabeto), §2 (fonotáctica), §3 (generador de raíces), CH-504 y CH-505 (morfemas), §6 (slots), §8 (paradigmas), CH-901 a CH-917 (sintaxis), CH-1102 (esquema de la idea) y CH-1104 a CH-1113 (pesos).

**Qué entra.** El texto de la especificación.

**Qué sale.** `CONFIG["alfabeto"]`, `CONFIG["morfemas"]`, `CONFIG["sintaxis"]`, etc. También `ORDEN_CONSTITUYENTES`, el orden de las partes de la oración (CH-901).

**Ejemplo.** De `CONFIG["morfemas"]` sale que el sufijo de 1.ª persona `P1` es `-tha` en B y `-phi` en A. Con eso, el motor construye:
- B: `tisqthwa` · A: `tisqphyi` · Español: "voy".

In [ ]:
# 1.1 · Extraer y leer los bloques yaml de la especificación.
import re
import json
import yaml
import pandas as pd
from IPython.display import display, Markdown

TEXTO_SPEC = RUTA_SPEC.read_text(encoding="utf-8")
VERSION_SPEC = re.search(r"\*\*Versión:\*\*\s*([\d.]+)", TEXTO_SPEC).group(1)

# Bloques que el generador necesita, con la sección de la especificación de donde salen.
BLOQUES_REQUERIDOS = {
    "alfabeto": "§1", "fonotactica": "§2", "generador_raices": "§3", "morfemas": "§5",
    "slots": "§6", "paradigmas": "§8", "sintaxis": "§9", "idea.schema": "§11", "pesos": "§11",
}

# Un valor del tipo "[NP, NP] | null" quiere decir "una lista de dos NP, o nada".
RE_ALTERNATIVA = re.compile(r'^(\s*[\w.]+:\s+)([\[{"].*\|.*?)(\s+#.*)?$')


def extraer_bloques_yaml(texto):
    """Encuentra los bloques ```yaml de la especificación.

    El nombre de cada bloque sale de su primera línea, un comentario como
    "# alfabeto.yaml" o "# sintaxis (parte de slots.yaml)".

    Parámetros:
        texto: la especificación completa.
    Devuelve:
        Diccionario {nombre del bloque: texto yaml}. El nombre va sin ".yaml".
    Reglas: todas las que tienen tabla de configuración (ver la celda de arriba).
    Ejemplo:
        >>> list(extraer_bloques_yaml(TEXTO_SPEC))[:3]
        ['alfabeto', 'fonotactica', 'generador_raices']
    """
    bloques = {}
    for m in re.finditer(r"```yaml\n(.*?)```", texto, re.S):
        fuente = m.group(1)
        nombre = re.match(r"#\s*([\w.]+)", fuente.lstrip().splitlines()[0])
        if not nombre:
            raise ValueError("Un bloque yaml de la especificación no empieza con '# nombre'.")
        clave = nombre.group(1).removesuffix(".yaml")
        if clave in bloques:
            raise ValueError(f"El bloque yaml '{clave}' aparece dos veces en la especificación.")
        bloques[clave] = fuente
    return bloques


def cargar_yaml(fuente):
    """Lee un bloque yaml. Si no es yaml válido, entrecomilla las alternativas "X | Y".

    Parámetros:
        fuente: texto del bloque.
    Devuelve:
        (datos, corregido). corregido = True si hubo que entrecomillar líneas.
    Reglas: CH-1102 (su bloque usa "X | Y", que no es yaml válido; ver PREGUNTAS.md, P-02).
    Ejemplo:
        >>> cargar_yaml("od_coord: [NP, NP] | null")
        ({'od_coord': '[NP, NP] | null'}, True)
    """
    try:
        return yaml.safe_load(fuente), False
    except yaml.YAMLError:
        lineas = []
        for linea in fuente.splitlines():
            m = RE_ALTERNATIVA.match(linea)
            lineas.append(m.group(1) + json.dumps(m.group(2), ensure_ascii=False) + (m.group(3) or "") if m else linea)
        return yaml.safe_load("\n".join(lineas)), True


BLOQUES = extraer_bloques_yaml(TEXTO_SPEC)
faltan = [b for b in BLOQUES_REQUERIDOS if b not in BLOQUES]
if faltan:
    raise ValueError(f"La especificación no tiene los bloques yaml: {faltan}")

CONFIG, CORREGIDOS = {}, []
for nombre, fuente in BLOQUES.items():
    CONFIG[nombre], corregido = cargar_yaml(fuente)
    if corregido:
        CORREGIDOS.append(nombre)

# El parámetro MODO_FILTRO manda sobre el valor escrito en la especificación (CH-306).
modo_spec = CONFIG["generador_raices"]["filtro_real"]["modo"]
CONFIG["generador_raices"]["filtro_real"]["modo"] = MODO_FILTRO

# CH-901: el orden de las partes de la oración se lee de la configuración.
ORDEN_CONSTITUYENTES = CONFIG["sintaxis"]["orden_constituyentes"]

if VERSION_SPEC != "1.1.2":
    print(f"AVISO: la especificación es la versión {VERSION_SPEC}; este notebook implementa la 1.1.2.")
print(f"Especificación versión {VERSION_SPEC}: {len(BLOQUES)} bloques yaml leídos.")
if CORREGIDOS:
    print("Bloques leídos con corrección (ver PREGUNTAS.md, P-02):", ", ".join(CORREGIDOS))
print(f"Modo del filtro de raíces: {MODO_FILTRO} (la especificación dice '{modo_spec}').")
print("Orden de constituyentes (CH-901):", " – ".join(ORDEN_CONSTITUYENTES))
display(pd.DataFrame(
    [{"bloque": n, "sección": BLOQUES_REQUERIDOS.get(n, "§7 (CH-710)"),
      "contenido": ", ".join(list(CONFIG[n])[:5]) + (" …" if len(CONFIG[n]) > 5 else ""),
      "corregido": "sí" if n in CORREGIDOS else ""} for n in BLOQUES]))

## 2. Alfabeto y fonotáctica

**Qué hace esta sección.** Define las herramientas para revisar la escritura:
- `normalizar`: unifica la escritura (apóstrofo, vocales largas, minúsculas).
- `segmentar`: corta una palabra en grafemas.
- `clases`: cambia cada grafema por su clase: `C` consonante, `V` vocal corta, `L` vocal larga.
- `validar_palabra`: revisa cualquier palabra de salida.
- `validar_raiz`: revisa una raíz inventada, con reglas más estrictas.

**Términos.**
- **Fonotáctica**: las reglas sobre qué sonidos pueden ir juntos y en qué orden.
- **Sílaba**: grupo de sonidos con una vocal en el centro: `tis.qa` tiene dos.
- **Coda**: consonante que cierra una sílaba. En `tis.qa`, la `s` es coda.
- **Vocal larga**: `ä ï ü`. Se pronuncia más larga que `a i u`.
- **Consonante aspirada**: se pronuncia con un soplo de aire. Se escribe con `h`: `ph th chh kh qh`.
- **Consonante glotalizada**: se pronuncia con un pequeño golpe de garganta. Se escribe con apóstrofo: `p' t' ch' k' q'`.
- **Laringalizada**: aspirada o glotalizada.
- **NFC**: una forma estándar de guardar letras con tilde o diéresis en el ordenador. Así `ä` es siempre un solo carácter.

**Reglas que implementa.** CH-101 a CH-106 (alfabeto) y CH-201 a CH-209 (fonotáctica).

**Qué entra.** Palabras o raíces. **Qué sale.** Listas de grafemas y listas de errores (vacías si todo está bien).

**Ejemplo.** `segmentar("ch'ixrthwa")` → `ch'` `i` `x` `r` `th` `w` `a`. Las clases son `CVCCCCV`: la palabra es válida.
- B: `ch'ixrthwa` · A: `ch'ixrphyi` · Español: "veo".

In [ ]:
# 2.1 · Alfabeto (§1) y fonotáctica (§2), leídos de CONFIG.
import unicodedata

ALF = CONFIG["alfabeto"]
FON = CONFIG["fonotactica"]

GRAFEMAS = ALF["orden_segmentacion"]                     # CH-105: el más largo primero
VOCALES_CORTAS = set(ALF["vocales"])
LARGA = dict(ALF["vocales_largas"])                      # a → ä, i → ï, u → ü
VOCALES_LARGAS = set(LARGA.values())
CONSONANTES = {g for lista in ALF["consonantes"].values() for g in lista}
APOSTROFO = ALF["normalizacion"]["apostrofo_interno"]   # CH-102: U+0027
LARINGALIZADOS = set(FON["raiz"]["laringalizados"])
PUNTO_DE_ARTICULACION = ALF["punto_de_articulacion"]
RE_RAIZ = re.compile(FON["raiz"]["regex_clases"])        # CH-201
RE_PALABRA = re.compile(FON["palabra"]["regex_clases"])  # CH-202, CH-203, CH-204, CH-208

# Comprobaciones de la propia configuración.
assert set(GRAFEMAS) == VOCALES_CORTAS | VOCALES_LARGAS | CONSONANTES, "alfabeto.yaml: el orden de segmentación no cubre el alfabeto"
assert not set(ALF["prohibidos"]) & set(GRAFEMAS), "alfabeto.yaml: una letra prohibida está en el alfabeto"
assert APOSTROFO == "\u0027", "CH-102: el apóstrofo interno debe ser U+0027"


def normalizar(texto):
    """Unifica la escritura antes de comparar o validar.

    Pasa el texto a NFC, cambia los apóstrofos tipográficos por U+0027 y pasa a minúsculas.

    Parámetros:
        texto: cualquier cadena.
    Devuelve:
        La cadena normalizada.
    Reglas: CH-102 (apóstrofo), CH-103 (vocal larga en NFC), CH-106 (los validadores trabajan en minúsculas).
    Ejemplo:
        >>> normalizar("Lamch’thwa")
        "lamch'thwa"
    """
    t = unicodedata.normalize(ALF["normalizacion"]["unicode"], texto)   # CH-103: a + U+0308 → ä
    for ap in ALF["normalizacion"]["apostrofos_a_convertir"]:           # CH-102
        t = t.replace(ap, APOSTROFO)
    return t.lower()


def segmentar(palabra):
    """Corta una palabra en grafemas, tomando siempre el más largo posible.

    Parámetros:
        palabra: cadena ya normalizada (minúsculas, NFC).
    Devuelve:
        Lista de grafemas.
    Lanza:
        ValueError si un carácter no forma ningún grafema (CH-104: letras prohibidas).
    Reglas: CH-101, CH-104, CH-105.
    Ejemplo:
        >>> segmentar("ch'ixrthwa")
        ["ch'", 'i', 'x', 'r', 'th', 'w', 'a']
    """
    out, i = [], 0
    while i < len(palabra):
        for g in GRAFEMAS:                                   # CH-105: el orden ya pone primero los largos
            if palabra.startswith(g, i):
                out.append(g)
                i += len(g)
                break
        else:
            raise ValueError(f"CH-104: carácter no permitido en {palabra!r} (posición {i}: {palabra[i]!r})")
    return out


def clases(grafemas):
    """Cambia cada grafema por su clase.

    Parámetros:
        grafemas: lista que devuelve segmentar().
    Devuelve:
        Cadena con C (consonante), V (vocal corta) y L (vocal larga).
    Reglas: notación de §2.
    Ejemplo:
        >>> clases(segmentar("tisqä"))
        'CVCCL'
    """
    return "".join("V" if g in VOCALES_CORTAS else "L" if g in VOCALES_LARGAS else "C" for g in grafemas)


def validar_palabra(palabra):
    """Revisa una palabra de salida (modo B o A).

    Parámetros:
        palabra: palabra en minúsculas y sin puntuación.
    Devuelve:
        Lista de errores, cada uno con su regla. Vacía si la palabra es válida.
    Reglas: CH-101 a CH-105, CH-202 a CH-208 (CH-709: toda palabra construida pasa por aquí).
    Ejemplo:
        >>> validar_palabra("tisqthwa")
        []
        >>> validar_palabra("tisqaiwa")
        ['CH-202/203/204/208: estructura inválida CVCCVVCV']
    """
    try:
        g = segmentar(palabra)
    except ValueError as e:
        return [str(e)]
    errores = []
    if not RE_PALABRA.match(clases(g)):
        errores.append(f"CH-202/203/204/208: estructura inválida {clases(g)}")
    if g[0] in FON["palabra"]["iniciales_prohibidas"]:
        errores.append(f"CH-202: empieza por {g[0]}")
    for cadena in FON["palabra"]["cadenas_prohibidas"]:
        if cadena in palabra:
            errores.append(f"CH-207: contiene {cadena}")
    return errores


def es_coda(grafemas, i):
    """Dice si el grafema i cierra sílaba: es consonante y le sigue otra consonante.

    Parámetros:
        grafemas: lista de grafemas de una raíz.
        i: posición que se revisa.
    Devuelve:
        True si es coda.
    Reglas: CH-201 (si hay dos consonantes entre vocales, la primera es coda).
    Ejemplo:
        >>> es_coda(segmentar("tisqa"), 2)   # la s de tis.qa
        True
    """
    return (grafemas[i] in CONSONANTES and i + 1 < len(grafemas) and grafemas[i + 1] in CONSONANTES)


def validar_raiz(raiz):
    """Revisa una raíz inventada con las reglas estrictas de la raíz.

    Parámetros:
        raiz: la raíz, en minúsculas.
    Devuelve:
        Lista de errores con su regla. Vacía si la raíz es válida.
    Reglas: CH-201, CH-202, CH-205, CH-206, CH-207, CH-209.
    Ejemplo:
        >>> validar_raiz("q'aspi")
        []
        >>> validar_raiz("tissa")
        ['CH-206: coda igual al inicio siguiente (s + s)']
    """
    try:
        g = segmentar(raiz)
    except ValueError as e:
        return [str(e)]
    R = FON["raiz"]
    errores = []
    if not RE_RAIZ.match(clases(g)):
        errores.append(f"CH-201: plantilla inválida {clases(g)}")
    if g[0] in FON["palabra"]["iniciales_prohibidas"]:
        errores.append(f"CH-202: empieza por {g[0]}")
    for v in g:
        if (v in VOCALES_CORTAS or v in VOCALES_LARGAS) and v not in R["vocales_permitidas"]:
            errores.append(f"CH-209: vocal {v} no permitida en la raíz")
    for i, c in enumerate(g):
        if not es_coda(g, i):
            continue
        sig = g[i + 1]
        if c in LARINGALIZADOS:
            errores.append(f"CH-205: laringalizada {c} en coda")
        elif c not in R["codas_permitidas"]:
            errores.append(f"CH-206: {c} no puede ser coda")
        if R["prohibido_coda_igual_a_inicio"] and c == sig:
            errores.append(f"CH-206: coda igual al inicio siguiente ({c} + {sig})")
        if c == "l" and sig in R["prohibido_l_antes_de"]:
            errores.append(f"CH-207: l + {sig}")
    lar = {x for x in g if x in LARINGALIZADOS}
    if len(lar) > R["max_laringalizados_distintos"]:
        errores.append(f"CH-205: más de un laringalizado distinto {sorted(lar)}")
    if R["armonia_mismo_punto"]:
        for punto, serie in PUNTO_DE_ARTICULACION.items():
            presentes = {x for x in g if x in serie}
            if len(presentes) > 1:
                errores.append(f"CH-205: oclusivas del punto {punto} distintas {sorted(presentes)}")
    for cadena in FON["palabra"]["cadenas_prohibidas"]:
        if cadena in raiz:
            errores.append(f"CH-207: contiene {cadena}")
    return errores


print("Grafemas:", len(GRAFEMAS), "·", " ".join(GRAFEMAS))
for w in ["tisqthwa", "ch'ixrthwa", "tisqaiwa", "tishwa"]:
    print(f"{w:12} errores: {validar_palabra(w) or 'ninguno'}")

### Pruebas de la sección 2

Una **prueba** compara lo que produce el código con lo que dice la especificación. Hay al menos una por regla, con los ejemplos de la propia especificación.

Si una prueba falla, el notebook **se detiene** con un mensaje que dice qué falló. Así nadie genera datos con un error.

Usamos `unittest`, que viene con Python. Funciona igual en Colab y en local.

In [ ]:
# 2.2 · Herramienta para correr pruebas y detener el notebook si alguna falla.
import unittest


def correr_pruebas(*clases_prueba, titulo):
    """Corre clases de prueba de unittest y detiene el notebook si alguna falla.

    Parámetros:
        clases_prueba: una o varias clases unittest.TestCase.
        titulo: nombre corto que aparece en el mensaje.
    Devuelve:
        Nada. Imprime el resultado de cada prueba.
    Lanza:
        RuntimeError si alguna prueba falla. Eso detiene "Ejecutar todo".
    Reglas: ninguna (infraestructura de pruebas).
    Ejemplo:
        >>> correr_pruebas(PruebasAlfabeto, titulo="alfabeto")
    """
    cargador = unittest.defaultTestLoader
    suite = unittest.TestSuite(cargador.loadTestsFromTestCase(c) for c in clases_prueba)
    res = unittest.TextTestRunner(verbosity=2, stream=sys.stdout).run(suite)
    if not res.wasSuccessful():
        raise RuntimeError(f"Fallaron {len(res.failures) + len(res.errors)} pruebas de {titulo}. "
                           "Lee el detalle de arriba. El notebook se detiene aquí.")
    print(f"✔ {titulo}: {res.testsRun} pruebas en verde.")


def tiene_regla(errores, regla):
    """Dice si alguna línea de una lista de errores cita una regla.

    Parámetros:
        errores: lista de cadenas (de validar_palabra o validar_raiz).
        regla: código, por ejemplo "CH-206".
    Devuelve:
        True si alguna línea contiene ese código.
    Reglas: ninguna (ayuda para las pruebas).
    Ejemplo:
        >>> tiene_regla(validar_raiz("tissa"), "CH-206")
        True
    """
    return any(regla in e for e in errores)

In [ ]:
# 2.3 · Pruebas de §1 (alfabeto) y §2 (fonotáctica). Ejemplos tomados de la especificación.
class PruebasAlfabeto(unittest.TestCase):
    def test_CH101_letras_permitidas(self):
        """CH-101: tisqthwa (B) y tisqphyi (A) usan solo letras del alfabeto."""
        self.assertEqual(segmentar("tisqthwa"), ["t", "i", "s", "q", "th", "w", "a"])
        self.assertEqual(segmentar("tisqphyi"), ["t", "i", "s", "q", "ph", "y", "i"])

    def test_CH102_apostrofo(self):
        """CH-102: los apóstrofos tipográficos pasan a U+0027."""
        self.assertEqual(normalizar("lamch’thwa"), "lamch'thwa")
        self.assertEqual(normalizar("lamch’phyi"), "lamch'phyi")
        for ap in ["’", "‘", "ʼ", "´", "`"]:
            self.assertEqual(normalizar(f"q{ap}aspi"), "q'aspi")

    def test_CH103_vocal_larga_nfc(self):
        """CH-103: a + diéresis combinante (U+0308) pasa a ä precompuesta."""
        self.assertEqual(normalizar("tisqa\u0308"), "tisqä")
        self.assertEqual(segmentar(normalizar("tisqa\u0308")), ["t", "i", "s", "q", "ä"])

    def test_CH104_letras_prohibidas(self):
        """CH-104: tisqe y tishwa son inválidas; tisqhwa es válida."""
        self.assertTrue(tiene_regla(validar_palabra("tisqe"), "CH-104"))
        self.assertTrue(tiene_regla(validar_palabra("tishwa"), "CH-104"))
        self.assertEqual(validar_palabra("tisqhwa"), [])
        for w in ["tisqo", "bata", "tisq-wa", "tisq1a", "tisqá"]:
            self.assertTrue(tiene_regla(validar_palabra(w), "CH-104"), w)

    def test_CH105_segmentacion(self):
        """CH-105: el grafema más largo primero (ch' antes que ch, chh antes que ch)."""
        self.assertEqual(segmentar("ch'ixrthwa"), ["ch'", "i", "x", "r", "th", "w", "a"])
        self.assertEqual(segmentar("ch'ixrphyi"), ["ch'", "i", "x", "r", "ph", "y", "i"])
        self.assertEqual(segmentar("chhalla"), ["chh", "a", "ll", "a"])

    def test_CH106_minusculas(self):
        """CH-106: los validadores trabajan sobre minúsculas (Ä pasa a ä)."""
        self.assertEqual(normalizar("Äta"), "äta")


class PruebasFonotactica(unittest.TestCase):
    def test_CH201_plantilla_raiz(self):
        """CH-201: lamch'a válida; lamch'tra inválida (4 consonantes seguidas)."""
        self.assertEqual(validar_raiz("lamch'a"), [])
        self.assertTrue(tiene_regla(validar_raiz("lamch'tra"), "CH-201"))
        self.assertTrue(tiene_regla(validar_raiz("tisq"), "CH-201"))      # la última sílaba es CV
        self.assertTrue(tiene_regla(validar_raiz("pa"), "CH-201"))        # mínimo 2 sílabas

    def test_CH202_inicio(self):
        """CH-202: xiku, rupa y smiqa son inválidas."""
        for w in ["xiku", "rupa", "smiqa"]:
            self.assertNotEqual(validar_palabra(w), [], w)
            self.assertNotEqual(validar_raiz(w), [], w)
        self.assertTrue(tiene_regla(validar_palabra("xiku"), "CH-202"))

    def test_CH203_final(self):
        """CH-203: tisqä válida; tisqth (B) y tisqph (A) inválidas."""
        self.assertEqual(validar_palabra("tisqä"), [])
        self.assertNotEqual(validar_palabra("tisqth"), [])
        self.assertNotEqual(validar_palabra("tisqph"), [])

    def test_CH204_sin_vocales_seguidas(self):
        """CH-204: tisqaiwa (B) y tisqauyi (A) inválidas."""
        self.assertNotEqual(validar_palabra("tisqaiwa"), [])
        self.assertNotEqual(validar_palabra("tisqauyi"), [])

    def test_CH205_laringalizadas(self):
        """CH-205: q'aspi y t'ant'a válidas; q'aspha y t'ata inválidas."""
        self.assertEqual(validar_raiz("q'aspi"), [])
        self.assertEqual(validar_raiz("t'ant'a"), [])
        self.assertTrue(tiene_regla(validar_raiz("q'aspha"), "CH-205"))
        self.assertTrue(tiene_regla(validar_raiz("t'ata"), "CH-205"))
        self.assertTrue(tiene_regla(validar_raiz("lap'ta"), "CH-205"))   # laringalizada en coda

    def test_CH206_codas(self):
        """CH-206: pajlla válida; tañqa (coda ñ) y tissa (s + s) inválidas."""
        self.assertEqual(validar_raiz("pajlla"), [])
        self.assertTrue(tiene_regla(validar_raiz("tañqa"), "CH-206"))
        self.assertTrue(tiene_regla(validar_raiz("tissa"), "CH-206"))

    def test_CH207_l_mas_l(self):
        """CH-207: kalla se lee ka.lla; ninguna palabra contiene lll."""
        self.assertEqual(segmentar("kalla"), ["k", "a", "ll", "a"])
        self.assertTrue(tiene_regla(validar_palabra("kallla"), "CH-207"))
        self.assertNotEqual(validar_raiz("kallla"), [])

    def test_CH208_grupos(self):
        """CH-208: hasta 6 consonantes seguidas fuera de la raíz (jiskht'kthti); 7 no."""
        self.assertEqual(validar_palabra("tisqkiti"), [])
        self.assertEqual(validar_palabra("tisqñuji"), [])
        self.assertEqual(validar_palabra("jiskht'kthti"), [])
        self.assertNotEqual(validar_palabra("a" + "k" * 7 + "a"), [])

    def test_CH209_raiz_sin_vocal_larga(self):
        """CH-209: tisqa válida; tïsqa inválida."""
        self.assertEqual(validar_raiz("tisqa"), [])
        self.assertTrue(tiene_regla(validar_raiz("tïsqa"), "CH-209"))


correr_pruebas(PruebasAlfabeto, PruebasFonotactica, titulo="alfabeto y fonotáctica (§1, §2)")

## 3. Motor morfológico

**Qué hace esta sección.** Construye una palabra a partir de una raíz y una lista de sufijos. Es el corazón del generador.

**Términos.**
- **Elisión**: caída de una vocal. En `tisqa` + `-tha`, la `a` final de la raíz cae: `tisqtha`.
- **Alargamiento**: un sufijo puede convertir la vocal anterior en larga. Se escribe `¨`: `tisqa` + `-¨` → `tisqä`.
- **`antes`**: qué le hace un sufijo a la vocal corta de lo que tiene delante. `V` = la conserva. `C` = la borra. `—` = no decide.
- **`después`**: qué pasa con la vocal final del propio sufijo cuando le sigue un oracional. `C` = la pierde. `V` = la conserva.
- **Oracional**: sufijo que va al final de la palabra y marca el tipo de oración: `-wa` (afirmativa), `-xa` (tema), `-ti` (pregunta o negación), `-sa` (pregunta con "qué/quién/dónde"). No decide nada: decide el sufijo anterior (CH-702).
- **Alomorfo**: otra forma del mismo sufijo según el contexto. El pasado `-yä-` se escribe `-¨-` ante la 3.ª persona (CH-511).
- **Transición**: en el verbo, una sola terminación marca quién hace la acción y a quién: `1>2` = "yo a ti" ("te veo").
- **Slot**: casilla de la palabra. Cada sufijo tiene su casilla, y las casillas van en orden fijo (§6).

**Reglas que implementa.** CH-501 a CH-505 (tabla de morfemas), CH-511 a CH-513, CH-601 a CH-606 (orden de los sufijos) y CH-701 a CH-709 (algoritmo de construcción).

**Qué entra.** Una raíz (o una palabra gramatical en su forma B), una lista de ids de morfemas y el modo (`"B"` o `"A"`).

**Qué sale.** La palabra construida. Con `traza=True`, también cada paso y cada decisión.

**Ejemplo** (`tisqa` "ir" + `P1` + `AFF`):

| Paso | Modo B | Modo A |
|---|---|---|
| raíz | `tisqa` | `tisqa` |
| + P1 (antes C) | borra `a` → `tisqtha` | borra `a` → `tisqphi` |
| + AFF (decide P1, después C) | borra `a` → `tisqthwa` | borra `i` → `tisqphyi` |

Español: "voy".

### Tabla de ids de morfemas

| Id | Qué significa | B | A |
|---|---|---|---|
| `PL` | plural del nombre | `-naka` | `-sulu` |
| `POSS.1` … `POSS.4` | poseedor: mi, tu, su, nuestro (tú y yo) | `-ja -ma -pa -sa` | `-lu -qi -chu -khu` |
| `POSR` | "que tiene" (para "tener") | `-ni` | `-phu` |
| `GEN` | "de" (poseedor) | `-na` | `-mu` |
| `LOC` | "en" | `-na` | `-mu` |
| `ALL` | "a, hacia" (lugar) | `-ru` | `-llu` |
| `COMPL.RU` | "a" (persona que recibe) | `-ru` | `-llu` |
| `ABL` | "de, desde" | `-ta` | `-ku` |
| `COM` | "con" | `-mpi` | `-nku` |
| `BEN` | "para" | `-taki` | `-puli` |
| `UNTIL` | "hasta" | `-kama` | `-chiwi` |
| `CAUSE` | "por, a causa de" | `-layku` | `-tiwqa` |
| `VBLZ.KA` | convierte "en X" en "estar en X" | `-ka` | `-ñu` |
| `VBLZ.LONG` | convierte un nombre o adjetivo en "ser X" | `-¨` | `-¨` |
| `PL.V` | plural del sujeto, en el verbo | `-pxa` | `-mtu` |
| `PROG` | progresivo ("estar + -ndo") | `-ska` | `-nchi` |
| `NEG` | parte de la negación en el verbo | `-ka` | `-ñu` |
| `REM.C` | pasado visto por quien habla | `-yä-` | `-jü-` |
| `REM.L` | pasado no visto | `-tä-` | `-wü-` |
| `P1` `P2` `P3` `P4` | persona del sujeto: yo, tú, él/ella, nosotros (tú y yo) | `-tha -ta -i -tana` | `-phi -ku -u -kilu` |
| `P3.REM` | 3.ª persona en los pasados | `-na` | `-mu` |
| `T12` `T21` `T31` `T32` `T34` | transiciones sujeto>objeto (1>2 = "yo a ti") | `-sma -ista -itu -tama -istu` | `-jmi -ulqa -uqu -kiqu -ulqu` |
| `FUT.1` … `FUT.4` | futuro, por persona | `-¨ -¨ta -ni -ñäni` | `-¨ -¨ku -phu -thäqi` |
| `AFF` | afirmativo | `-wa` | `-yi` |
| `TOP` | tema de la oración (marca el sujeto) | `-xa` | `-xu` |
| `Q.YN` | pregunta de sí o no | `-ti` | `-ji` |
| `NEG.TI` | final de la negación | `-ti` | `-ji` |
| `Q.INF` | pregunta con qué, quién o dónde | `-sa` | `-khu` |

Los tiempos: **SIMP** = presente o pasado simple; **FUT** = futuro; **REM.C** y **REM.L** = pasados remotos. "4.ª persona" = "nosotros" incluyendo a quien escucha.

In [ ]:
# 3.1 · Tabla de morfemas (CH-504) y palabras gramaticales (CH-505), leídas de CONFIG.
MORFEMAS = {}
for m in CONFIG["morfemas"]["sufijos"]:
    if m["id"] in MORFEMAS:
        raise ValueError(f"morfemas.yaml: id repetido {m['id']}")
    MORFEMAS[m["id"]] = {**m, "despues": m.get("despues", "-")}   # los oracionales no traen 'despues'

GRAMATICALES = {p["id"]: p for p in CONFIG["morfemas"]["palabras_gramaticales"]}
GRAM_B_A = {p["b"]: p["a"] for p in GRAMATICALES.values()}         # CH-708: forma B → forma A
ORACIONALES = {i for i, m in MORFEMAS.items() if m["antes"] == "-" and m["slot"] in (["N7", "V6"], "N7", "V6")}

print(f"{len(MORFEMAS)} sufijos y {len(GRAMATICALES)} palabras gramaticales.")
print("Oracionales:", ", ".join(sorted(ORACIONALES)))

In [ ]:
# 3.2 · Algoritmo de construcción de palabras (CH-701 a CH-709).
def construir_palabra(base, ids, modo="B", traza=False):
    """Construye una palabra: base + sufijos, aplicando la caída y el alargamiento de vocales.

    Parámetros:
        base: raíz (de lexico.csv o real) o palabra gramatical en su forma B ("naya", "jani"…).
        ids: lista de ids de morfemas, en orden (CH-601, CH-602). Ejemplo: ["P1", "AFF"].
        modo: "B" (sufijos reales) o "A" (sufijos inventados).
        traza: si es True, devuelve también los pasos y las decisiones.
    Devuelve:
        La palabra. Con traza=True: (palabra, pasos, decisiones).
        'decisiones' es la lista que compara la prueba de paralelismo (CH-710.2):
        borrar, conservar, alargar, proteger, larga-no-borra, protegida-no-borra.
    Lanza:
        ValueError si la palabra final no es válida (CH-709) o el modo no existe.
        KeyError si un id no está en morfemas.yaml.
    Reglas: CH-701 (algoritmo), CH-702 (oracionales), CH-703 (la larga no se borra),
        CH-704 (alargamiento), CH-705 (VBLZ.LONG tras POSR), CH-706 (grupos enteros),
        CH-708 (palabras gramaticales en A), CH-709 (validación), CH-511 (alomorfo de REM.C).
    Ejemplo:
        >>> construir_palabra("tisqa", ["P1", "AFF"], "B")
        'tisqthwa'
        >>> construir_palabra("tisqa", ["P1", "AFF"], "A")
        'tisqphyi'
    """
    if modo not in ("B", "A"):
        raise ValueError("modo debe ser 'B' o 'A'")
    clave = modo.lower()
    forma0 = GRAM_B_A.get(base, base) if modo == "A" else base     # CH-708: la palabra gramatical cambia antes de los sufijos
    g = segmentar(forma0)
    despues_prev, id_prev, protegida = "V", "RAIZ", False
    pasos, decisiones = [f"raíz `{forma0}`"], []
    for k, mid in enumerate(ids):
        if mid not in MORFEMAS:
            raise KeyError(f"El morfema {mid!r} no está en morfemas.yaml")
        m = MORFEMAS[mid]
        f = m[clave]
        alo = m.get("alomorfo")
        if alo and k + 1 < len(ids) and ids[k + 1] == alo["ante"]:   # CH-511: REM.C es -¨- ante P3.REM
            f = alo[clave]
        if id_prev in m.get("invisible_tras", []):                    # CH-705: VBLZ.LONG tras POSR no se escribe
            protegida, despues_prev, id_prev = True, "V", mid
            decisiones.append("proteger")
            pasos.append(f"+ {mid} tras {m['invisible_tras'][0]}: no se escribe; protege `{g[-1]}` (CH-705) → `{''.join(g)}`")
            continue
        if f.startswith("¨"):                                         # CH-704: alarga la vocal anterior
            if g[-1] in VOCALES_CORTAS:
                g[-1] = LARGA[g[-1]]
            if f[1:]:
                g += segmentar(f[1:])
            protegida, despues_prev, id_prev = False, m["despues"], mid
            decisiones.append("alargar")
            pasos.append(f"+ {mid} `{f}`: alarga → `{''.join(g)}`")
            continue
        # CH-702: el oracional (antes "-") no decide; manda la columna 'después' del morfema anterior.
        borrar = m["antes"] == "C" or (m["antes"] == "-" and despues_prev == "C")
        quien = f"antes {m['antes']}" if m["antes"] != "-" else f"antes —; decide {id_prev} (después {despues_prev})"
        ultima = g[-1]
        if borrar and ultima in VOCALES_CORTAS and not protegida:
            g = g[:-1]
            d, texto = "borrar", f"borra `{ultima}`"
        elif borrar and protegida:
            d, texto = "protegida-no-borra", f"`{ultima}` protegida: no se borra (CH-705)"
        elif borrar and ultima in VOCALES_LARGAS:
            d, texto = "larga-no-borra", "no hay vocal corta que borrar (CH-703)"
        else:
            d, texto = "conservar", "conserva la vocal"
        decisiones.append(d)
        protegida = False
        g += segmentar(f)                                             # CH-706: los grupos se escriben enteros
        despues_prev, id_prev = m["despues"], mid
        pasos.append(f"+ {mid} `{f}` ({quien}): {texto} → `{''.join(g)}`")
    palabra = "".join(g)
    errores = validar_palabra(palabra)
    if errores:                                                       # CH-709: error del programa, no se descarta en silencio
        raise ValueError(f"CH-709: palabra inválida {palabra!r} ({base} + {ids}, modo {modo}): {errores}")
    return (palabra, pasos, decisiones) if traza else palabra


def construir_ambos(base, ids):
    """Construye la misma palabra en los dos modos.

    Parámetros:
        base: raíz o palabra gramatical (forma B).
        ids: lista de ids de morfemas.
    Devuelve:
        (forma B, forma A).
    Reglas: CH-501, CH-701.
    Ejemplo:
        >>> construir_ambos("chimula", ["PL", "POSS.1", "LOC"])
        ('chimulanakajana', 'chimulasululumu')
    """
    return construir_palabra(base, ids, "B"), construir_palabra(base, ids, "A")


def mostrar_traza(base, ids):
    """Muestra paso a paso cómo se construye una palabra en los dos modos.

    Parámetros:
        base: raíz o palabra gramatical (forma B).
        ids: lista de ids de morfemas.
    Devuelve:
        Una tabla (DataFrame) con una fila por paso y la salida al final.
    Reglas: CH-701 (formato de los ejemplos de §7.1), CH-710.2 (las decisiones de B y A coinciden).
    Ejemplo:
        >>> mostrar_traza("tisqa", ["P1", "AFF"])
    """
    pb, pasos_b, dec_b = construir_palabra(base, ids, "B", traza=True)
    pa, pasos_a, dec_a = construir_palabra(base, ids, "A", traza=True)
    filas = [{"paso": i, "modo B": b, "modo A": a} for i, (b, a) in enumerate(zip(pasos_b, pasos_a))]
    filas.append({"paso": "salida", "modo B": f"**{pb}**", "modo A": f"**{pa}**"})
    print(f"{base} + {' '.join(ids)} · mismas decisiones en B y A: {'sí' if dec_b == dec_a else 'NO'}")
    return pd.DataFrame(filas).set_index("paso")


with pd.option_context("display.max_colwidth", None):
    display(mostrar_traza("tisqa", ["P1", "AFF"]))
    display(mostrar_traza("chimula", ["POSR", "VBLZ.LONG", "P1", "AFF"]))

In [ ]:
# 3.3 · Orden de los sufijos (§6): ¿la lista de ids es una palabra posible?
SLOTS = CONFIG["slots"]
PLANTILLAS_NOMBRE = {n: re.compile(rx) for n, rx in SLOTS["nombre"]["plantillas"].items()}
PLANTILLA_VERBO = re.compile(SLOTS["verbo"]["plantilla"])
GRUPOS_ID = {                                 # nombres de grupo que usa 'excluye' en slots.yaml
    "POSS": SLOTS["nombre"]["slots"]["N2"],
    "CASO": SLOTS["nombre"]["slots"]["N4"],
}
INTERROGATIVOS_B = {p["b"] for i, p in GRAMATICALES.items() if i.startswith("INT.")}


def errores_ids_nombre(ids, base=None):
    """Revisa la secuencia de ids de una palabra nominal (nombre, pronombre, predicado).

    Parámetros:
        ids: lista de ids de morfemas (sin la raíz).
        base: forma B de la base, opcional. Sirve para la regla de Q.INF.
    Devuelve:
        Lista de errores con su regla. Vacía si la secuencia es posible.
    Reglas: CH-601 (plantillas NP_NORMAL, ESTAR, TENER, SER y sus restricciones), CH-514, CH-906.
        La restricción "PL no va con numeral" es de la frase, no de la palabra: se revisa en la sección 8.
    Ejemplo:
        >>> errores_ids_nombre(["PL", "POSS.1", "LOC"])
        []
        >>> errores_ids_nombre(["GEN", "TOP"])
        ['CH-601: GEN no lleva nada después']
    """
    cadena = " ".join(["RAIZ_N"] + list(ids))
    cumple = [n for n, rx in PLANTILLAS_NOMBRE.items() if rx.match(cadena)]
    errores = [] if cumple else [f"CH-601: '{' '.join(ids)}' no cumple ninguna plantilla del nombre"]
    presentes = lambda grupo: any(i in GRUPOS_ID.get(grupo, [grupo]) for i in ids)
    # 'excluye' se aplica a NP_NORMAL: ESTAR, TENER y SER ya fijan su estructura (utankthwa = LOC … AFF). Ver P-03.
    for a, b in (SLOTS["nombre"]["excluye"] if "NP_NORMAL" in cumple or not cumple else []):
        if b == "CUALQUIERA_DESPUES":
            if a in ids and ids.index(a) < len(ids) - 1:
                errores.append(f"CH-601: {a} no lleva nada después")
        elif b == "NUMERAL_EN_FRASE":
            continue                                     # se revisa en la frase (sección 8)
        elif presentes(a) and presentes(b):
            errores.append(f"CH-601: {a} y {b} no van juntos")
    if "LOC" in ids and ids.index("LOC") < len(ids) - 1 and ids[ids.index("LOC") + 1] != "VBLZ.KA":
        errores.append("CH-601: tras LOC solo puede ir VBLZ.KA")   # texto de CH-601 (ver PREGUNTAS.md, P-03)
    if base is not None and "Q.INF" in ids and base not in INTERROGATIVOS_B:
        errores.append("CH-906: Q.INF solo va en interrogativos")
    return sorted(set(errores))


def errores_ids_verbo(ids):
    """Revisa la secuencia de ids de un verbo.

    Parámetros:
        ids: lista de ids de morfemas (sin la raíz).
    Devuelve:
        Lista de errores con su regla. Vacía si la secuencia es posible.
    Reglas: CH-602 (plantilla y NEG ⇔ NEG.TI), CH-603 (un solo morfema en V1), CH-604 (transiciones solo
        en SIMP), CH-605 (negación solo en SIMP y FUT), CH-606 (progresivo solo en SIMP).
        La condición "obj_persona = s" depende del verbo: se revisa en la idea (R1, sección 9).
    Ejemplo:
        >>> errores_ids_verbo(["PL.V", "REM.C", "P1"])
        []
        >>> errores_ids_verbo(["NEG", "REM.C", "P1", "NEG.TI"])
        ['CH-605: NEG solo en SIMP y FUT']
    """
    errores = []
    if not PLANTILLA_VERBO.match(" ".join(["RAIZ_V"] + list(ids))):
        errores.append(f"CH-602: '{' '.join(ids)}' no cumple la plantilla del verbo")
    if ("NEG" in ids) != ("NEG.TI" in ids):
        errores.append("CH-602: NEG y NEG.TI van siempre juntos")
    if sum(i in SLOTS["verbo"]["slots"]["V1"] for i in ids) > 1:
        errores.append("CH-603: PL.V, PROG y NEG se excluyen")
    remoto = any(i in SLOTS["verbo"]["slots"]["V4"] for i in ids)
    futuro = any(i in SLOTS["verbo"]["slots"]["V5"]["FUT"] for i in ids)
    if "NEG" in ids and remoto:
        errores.append("CH-605: NEG solo en SIMP y FUT")
    if "PROG" in ids and (remoto or futuro):
        errores.append("CH-606: PROG solo en SIMP")
    return errores


print(errores_ids_nombre(["PL", "POSS.1", "LOC"]), errores_ids_nombre(["GEN", "TOP"]))
print(errores_ids_verbo(["PL.V", "REM.C", "P1"]), errores_ids_verbo(["PL.V", "NEG", "P3", "NEG.TI"]))

### Pruebas de la sección 3

- **§5 (tabla de morfemas):** el mapeo B → A cumple CH-501, CH-502 y CH-503.
- **§6 (orden de los sufijos):** al menos una prueba por regla CH-601 a CH-606.
- **§7 (algoritmo):** los 16 ejemplos paso a paso de §7.1 de la especificación, en los dos modos.

In [ ]:
# 3.4 · Pruebas de §5, §6 y §7.
def silabas(forma):
    """Cuenta las sílabas de una forma (una por vocal, corta o larga). El '¨' no cuenta.

    Parámetros:
        forma: forma de un sufijo, como "tana" o "¨ta".
    Devuelve:
        Número de sílabas.
    Reglas: CH-503 (condición 1).
    Ejemplo:
        >>> silabas("ñäni")
        2
    """
    f = forma.lstrip("¨")
    return sum(1 for g in segmentar(f) if g in VOCALES_CORTAS or g in VOCALES_LARGAS) if f else 0


def tipo_inicio(forma):
    """Clasifica el inicio de una forma: 'V', 'C' (una consonante) o 'CC' (dos).

    Parámetros:
        forma: forma de un sufijo.
    Devuelve:
        'V', 'C', 'CC' o '' si la forma es solo '¨'.
    Reglas: CH-503 (condición 2).
    Ejemplo:
        >>> tipo_inicio("sma")
        'CC'
    """
    c = clases(segmentar(forma.lstrip("¨"))) if forma.lstrip("¨") else ""
    return "" if not c else "V" if c[0] in "VL" else "CC" if c[:2] == "CC" else "C"


# Los 16 ejemplos de §7.1: (base, ids, B, A).
EJEMPLOS_7_1 = [
    ("tisqa", "P1 AFF", "tisqthwa", "tisqphyi"),
    ("tisqa", "P2 AFF", "tisqtawa", "tisqkuyi"),
    ("tisqa", "P4 AFF", "tisqtanwa", "tisqkilyi"),
    ("ch'ixra", "T31 AFF", "ch'ixrituwa", "ch'ixruquyi"),
    ("ch'ixra", "T32 AFF", "ch'ixrtamwa", "ch'ixrkiqyi"),
    ("tisqa", "NEG FUT.1 NEG.TI", "tisqkäti", "tisqñüji"),
    ("tisqa", "NEG P1 NEG.TI", "tisqkthti", "tisqñphji"),
    ("tisqa", "NEG FUT.3 NEG.TI", "tisqkaniti", "tisqñuphuji"),
    ("tisqa", "PL.V REM.C P1", "tisqapxayätha", "tisqamtujüphi"),
    ("tisqa", "PL.V REM.C P3.REM", "tisqapxäna", "tisqamtümu"),
    ("tisqa", "REM.L P4 AFF", "tisqatätanwa", "tisqawükilyi"),
    ("tisqa", "PROG P3 AFF", "tisqaskiwa", "tisqanchuyi"),
    ("chimula", "LOC VBLZ.KA P1 AFF", "chimulankthwa", "chimulamñphyi"),
    ("chimula", "POSR VBLZ.LONG P1 AFF", "chimulanithwa", "chimulaphuphyi"),
    ("jumpu", "VBLZ.LONG P1 AFF", "jumpüthwa", "jumpüphyi"),
    ("chimula", "PL POSS.1 LOC", "chimulanakajana", "chimulasululumu"),
]


class PruebasMorfemas(unittest.TestCase):
    def test_CH501_mapeo_por_forma(self):
        """CH-501: una cadena B da siempre la misma cadena A (homófonos de la tabla)."""
        a_de_b = {}
        for m in MORFEMAS.values():
            a_de_b.setdefault(m["b"], set()).add(m["a"])
        self.assertEqual({b: a for b, a in a_de_b.items() if len(a) > 1}, {})
        homofonos = {"ta": "ku", "sa": "khu", "ni": "phu", "na": "mu", "ka": "ñu", "ti": "ji", "ru": "llu"}
        for b, a in homofonos.items():
            self.assertEqual(a_de_b[b], {a}, b)
        for b, a in ((m["b"], m["a"]) for m in MORFEMAS.values()):   # ¨ta → ¨ + A(ta)
            if b.startswith("¨") and b[1:] in a_de_b:
                self.assertEqual(a, "¨" + next(iter(a_de_b[b[1:]])), b)

    def test_CH502_alargamiento_igual_en_A(self):
        """CH-502: los morfemas que son solo '¨' son iguales en A; tisqa + FUT.1 = tisqä en los dos modos."""
        for m in MORFEMAS.values():
            if m["b"] == "¨":
                self.assertEqual(m["a"], "¨", m["id"])
        self.assertEqual(construir_ambos("tisqa", ["FUT.1"]), ("tisqä", "tisqä"))

    def test_CH503_condiciones_de_las_formas_A(self):
        """CH-503: mismas sílabas, mismo tipo de inicio, larga en la misma sílaba, alfabeto y B distintas → A distintas."""
        for m in MORFEMAS.values():
            b, a = m["b"], m["a"]
            with self.subTest(morfema=m["id"]):
                segmentar(a.lstrip("¨") or "a")                                  # condición 4
                self.assertEqual(silabas(b), silabas(a))                         # condición 1
                self.assertEqual(tipo_inicio(b), tipo_inicio(a))                 # condición 2
                larga = lambda f: [g in VOCALES_LARGAS for g in segmentar(f.lstrip("¨")) if g in VOCALES_CORTAS | VOCALES_LARGAS] if f.lstrip("¨") else []
                self.assertEqual(larga(b), larga(a))                             # condición 3
        b_de_a = {}
        for m in MORFEMAS.values():
            b_de_a.setdefault(m["a"], set()).add(m["b"])
        self.assertEqual({a: b for a, b in b_de_a.items() if len(b) > 1}, {})   # condición 6
        formas_b = {m["b"] for m in MORFEMAS.values()} - {"¨"}
        self.assertEqual({m["a"] for m in MORFEMAS.values()} & formas_b, set())  # condición 5 (contra los sufijos B)

    def test_CH505_palabras_gramaticales(self):
        """CH-505: cada palabra gramatical tiene forma B y A con el mismo número de sílabas, y A ≠ B."""
        for p in GRAMATICALES.values():
            with self.subTest(palabra=p["id"]):
                self.assertEqual(silabas(p["b"]), silabas(p["a"]))
                self.assertNotEqual(p["b"], p["a"])
        self.assertEqual(construir_ambos("jiwasa", ["PL", "TOP"]), ("jiwasanakaxa", "wiqutusuluxu"))   # CH-708

    def test_CH511_remoto_tercera(self):
        """CH-511: REM.C toma -¨- ante P3.REM (tisqäna / tisqämu)."""
        self.assertEqual(construir_ambos("tisqa", ["REM.C", "P1"]), ("tisqayätha", "tisqajüphi"))
        self.assertEqual(construir_ambos("tisqa", ["REM.C", "P3.REM"]), ("tisqäna", "tisqämu"))

    def test_CH512_rem_l_con_ta(self):
        """CH-512: REM.L usa -tä- (tisqatäna / tisqawümu), nunca -tayna."""
        self.assertEqual(construir_ambos("tisqa", ["REM.L", "P3.REM"]), ("tisqatäna", "tisqawümu"))

    def test_CH513_futuro_primera(self):
        """CH-513: FUT.1 = -¨ (tisqä)."""
        self.assertEqual(construir_palabra("tisqa", ["FUT.1"]), "tisqä")


class PruebasSlots(unittest.TestCase):
    def test_CH601_plantilla_nombre(self):
        """CH-601: plantillas NP_NORMAL, ESTAR, TENER y SER, y sus restricciones."""
        for ids in ["PL POSS.1 LOC", "LOC VBLZ.KA P1 AFF", "PL POSR VBLZ.LONG P1 AFF", "POSR AFF", "VBLZ.LONG P2 Q.YN", "COM AFF"]:
            self.assertEqual(errores_ids_nombre(ids.split()), [], ids)
        for ids in ["POSS.1 POSR", "POSR LOC", "UNTIL PL", "PL UNTIL", "POSS.2 UNTIL", "GEN TOP",
                    "ABL AFF", "LOC AFF", "LOC Q.INF", "LOC TOP", "VBLZ.LONG P3 AFF", "POSR VBLZ.LONG P3", "LOC PL"]:
            self.assertNotEqual(errores_ids_nombre(ids.split()), [], ids)
        self.assertEqual(errores_ids_nombre(["Q.INF"], base="khiti"), [])
        self.assertTrue(tiene_regla(errores_ids_nombre(["Q.INF"], base="naya"), "CH-906"))

    def test_CH602_plantilla_verbo(self):
        """CH-602: tisqa-pxa-yä-tha es válida; NEG sin NEG.TI no."""
        self.assertEqual(errores_ids_verbo(["PL.V", "REM.C", "P1"]), [])
        self.assertEqual(errores_ids_verbo(["NEG", "FUT.3", "NEG.TI"]), [])
        self.assertTrue(tiene_regla(errores_ids_verbo(["NEG", "P3"]), "CH-602"))
        self.assertTrue(tiene_regla(errores_ids_verbo(["P3", "P1"]), "CH-602"))

    def test_CH603_v1_un_solo_morfema(self):
        """CH-603: PL.V + NEG es inválido (tisqa-pxa-ka-i-ti)."""
        self.assertNotEqual(errores_ids_verbo(["PL.V", "NEG", "P3", "NEG.TI"]), [])
        self.assertNotEqual(errores_ids_verbo(["PL.V", "PROG", "P3"]), [])

    def test_CH604_transiciones_solo_simp(self):
        """CH-604: no hay transición en los remotos ni en el futuro (REM.C + T31)."""
        self.assertEqual(errores_ids_verbo(["T31", "AFF"]), [])
        self.assertNotEqual(errores_ids_verbo(["REM.C", "T31"]), [])
        self.assertNotEqual(errores_ids_verbo(["T31", "FUT.1"]), [])

    def test_CH605_negacion_simp_fut(self):
        """CH-605: NEG + REM.C o REM.L es inválido."""
        self.assertTrue(tiene_regla(errores_ids_verbo(["NEG", "REM.C", "P1", "NEG.TI"]), "CH-605"))
        self.assertTrue(tiene_regla(errores_ids_verbo(["NEG", "REM.L", "P3.REM", "NEG.TI"]), "CH-605"))

    def test_CH606_progresivo_simp(self):
        """CH-606: PROG + FUT es inválido; PROG + P3 es válido."""
        self.assertEqual(errores_ids_verbo(["PROG", "P3", "AFF"]), [])
        self.assertTrue(tiene_regla(errores_ids_verbo(["PROG", "FUT.3"]), "CH-606"))


class PruebasMotor(unittest.TestCase):
    def test_CH701_ejemplos_paso_a_paso(self):
        """CH-701: los 16 ejemplos de §7.1 dan la salida de la especificación en B y en A."""
        for base, ids, b, a in EJEMPLOS_7_1:
            with self.subTest(base=base, ids=ids):
                self.assertEqual(construir_ambos(base, ids.split()), (b, a))

    def test_CH702_oracional_no_decide(self):
        """CH-702: tras P1 (después C) el oracional borra; tras P2 (después V) conserva."""
        self.assertEqual(construir_palabra("tisqa", ["P1", "AFF"]), "tisqthwa")
        self.assertEqual(construir_palabra("tisqa", ["P2", "AFF"]), "tisqtawa")

    def test_CH703_la_larga_no_se_borra(self):
        """CH-703: tisqa-yä-tha → tisqayätha (P1 pide borrar, pero ä es larga)."""
        _, _, dec = construir_palabra("tisqa", ["REM.C", "P1"], traza=True)
        self.assertEqual(dec[-1], "larga-no-borra")

    def test_CH704_alargamiento(self):
        """CH-704: tisqa-¨ta → tisqäta (B) / tisqäku (A)."""
        self.assertEqual(construir_ambos("tisqa", ["FUT.2"]), ("tisqäta", "tisqäku"))

    def test_CH705_vblz_long_tras_posr(self):
        """CH-705: chimula-ni-(¨)-tha-wa → chimulanithwa; la decisión es 'proteger'."""
        _, _, dec = construir_palabra("chimula", ["POSR", "VBLZ.LONG", "P1", "AFF"], traza=True)
        self.assertEqual(dec, ["conservar", "proteger", "protegida-no-borra", "borrar"])

    def test_CH706_grupos_enteros(self):
        """CH-706: tisqkthti y tisqñphji conservan todo el grupo de consonantes."""
        self.assertEqual(construir_ambos("tisqa", ["NEG", "P1", "NEG.TI"]), ("tisqkthti", "tisqñphji"))

    def test_CH708_gramaticales_en_A(self):
        """CH-708: en A, la palabra gramatical cambia antes de añadir sufijos (naya → muqi)."""
        self.assertEqual(construir_ambos("naya", ["TOP"]), ("nayaxa", "muqixu"))
        self.assertEqual(construir_ambos("jani", ["AFF"]), ("janiwa", "yuliyi"))

    def test_CH709_palabra_invalida_es_error(self):
        """CH-709: una palabra inválida es un error del programa (xiku empieza por x)."""
        with self.assertRaises(ValueError):
            construir_palabra("xiku", [])


correr_pruebas(PruebasMorfemas, PruebasSlots, PruebasMotor, titulo="motor morfológico (§5, §6, §7)")

## 4. Pruebas de referencia (CH-710)

**Qué hace esta sección.** Prueba el motor con **raíces reales** del aymara. Si el motor reproduce las formas de los libros, sus reglas son correctas.

1. **Prueba de igualdad (modo B).** Los 102 casos de `regresion_ch710.json`. Cada caso es una raíz real, una lista de ids y la forma que aparece en el libro L2. El motor debe dar **exactamente** esa forma.
2. **Prueba de paralelismo (modo A).** Para cada caso, el motor también construye la palabra en modo A. En cada paso, la decisión (borrar, conservar, alargar o proteger) debe ser la misma que en B. Así sabemos que A es un "espejo" fiel de B.
3. **Paradigmas (§8).** Las 64 celdas de `paradigmas.yaml` con el verbo de ejemplo `ch'ixra` "ver", en B y en A.

Los datos se usan **tal como están**, sin cambios.

Si un solo caso falla, el notebook se detiene: alguna regla está mal y no se deben generar datos.

**Dónde están los 40 casos de CH-1201.** Necesitan el constructor de oraciones y el español. Por eso se prueban al final de la sección 8. La sección 10 repite todas las pruebas de referencia antes de generar (CH-1201: "si un caso falla, la generación no se hace").

**Reglas que implementa.** CH-710 (los tres puntos) y las tablas de §8.

**Qué entra.** `regresion_ch710.json` y el bloque `paradigmas.yaml`. **Qué sale.** Una tabla de resultados.

**Ejemplo** (caso de CH-710): `sara` + `PL.V REM.C P1`:
- B: `sarapxayätha` (L2 p.181) ✓ · A: `saramtujüphi` · Español: "hemos ido".
- Decisiones en los dos modos: conservar, conservar, larga-no-borra.

In [ ]:
# 4.1 · Cargar los 102 casos de CH-710 y compararlos con el bloque de la especificación.
REGRESION = json.loads(ruta("regresion_ch710.json").read_text(encoding="utf-8"))
clave_caso = lambda c: (c["raiz"], tuple(c["ids"]), c["esperado"])
if "regresion_raices_reales" in CONFIG:
    en_spec = {clave_caso(c) for c in CONFIG["regresion_raices_reales"]["casos"]}
    distintos = en_spec ^ {clave_caso(c) for c in REGRESION}
    print("El JSON y el bloque CH-710 de la especificación", "coinciden." if not distintos else
          f"NO coinciden en {len(distintos)} casos (se usa el JSON): {sorted(distintos)[:5]}")
print(f"{len(REGRESION)} casos de regresión cargados.")


def resultado_caso(c):
    """Construye un caso de CH-710 en los dos modos y compara con lo esperado.

    Parámetros:
        c: caso del JSON, con 'raiz', 'ids', 'esperado' y 'fuente'.
    Devuelve:
        Diccionario con la salida en B y en A, si B es igual a lo esperado y si las decisiones coinciden.
    Reglas: CH-710.1 (igualdad) y CH-710.2 (paralelismo).
    Ejemplo:
        >>> resultado_caso({"raiz": "sara", "ids": ["P1", "AFF"], "esperado": "sarthwa", "fuente": "L2 p.176"})["igual"]
        True
    """
    b, _, dec_b = construir_palabra(c["raiz"], c["ids"], "B", traza=True)
    a, _, dec_a = construir_palabra(c["raiz"], c["ids"], "A", traza=True)
    return {"raíz": c["raiz"], "ids": " ".join(c["ids"]), "esperado": c["esperado"], "B": b, "A": a,
            "igual": b == c["esperado"], "paralelo": dec_b == dec_a, "decisiones": ", ".join(dec_b),
            "fuente": c["fuente"]}


TABLA_CH710 = pd.DataFrame([resultado_caso(c) for c in REGRESION])
display(TABLA_CH710.sample(12, random_state=0).drop(columns="fuente"))

In [ ]:
# 4.2 · Pruebas de referencia. Si una falla, el notebook se detiene.
class PruebasReferencia(unittest.TestCase):
    def test_CH710_cantidad(self):
        """CH-710: la lista tiene 102 casos."""
        self.assertEqual(len(REGRESION), 102)

    def test_CH710_1_igualdad_modo_B(self):
        """CH-710.1: el modo B reproduce exactamente las 102 formas de la guía."""
        fallas = TABLA_CH710[~TABLA_CH710["igual"]][["raíz", "ids", "esperado", "B", "fuente"]]
        self.assertTrue(fallas.empty, "\n" + fallas.to_string())

    def test_CH710_2_paralelismo_modo_A(self):
        """CH-710.2: en los 102 casos, el modo A toma las mismas decisiones que el B."""
        fallas = TABLA_CH710[~TABLA_CH710["paralelo"]][["raíz", "ids", "B", "A"]]
        self.assertTrue(fallas.empty, "\n" + fallas.to_string())

    def test_CH710_ejemplo(self):
        """CH-710: sara + PL.V REM.C P1 → sarapxayätha / saramtujüphi, con decisiones [conservar, conservar, larga-no-borra]."""
        b, _, dec = construir_palabra("sara", ["PL.V", "REM.C", "P1"], "B", traza=True)
        self.assertEqual((b, construir_palabra("sara", ["PL.V", "REM.C", "P1"], "A")), ("sarapxayätha", "saramtujüphi"))
        self.assertEqual(dec, ["conservar", "conservar", "larga-no-borra"])

    def test_paradigmas_seccion_8(self):
        """§8: las celdas de paradigmas.yaml dan las formas de prueba en B y en A."""
        P = CONFIG["paradigmas"]
        for celda in P["celdas"]:
            with self.subTest(tiempo=celda["tiempo"], celda=celda["celda"], ids=celda["ids"]):
                self.assertEqual(construir_ambos(P["verbo_prueba"], celda["ids"]),
                                 (celda["prueba"]["b"], celda["prueba"]["a"]))
        print(f"\n   ({len(P['celdas'])} celdas de paradigma probadas)", end=" ")


correr_pruebas(PruebasReferencia, titulo="pruebas de referencia CH-710")
resumen = pd.DataFrame([
    {"prueba": "CH-710.1 igualdad (modo B)", "casos": len(TABLA_CH710), "en verde": int(TABLA_CH710["igual"].sum())},
    {"prueba": "CH-710.2 paralelismo (modo A)", "casos": len(TABLA_CH710), "en verde": int(TABLA_CH710["paralelo"].sum())},
    {"prueba": "§8 paradigmas (B y A)", "casos": len(CONFIG["paradigmas"]["celdas"]), "en verde": len(CONFIG["paradigmas"]["celdas"])},
])
display(resumen)

## 5. Léxico

**Qué hace esta sección.** Lee el **léxico**: la lista de palabras con significado (nombres, verbos, adjetivos y adverbios de tiempo). Cada fila es un **sentido** del español ("niño", "ir"…) con sus datos.

**De dónde sale.**
- Si en `datos/` está `lexico_sin_raices.csv` (lo entrega el rol 1), se usa ese archivo. Sus raíces las fabrica la sección 6.
- Si no está, se usan las **25 entradas de ejemplo** de §4.1 de la especificación. Esas ya traen raíz: la sección 6 solo las revisa (ver PREGUNTAS.md, P-05).

**Qué se revisa** (CH-402 y CH-406). Que estén todas las columnas, que cada valor sea uno de los permitidos y que las listas de clases sean coherentes. Por ejemplo: un `N_HUM` siempre tiene `clase_sem = persona`, y un objeto directo nunca es `persona` (CH-914).

**Términos.**
- **categoría** (`cat`): tipo de palabra. `N_HUM` = nombre de persona, `V_TR` = verbo transitivo, etc. (CH-401).
- **clase semántica** (`clase_sem`): grupo de significado de un nombre: `comida`, `animal`, `lugar`… (CH-406).
- **rango**: puesto de frecuencia dentro de la categoría. 1 = la palabra más común (CH-403).

**Qué entra.** `lexico_sin_raices.csv` o el bloque `csv` de §4.1. **Qué sale.** La tabla `LEXICO`.

**Ejemplo** (fila L0013 de §4.1): "ir" es `V_MOV`, su sujeto es `persona` o `animal` y no admite LOC (`NINGUNA`). Su raíz es `tisqa`:
- B: `tisqthwa` — ir-1>3.SIMP-AFF — "voy".
- A: `tisqphyi`.

In [ ]:
# 5.1 · Leer el léxico y revisar su formato (CH-401, CH-402, CH-406).
import io

CATEGORIAS = list(CONFIG["generador_raices"]["silabas"])          # CH-401: las 8 categorías
VERBOS = [c for c in CATEGORIAS if c.startswith("V_")]
NOMBRES = ["N_HUM", "N_NOHUM"]
CLASES_SEM = ["persona", "animal", "comida", "bebida", "ropa", "herramienta", "recipiente",
              "planta", "lugar", "vivienda", "tiempo", "abstracto", "objeto"]   # tabla 1 de CH-406
SUBCAT = {"ADJ": ["tamano", "color", "otro"], "ADV_T": ["pasado", "pasado_remoto", "presente", "futuro"]}   # CH-401 (v1.1.2)

CSV_EJEMPLO = re.search(r"```csv\n(.*?)```", TEXTO_SPEC, re.S).group(1)   # §4.1
COLUMNAS_LEXICO = CSV_EJEMPLO.splitlines()[0].split(",")                   # CH-402


def leer_csv(fuente):
    """Lee un léxico en formato CH-402. Todo se lee como texto: "-" y las celdas vacías se conservan.

    Parámetros:
        fuente: ruta del archivo o texto csv (io.StringIO).
    Devuelve:
        DataFrame con una fila por sentido.
    Reglas: CH-402 (UTF-8, separador ",").
    Ejemplo:
        >>> leer_csv(io.StringIO(CSV_EJEMPLO)).shape
        (26, 22)
    """
    df = pd.read_csv(fuente, dtype=str, keep_default_na=False, encoding="utf-8")
    return df.apply(lambda col: col.map(lambda v: unicodedata.normalize("NFC", v)))   # CH-402: NFC


def errores_lexico(df):
    """Revisa el formato y la coherencia de un léxico.

    Parámetros:
        df: DataFrame que devuelve leer_csv().
    Devuelve:
        Lista de errores, cada uno con el id de la fila y su regla. Vacía si todo está bien.
    Reglas: CH-401 (categorías y subcategorías), CH-402 (columnas y valores), CH-405 (género),
        CH-406 (clases), CH-909 (rige_ru solo en V_DITR), CH-914 (el OD nunca es persona).
    Ejemplo:
        >>> errores_lexico(LEXICO_EJEMPLO)
        []
    """
    faltan = [c for c in COLUMNAS_LEXICO if c not in df.columns]
    if faltan:
        return [f"CH-402: faltan columnas {faltan}"]
    errores = [f"CH-402: id repetido {i}" for i in df["id"][df["id"].duplicated()]]

    def lista(v):
        return v.split("|") if v not in ("-", "", "NINGUNA") else []

    for f in df.itertuples():
        e = []
        cat = f.cat
        if not re.fullmatch(r"L\d{4,}", f.id):
            e.append("CH-402: id con formato distinto de L + 4 o más cifras (L0001, L10001)")
        if cat not in CATEGORIAS:
            errores.append(f"{f.id}: CH-401: categoría desconocida {cat!r}")
            continue
        if f.subcat not in SUBCAT.get(cat, ["-"]):
            e.append(f"CH-401: subcat {f.subcat!r} no vale para {cat}")
        es_nombre, es_verbo, con_od = cat in NOMBRES, cat in VERBOS, cat in ("V_TR", "V_DITR")
        clase_ok = ("persona",) if cat == "N_HUM" else CLASES_SEM[1:] if cat == "N_NOHUM" else ("-",)
        if f.clase_sem not in clase_ok:
            e.append(f"CH-406: clase_sem {f.clase_sem!r} no vale para {cat}")
        if not f.es_m:
            e.append("CH-402: es_m vacío")
        if f.genero not in (["m", "f", "var"] if es_nombre else ["-"]):
            e.append(f"CH-405: genero {f.genero!r} no vale para {cat}")
        if f.genero == "var" and not f.es_f:
            e.append("CH-405: genero = var sin forma femenina (es_f)")
        if f.contable not in (["s", "n"] if es_nombre else ["-"]):
            e.append(f"CH-402: contable {f.contable!r} no vale para {cat}")
        if f.rige_ru not in ((["s", "n"] if cat == "V_DITR" else ["n"]) if es_verbo else ["-"]):
            e.append(f"CH-909: rige_ru {f.rige_ru!r} no vale para {cat}")
        if f.obj_persona not in (["s", "n"] if con_od else ["-"]):
            e.append(f"CH-604: obj_persona {f.obj_persona!r} no vale para {cat}")
        if f.admite_ben not in (["s", "n"] if con_od else ["-"]):
            e.append(f"CH-406: admite_ben {f.admite_ben!r} no vale para {cat}")
        if not f.rango.isdigit() or int(f.rango) < 1:
            e.append(f"CH-403: rango {f.rango!r} no es un entero ≥ 1")
        # Listas de clases: obligatorias u opcionales según CH-402; con clases de CH-406.
        obligatorias = {"suj_clases": es_verbo, "obj_clases": con_od, "mod_clases": cat == "ADJ"}
        permitidas = {"suj_clases": es_verbo, "obj_clases": con_od, "instr_clases": es_verbo,
                      "loc_clases": es_verbo, "mod_clases": cat == "ADJ"}
        for col, permitida in permitidas.items():
            v = getattr(f, col)
            if not permitida and v != "-":
                e.append(f"CH-402: {col} debe ser '-' en {cat}")
            elif obligatorias.get(col) and not lista(v):
                e.append(f"CH-402: {col} es obligatoria en {cat}")
            elif v == "NINGUNA" and col != "loc_clases":
                e.append(f"CH-402: NINGUNA solo vale en loc_clases")
            otras = [c for c in lista(v) if c not in CLASES_SEM]
            if otras:
                e.append(f"CH-406: {col} con clases desconocidas {otras}")
        if "persona" in lista(f.obj_clases):
            e.append("CH-914: el objeto directo nunca es persona")
        if cat == "V_MOV" and f.loc_clases != "NINGUNA":
            e.append("CH-406: los V_MOV llevan siempre loc_clases = NINGUNA")
        errores += [f"{f.id}: {x}" for x in e]
    rep = df[df.duplicated(["cat", "rango"], keep=False)]
    errores += [f"{i}: CH-403: rango repetido dentro de {c}" for i, c in zip(rep["id"], rep["cat"])]
    return errores


LEXICO_EJEMPLO = leer_csv(io.StringIO(CSV_EJEMPLO))
RUTA_LEXICO_PROD = ruta("lexico_sin_raices.csv")
LEXICO_PRODUCCION = RUTA_LEXICO_PROD.exists()                  # P-05
LEXICO = leer_csv(RUTA_LEXICO_PROD) if LEXICO_PRODUCCION else LEXICO_EJEMPLO.copy()
if LEXICO_PRODUCCION:                                         # el rol 1 entrega el archivo sin raíz ni sílabas
    LEXICO = LEXICO.assign(raiz="", silabas="").reindex(columns=COLUMNAS_LEXICO)

errores = errores_lexico(LEXICO)
if errores:
    raise ValueError(f"El léxico tiene {len(errores)} errores:\n" + "\n".join(errores[:50]))
print(f"Léxico: {'lexico_sin_raices.csv' if LEXICO_PRODUCCION else '§4.1 de la especificación (ejemplo, P-05)'}"
      f" · {len(LEXICO)} entradas · sin errores de formato.")
display(LEXICO.groupby("cat").size().reindex(CATEGORIAS, fill_value=0).rename("entradas").to_frame().T)

### Pruebas de la sección 5

- **CH-402:** el léxico de §4.1 tiene las 22 columnas y ningún error.
- **CH-401, CH-406, CH-909, CH-914:** si se estropea una fila a propósito, la revisión lo detecta y cita la regla.

In [ ]:
# 5.2 · Pruebas del léxico.
class PruebasLexico(unittest.TestCase):
    def roto(self, i, **cambios):
        """Copia del léxico de §4.1 con la fila i cambiada."""
        df = LEXICO_EJEMPLO.copy()
        for col, v in cambios.items():
            df.loc[i, col] = v
        return errores_lexico(df)

    def test_CH402_ejemplo_correcto(self):
        """CH-402: las 26 entradas de §4.1 tienen las 22 columnas y pasan la revisión."""
        self.assertEqual(list(LEXICO_EJEMPLO.columns), COLUMNAS_LEXICO)
        self.assertEqual((len(LEXICO_EJEMPLO), len(COLUMNAS_LEXICO)), (26, 22))
        self.assertEqual(errores_lexico(LEXICO_EJEMPLO), [])

    def test_CH402_falta_columna(self):
        """CH-402: sin la columna rango, el léxico no se acepta."""
        self.assertTrue(tiene_regla(errores_lexico(LEXICO_EJEMPLO.drop(columns="rango")), "CH-402"))

    def test_CH402_formato_id(self):
        """CH-402 (decisión 20-11-2026): L + 4 o más cifras; L10001 vale, L001 y X0001 no."""
        self.assertFalse(tiene_regla(self.roto(0, id="L10001"), "CH-402"))
        self.assertTrue(tiene_regla(self.roto(0, id="L001"), "CH-402"))
        self.assertTrue(tiene_regla(self.roto(0, id="X0001"), "CH-402"))

    def test_CH402_id_repetido(self):
        """CH-402: dos filas con el mismo id."""
        self.assertTrue(tiene_regla(self.roto(1, id="L0001"), "CH-402"))

    def test_CH401_categoria_desconocida(self):
        """CH-401: 'PRON' no es una categoría del léxico (los pronombres son palabras gramaticales)."""
        self.assertTrue(tiene_regla(self.roto(0, cat="PRON"), "CH-401"))

    def test_CH406_nhum_es_persona(self):
        """CH-406: un N_HUM ("niño", fila 0) con clase animal."""
        self.assertTrue(tiene_regla(self.roto(0, clase_sem="animal"), "CH-406"))

    def test_CH914_od_no_persona(self):
        """CH-914: "comer" (fila 15) con persona entre las clases del OD."""
        self.assertTrue(tiene_regla(self.roto(15, obj_clases="comida|persona"), "CH-914"))

    def test_CH406_admite_ben(self):
        """CH-406 (v1.1.2): admite_ben es s/n en V_TR y V_DITR, y "-" en el resto."""
        self.assertTrue(tiene_regla(self.roto(15, admite_ben="-"), "CH-406"))
        self.assertTrue(tiene_regla(self.roto(12, admite_ben="s"), "CH-406"))

    def test_CH909_rige_ru_solo_ditr(self):
        """CH-909: "comer" (V_TR) con rige_ru = s."""
        self.assertTrue(tiene_regla(self.roto(15, rige_ru="s"), "CH-909"))

    def test_CH406_vmov_sin_loc(self):
        """CH-406: "ir" (V_MOV, fila 12) que acepte LOC."""
        self.assertTrue(tiene_regla(self.roto(12, loc_clases="-"), "CH-406"))

    def test_CH406_clase_desconocida(self):
        """CH-406: "grande" (fila 20) que modifique la clase inexistente 'mueble'."""
        self.assertTrue(tiene_regla(self.roto(20, mod_clases="objeto|mueble"), "CH-406"))


correr_pruebas(PruebasLexico, titulo="léxico (§4)")

## 6. Raíces inventadas

**Qué hace esta sección.** Fabrica una **raíz inventada** para cada sentido del léxico y escribe `lexico.csv`. Corre una sola vez, con una semilla fija: así el diccionario siempre sale igual (CH-309). Las oraciones nunca inventan raíces: solo las leen de `lexico.csv`.

**Cómo se fabrica una raíz** (§3.1 de la especificación):
1. Se sortea el número de sílabas: 2 o 3, según la categoría (CH-301).
2. Se sortea la forma de cada sílaba (V, CV, VC o CVC) y sus letras, con los pesos de CH-302 y CH-303. La primera sílaba no empieza por `x` ni `r`.
3. Si es verbo, la última vocal es `a` (CH-304).
4. A veces, una consonante como `ch` se vuelve `chh` o `ch'` (**laringalización**, CH-305).
5. La raíz pasa por los **filtros**. Si falla uno, se descarta y se sortea otra (hasta 1.000 veces).

**Los filtros.**

| Regla | Se descarta la raíz si… | Ejemplo descartado |
|---|---|---|
| CH-201 a CH-209 | no cumple las reglas de sonidos de la raíz | `tissa` (s + s) |
| CH-306 | es una palabra real del aymara, o empieza igual que palabras reales | `sara` |
| CH-307 | es igual a un sufijo o a una palabra gramatical | `muqi`, `naya` |
| CH-308 | termina en un sufijo de 3 letras o más | `kurnaka` (-naka) |
| CH-309 | ya la tiene otro sentido | — |
| CH-310 | es otra raíz + un sufijo, o al revés | `kirpuru` (= `kirpu` + `-ru`) |

**El filtro de palabras reales (CH-306).** Usa `palabras_reales.txt`: una palabra aymara real por línea. Cuenta cuántas palabras reales **empiezan** por la raíz:
- Modo `"guia"`: se descarta si hay 1 o más.
- Modo `"corpus"`: se descarta si hay 5 o más.

El modo se elige en la celda de parámetros (`MODO_FILTRO`). Si el archivo falta, se avisa y las raíces se generan sin este filtro.

**Léxico de ejemplo.** Las raíces de §4.1 ya están fijadas por la especificación. No se regeneran: se revisan con los mismos filtros (P-05). Con `lexico_sin_raices.csv` sí se generan.

**Qué entra.** `LEXICO`, `palabras_reales.txt`, el bloque `generador_raices.yaml`. **Qué sale.** `salida/lexico.csv`.

**Ejemplo** ("ir", verbo): 2 sílabas, [CVC, CV] → `tis` + `qa` → `tisqa`. Pasa todos los filtros:
- B: `tisq-th-wa` → `tisqthwa` — ir-1>3.SIMP-AFF — "voy".
- A: `tisq-ph-yi` → `tisqphyi`.

In [ ]:
# 6.1 · Palabras reales (CH-306) y listas de formas prohibidas (CH-307, CH-308, CH-310).
import bisect

GR = CONFIG["generador_raices"]
FR = GR["filtro_real"]
LETRAS = "".join(sorted(set("".join(GRAFEMAS))))
RE_NO_LETRA = re.compile(f"[^{re.escape(LETRAS + APOSTROFO)}]+")


def tokens_reales(texto):
    """Saca las palabras de un texto, normalizadas como las raíces.

    Parámetros:
        texto: contenido de palabras_reales.txt (o cualquier texto aymara).
    Devuelve:
        Conjunto de palabras distintas.
    Reglas: CH-306.2 (NFC, minúsculas, apóstrofos de CH-102; se corta en todo lo que no es letra ni ').
    Ejemplo:
        >>> sorted(tokens_reales("Sara-ña, ch’uxña"))
        ["ch'uxña", 'sara', 'ña']
    """
    return set(RE_NO_LETRA.split(normalizar(texto))) - {""}


RUTA_REALES = ruta(FR["archivo"])
FILTRO_REAL_ACTIVO = RUTA_REALES.exists()
REALES = tokens_reales(RUTA_REALES.read_text(encoding="utf-8")) if FILTRO_REAL_ACTIVO else set()
REALES_ORDENADAS = sorted(REALES)
UMBRAL_REAL = FR["umbral_prefijo"]


def n_prefijo(r):
    """Cuenta las palabras reales distintas que empiezan por r (r incluida).

    Hace la misma cuenta que el trie de CH-306.4, con búsqueda binaria en la lista ordenada.

    Parámetros:
        r: raíz candidata.
    Devuelve:
        n(r), un entero.
    Reglas: CH-306.3, CH-306.4.
    Ejemplo:
        >>> n_prefijo("lapi")
        3
    """
    return bisect.bisect_left(REALES_ORDENADAS, r + "￿") - bisect.bisect_left(REALES_ORDENADAS, r)


def motivo_real(r, modo=None):
    """Dice si el filtro de palabras reales descarta una raíz.

    Parámetros:
        r: raíz candidata.
        modo: "guia" o "corpus". Si se omite, el de MODO_FILTRO.
    Devuelve:
        Texto con el motivo y la regla, o None si la raíz pasa.
    Reglas: CH-306.3 (guía: n(r) ≥ 1; corpus: n(r) ≥ 5; siempre se descarta una palabra real).
    Ejemplo:
        >>> motivo_real("sara")
        'CH-306: es una palabra real'
    """
    modo = modo or FR["modo"]
    if FR["descartar_si_igual"] and r in REALES:
        return "CH-306: es una palabra real"
    n = n_prefijo(r)
    if n >= UMBRAL_REAL[modo]:
        return f"CH-306: {n} palabras reales empiezan por {r} (modo {modo}, umbral {UMBRAL_REAL[modo]})"
    return None


# CH-307: formas de sufijos (también el alomorfo de REM.C) y de palabras gramaticales, en B y A.
FORMAS_SUFIJOS = {f for m in MORFEMAS.values()
                  for f in [m["b"], m["a"]] + ([m["alomorfo"]["b"], m["alomorfo"]["a"]] if m.get("alomorfo") else [])}
PROHIBIDAS_IGUALES = FORMAS_SUFIJOS | {p[k] for p in GRAMATICALES.values() for k in ("b", "a")}
# CH-308: sufijos de 3 grafemas o más. El "¨" no es grafema: "¨ta" tiene 2.
TERMINACIONES_PROHIBIDAS = {f for f in FORMAS_SUFIJOS
                            if len(segmentar(f.replace("¨", ""))) >= GR["terminacion_prohibida_min_grafemas"]}
# CH-310: sufijos de 1 grafema o más. Se dejan fuera los que llevan "¨": una raíz no tiene vocal larga (CH-209),
# así que r1 + "¨…" nunca puede ser otra raíz.
SUFIJOS_CH310 = {f for f in FORMAS_SUFIJOS if "¨" not in f}

if LEXICO_PRODUCCION and not FILTRO_REAL_ACTIVO:   # CH-306: sin la lista, las raíces podrían ser palabras reales
    raise FileNotFoundError(f"Falta {RUTA_REALES}. Con el léxico de producción es obligatorio (CH-306).")
if FILTRO_REAL_ACTIVO:
    print(f"{RUTA_REALES.name}: {len(REALES):,} palabras distintas · modo {FR['modo']!r} "
          f"(se descarta si n(r) ≥ {UMBRAL_REAL[FR['modo']]}).")
else:
    print(f"AVISO: falta {RUTA_REALES.name}. Las raíces se generan SIN el filtro CH-306 (pendiente, ver P-04).")
print("CH-308, terminaciones prohibidas:", " ".join(sorted(TERMINACIONES_PROHIBIDAS)))

In [ ]:
# 6.2 · Filtros de una raíz y generador (§3.1: CH-301 a CH-310).
import random

OCLUSIVAS_SIMPLES = {serie[0]: serie for serie in PUNTO_DE_ARTICULACION.values()}   # p → [p, ph, p']
assert all(len(s) == 3 for s in OCLUSIVAS_SIMPLES.values()), "alfabeto.yaml: cada serie es [simple, aspirada, glotalizada]"


def motivos_rechazo(r, aceptadas=()):
    """Revisa una raíz candidata con todos los filtros.

    Parámetros:
        r: raíz candidata.
        aceptadas: raíces ya aceptadas (conjunto o diccionario).
    Devuelve:
        Lista de motivos, cada uno con su regla. Vacía si la raíz se acepta.
    Reglas: CH-201 a CH-209 (validar_raiz), CH-306, CH-307, CH-308, CH-309, CH-310.
    Ejemplo:
        >>> motivos_rechazo("tisqa")
        []
        >>> motivos_rechazo("kirpuru", {"kirpu"})
        ['CH-310: kirpuru = kirpu + -ru']
    """
    motivos = validar_raiz(r)
    if FILTRO_REAL_ACTIVO and (m := motivo_real(r)):
        motivos.append(m)
    if r in PROHIBIDAS_IGUALES:
        motivos.append("CH-307: igual a un sufijo o a una palabra gramatical")
    motivos += [f"CH-308: termina en -{s}" for s in sorted(TERMINACIONES_PROHIBIDAS) if r.endswith(s)]
    if r in aceptadas:
        motivos.append("CH-309: otra entrada ya tiene esta raíz")
    for s in sorted(SUFIJOS_CH310):
        if r.endswith(s) and r[:-len(s)] in aceptadas:
            motivos.append(f"CH-310: {r} = {r[:-len(s)]} + -{s}")
        if r + s in aceptadas:
            motivos.append(f"CH-310: {r + s} = {r} + -{s}")
    return motivos


def sortear(pesos, rng):
    """Elige una clave de un diccionario {opción: peso}. Los pesos no tienen que sumar 1.

    Parámetros:
        pesos: diccionario de pesos.
        rng: generador random.Random.
    Devuelve:
        La opción elegida.
    Reglas: CH-301, CH-302, CH-303 (renormalizar = no hace falta que sumen 1).
    Ejemplo:
        >>> sortear({"a": 1.0}, random.Random(0))
        'a'
    """
    return rng.choices(list(pesos), weights=list(pesos.values()))[0]


def laringalizar(silabas, rng):
    """Paso de laringalización: a veces una oclusiva simple se vuelve aspirada o glotalizada.

    Parámetros:
        silabas: lista de sílabas [inicio, vocal, coda]; inicio y coda pueden ser "".
        rng: generador random.Random.
    Devuelve:
        Las sílabas (cambiadas o no), o None si la raíz se descarta (regla 4).
    Reglas: CH-305 (1: probabilidad 0,30; 2: aspirada o glotalizada; 3: armonía; 4: descarte), CH-205.
    Ejemplo:
        lamcha → se elige ch, glotalizada → lamch'a.
    """
    L = GR["laringalizacion"]
    if rng.random() >= L["probabilidad"]:
        return silabas
    candidatas = [s[0] for s in silabas if s[0] in OCLUSIVAS_SIMPLES]
    if not candidatas:
        return silabas
    simple = rng.choice(candidatas)
    serie = OCLUSIVAS_SIMPLES[simple]
    nueva = serie[1] if sortear(L["tipo"], rng) == "aspirada" else serie[2]
    if any(s[2] in serie for s in silabas):                      # 4: misma serie en coda → descarte
        return None
    return [[nueva if s[0] == simple else s[0], s[1], s[2]] for s in silabas]   # 3: armonía en los inicios


def candidata(cat, rng):
    """Fabrica una raíz candidata, sin filtrar.

    Parámetros:
        cat: categoría del léxico (CH-401).
        rng: generador random.Random.
    Devuelve:
        La raíz (cadena), o None si la laringalización la descarta.
    Reglas: CH-301 (sílabas), CH-302 (plantillas), CH-303 (pesos; sin x ni r en la 1.ª sílaba),
        CH-304 (verbos en -a), CH-305 (laringalización).
    Ejemplo:
        >>> candidata("V_MOV", random.Random(1))   # una raíz de 2 o 3 sílabas que termina en a
    """
    n = int(sortear(GR["silabas"][cat], rng))
    inicio_1 = {c: p for c, p in GR["inicio"].items() if c not in GR["inicio_prohibido_primera_silaba"]}
    silabas = []
    for i in range(n):
        pos = "primera" if i == 0 else "ultima" if i == n - 1 else "intermedia"
        plantilla = sortear(GR["plantillas"][pos], rng)
        inicio = sortear(inicio_1 if i == 0 else GR["inicio"], rng) if plantilla.startswith("C") else ""
        vocal = sortear(GR["vocales"], rng)
        coda = sortear(GR["coda"], rng) if plantilla.endswith("VC") else ""
        silabas.append([inicio, vocal, coda])
    if cat in VERBOS:
        silabas[-1][1] = GR["vocal_final_verbos"]               # CH-304
    silabas = laringalizar(silabas, rng)
    return None if silabas is None else "".join("".join(s) for s in silabas)


def generar_raices(lexico, semilla=GR["semilla_raices"], previas=None):
    """Da una raíz a cada entrada del léxico, recorriéndolo por id.

    Las entradas que ya tenían raíz (previas) la conservan: así, añadir o quitar una fila no cambia
    las raíces de las demás (CH-309). Solo las entradas nuevas reciben una raíz nueva.

    Parámetros:
        lexico: DataFrame del léxico (columnas id y cat).
        semilla: semilla fija (CH-309).
        previas: {id: raíz} de una ejecución anterior con la misma VERSION_LEXICO (o None).
    Devuelve:
        (raíces, estadística). raíces = {id: raíz}. estadística = DataFrame con los candidatos
        probados y descartados por categoría, y la regla que más descarta.
    Lanza:
        RuntimeError si una entrada no consigue raíz en max_intentos (CH-306.5), o si una raíz previa
        ya no pasa los filtros (V31: hay que subir VERSION_LEXICO y regenerar).
    Reglas: §3.1 completo; CH-306.5 (aviso si se descarta más del 60 %), CH-309 (semilla, orden por id).
    Ejemplo:
        >>> generar_raices(LEXICO_EJEMPLO.head(2))[0]
        {'L0001': '…', 'L0002': '…'}
    """
    rng = random.Random(semilla)
    aceptadas, cuenta, previas = {}, {}, previas or {}
    orden = list(lexico.sort_values("id").itertuples())
    for e in orden:                                                  # 1. las raíces guardadas se conservan
        r = previas.get(e.id)
        if r:
            m = motivos_rechazo(r, aceptadas)
            if m:
                raise RuntimeError(f"V31 / CH-309: la raíz guardada {r!r} de {e.id} ya no pasa los filtros: {m}. "
                                   "Sube VERSION_LEXICO para regenerar el léxico.")
            aceptadas[r] = e.id
    for e in orden:                                                  # 2. las entradas nuevas reciben raíz
        if e.id in previas:
            continue
        c = cuenta.setdefault(e.cat, {"probadas": 0, "descartadas": 0, "motivos": {}})
        for _ in range(FR["max_intentos"]):
            r = candidata(e.cat, rng)
            c["probadas"] += 1
            motivos = ["CH-305: oclusiva del mismo punto en coda"] if r is None else motivos_rechazo(r, aceptadas)
            if not motivos:
                aceptadas[r] = e.id
                break
            c["descartadas"] += 1
            regla = motivos[0].split(":")[0]
            c["motivos"][regla] = c["motivos"].get(regla, 0) + 1
        else:
            raise RuntimeError(f"CH-306.5: {e.id} ({e.cat}) sin raíz tras {FR['max_intentos']} intentos.")
    estad = pd.DataFrame([{"categoría": cat, "probadas": c["probadas"], "descartadas": c["descartadas"],
                           "% descartadas": round(100 * c["descartadas"] / c["probadas"], 1),
                           "regla que más descarta": max(c["motivos"], key=c["motivos"].get) if c["motivos"] else "—"}
                          for cat, c in cuenta.items()])
    return {i: r for r, i in aceptadas.items()}, estad


print("Ejemplos de candidatas (sin filtrar):",
      ", ".join(str(candidata(c, random.Random(k))) for k, c in enumerate(CATEGORIAS)))
for r, acept in [("tisqa", {}), ("sara", {}), ("muqi", {}), ("kurnaka", {}), ("kirpuru", {"kirpu"})]:
    print(f"{r:8} → {motivos_rechazo(r, acept) or 'aceptada'}")

In [ ]:
# 6.3 · Raíces del léxico y escritura de lexico.csv (CH-309: las raíces guardadas no cambian).
RUTA_LEXICO = ruta("lexico.csv")
RUTA_VERSION = ruta("lexico_version.json")
ESTADO_LEXICO = {"version": VERSION_LEXICO, "origen": "produccion" if LEXICO_PRODUCCION else "ejemplo",
                 "modo": FR["modo"], "semilla": GR["semilla_raices"]}
if LEXICO_PRODUCCION:
    previas = {}
    antes = json.loads(RUTA_VERSION.read_text(encoding="utf-8")) if RUTA_VERSION.exists() and RUTA_LEXICO.exists() else None
    if antes and antes.get("origen") == "produccion":
        if antes["version"] != VERSION_LEXICO:
            print(f"VERSION_LEXICO {antes['version']} → {VERSION_LEXICO}: se regeneran todas las raíces (CH-309).")
        elif (antes["modo"], antes["semilla"]) != (FR["modo"], GR["semilla_raices"]):
            raise ValueError(f"CH-309: el léxico {VERSION_LEXICO} se hizo con modo {antes['modo']!r}. Cambiar el modo o la semilla "
                             "cambia las raíces: sube VERSION_LEXICO si de verdad quieres un léxico nuevo.")
        else:
            viejo = leer_csv(RUTA_LEXICO)
            cat_ahora = dict(zip(LEXICO["id"], LEXICO["cat"]))
            previas = {i: r for i, c, r in zip(viejo["id"], viejo["cat"], viejo["raiz"]) if cat_ahora.get(i) == c}
    RAICES, ESTADISTICA = generar_raices(LEXICO, previas=previas)
    LEXICO["raiz"] = LEXICO["id"].map(RAICES)
    nuevas = len(LEXICO) - len(previas)
    print(f"Raíces: {len(previas):,} conservadas de {RUTA_LEXICO.name} y {nuevas:,} nuevas "
          f"(semilla {GR['semilla_raices']}, modo {FR['modo']!r}, léxico {VERSION_LEXICO}).")
    if nuevas:
        display(ESTADISTICA)
        for f in ESTADISTICA.itertuples():
            if f[4] > 100 * FR["aviso_si_rechazo_mayor_que"]:
                print(f"AVISO CH-306.5: en {f.categoría} se descartó el {f[4]} % de los candidatos.")
else:
    # P-05 (a): las raíces de §4.1 se conservan; se revisan en orden de id, como si se aceptaran una a una.
    aceptadas, fallas = set(), {}
    for f in LEXICO.sort_values("id").itertuples():
        if m := motivos_rechazo(f.raiz, aceptadas):
            fallas[f.id] = m
        aceptadas.add(f.raiz)
    if fallas:
        raise ValueError(f"Raíces de §4.1 que no pasan los filtros: {fallas}")
    print(f"Las {len(LEXICO)} raíces de §4.1 pasan todos los filtros (CH-201 a CH-310"
          f"{'' if FILTRO_REAL_ACTIVO else ', salvo CH-306: falta la lista'}).")

LEXICO["silabas"] = LEXICO["raiz"].map(lambda r: str(silabas(r)))
assert LEXICO["raiz"].is_unique, "CH-309: dos sentidos comparten raíz"
LEXICO.to_csv(RUTA_LEXICO, index=False, encoding="utf-8")
RUTA_VERSION.write_text(json.dumps(ESTADO_LEXICO, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"Escrito {RUTA_LEXICO} ({len(LEXICO)} filas) y {RUTA_VERSION.name}.")
display(LEXICO[["id", "cat", "es_m", "raiz", "silabas"]].head(25))

### Pruebas de la sección 6

- **CH-306 a CH-310:** los ejemplos de la especificación (`sara`, `kulla`, `lapi`, `tisqa`, `muqi`, `naya`, `kurnaka`, `pilampi`, `wisulu`, `chimuka`, `kirpuru`, `kirpullu`).
- **CH-301 a CH-305:** se fabrican miles de candidatas y se mira que tengan 2 o 3 sílabas, que los verbos acaben en `a` y que la laringalización cumpla CH-205.
- **CH-309:** con la misma semilla salen las mismas raíces, y nunca dos sentidos con la misma.

In [ ]:
# 6.4 · Pruebas de las raíces.
class PruebasRaices(unittest.TestCase):
    @unittest.skipUnless(FILTRO_REAL_ACTIVO, "falta palabras_reales.txt")
    def test_CH306_ejemplos(self):
        """CH-306: sara ✗ (real); kulla ✗ y lapi ✗ en modo guía (inicio de kullaka, lapisa); tisqa ✓."""
        self.assertIn("palabra real", motivo_real("sara", "guia"))
        self.assertIn("palabra real", motivo_real("sara", "corpus"))
        self.assertIsNotNone(motivo_real("kulla", "guia"))
        self.assertIsNotNone(motivo_real("lapi", "guia"))
        self.assertIsNone(motivo_real("tisqa", "guia"))
        self.assertIsNone(motivo_real("tisqa", "corpus"))

    @unittest.skipUnless(FILTRO_REAL_ACTIVO, "falta palabras_reales.txt")
    def test_CH306_umbral_corpus(self):
        """CH-306.3: en modo corpus, una raíz con menos de 5 palabras que empiezan por ella pasa."""
        n = n_prefijo("lapi")
        self.assertEqual(motivo_real("lapi", "corpus") is None, n < 5)

    @unittest.skipUnless(FILTRO_REAL_ACTIVO, "falta palabras_reales.txt")
    def test_CH306_gramaticales_modo_A(self):
        """CH-306 (v1.1.2): ninguna palabra gramatical del modo A es una palabra real (antes: mulu), y todas pasan el
        umbral del modo corpus. En modo guía, muqi, tilu y yuli no pasarían (P-14)."""
        for g in GRAMATICALES.values():
            with self.subTest(id=g["id"], a=g["a"]):
                self.assertNotIn(g["a"], REALES)
                self.assertIsNone(motivo_real(g["a"], "corpus"))
        self.assertEqual(GRAMATICALES["INT.QUE"]["a"], "sipu")

    def test_CH306_normalizacion(self):
        """CH-306.2: NFC, minúsculas, apóstrofo U+0027 y corte por lo que no es letra."""
        self.assertEqual(tokens_reales("Sara-ña, ch’uxña (t'ant'a)"), {"sara", "ña", "ch'uxña", "t'ant'a"})

    def test_CH307(self):
        """CH-307: muqi (pronombre 1, modo A) y naya (pronombre 1, modo B) ✗."""
        self.assertTrue(tiene_regla(motivos_rechazo("muqi"), "CH-307"))
        self.assertTrue(tiene_regla(motivos_rechazo("naya"), "CH-307"))

    def test_CH308_lista(self):
        """CH-308: la lista calculada desde morfemas.yaml es la de la especificación."""
        b = "naka taki kama layku tana tama ista istu itu ñäni pxa ska sma mpi".split()
        a = "sulu puli chiwi tiwqa kilu kiqu ulqa ulqu uqu thäqi mtu nchi jmi nku".split()
        self.assertEqual(TERMINACIONES_PROHIBIDAS, set(b + a))

    def test_CH308_ejemplos(self):
        """CH-308: kurnaka, pilampi, wisulu ✗; chimuka y chimula ✓."""
        for r in ["kurnaka", "pilampi", "wisulu"]:
            self.assertTrue(tiene_regla(motivos_rechazo(r), "CH-308"), r)
        for r in ["chimuka", "chimula"]:
            self.assertFalse(tiene_regla(motivos_rechazo(r), "CH-308"), r)

    def test_CH309_conserva_raices(self):
        """CH-309: con raíces previas, quitar o añadir una fila no cambia las raíces de las demás."""
        lex = LEXICO_EJEMPLO.drop(columns="raiz")
        r1, _ = generar_raices(lex)
        r2, _ = generar_raices(lex.drop(index=5), previas={i: r for i, r in r1.items() if i != "L0006"})
        self.assertEqual({i: r2[i] for i in r2}, {i: r1[i] for i in r2})
        nueva = lex.iloc[[12]].assign(id="L09999")
        r3, _ = generar_raices(pd.concat([lex, nueva]), previas=r1)
        self.assertEqual({i: r3[i] for i in r1}, r1)
        self.assertNotIn(r3["L09999"], r1.values())
        self.assertTrue(r3["L09999"].endswith("a"))                          # CH-304: es un verbo
        with self.assertRaises(RuntimeError):                                 # V31: una raíz previa que ya no vale
            generar_raices(lex, previas={**r1, "L0001": "sara"})

    def test_CH309_repetida(self):
        """CH-309: una raíz ya aceptada no se acepta otra vez."""
        self.assertTrue(tiene_regla(motivos_rechazo("tisqa", {"tisqa"}), "CH-309"))

    def test_CH310(self):
        """CH-310: con kirpu, se rechazan kirpuru (B -ru) y kirpullu (A -llu); y al revés."""
        self.assertTrue(tiene_regla(motivos_rechazo("kirpuru", {"kirpu"}), "CH-310"))
        self.assertTrue(tiene_regla(motivos_rechazo("kirpullu", {"kirpu"}), "CH-310"))
        self.assertTrue(tiene_regla(motivos_rechazo("kirpu", {"kirpuru"}), "CH-310"))
        self.assertFalse(tiene_regla(motivos_rechazo("kirpu", {"tisqa"}), "CH-310"))

    def test_CH301_a_CH305_candidatas(self):
        """CH-301 a CH-305: 4.000 candidatas por categoría (sin filtrar)."""
        for cat in CATEGORIAS:
            rng = random.Random(7)
            cands = [c for c in (candidata(cat, rng) for _ in range(4000)) if c is not None]
            dos = sum(silabas(c) == 2 for c in cands) / len(cands)
            with self.subTest(cat=cat):
                self.assertTrue(all(silabas(c) in (2, 3) for c in cands))                    # CH-301
                self.assertAlmostEqual(dos, GR["silabas"][cat][2], delta=0.04)                # CH-301: proporción
                self.assertTrue(all(RE_RAIZ.match(clases(segmentar(c))) for c in cands))      # CH-201, CH-302
                self.assertFalse(any(segmentar(c)[0] in ("x", "r") for c in cands))           # CH-303
                self.assertFalse(any(tiene_regla(validar_raiz(c), "CH-205") for c in cands))  # CH-305
                if cat in VERBOS:
                    self.assertTrue(all(c.endswith("a") for c in cands))                      # CH-304
                lar = sum(any(g in LARINGALIZADOS for g in segmentar(c)) for c in cands) / len(cands)
                self.assertTrue(0.10 < lar < 0.30, f"laringalizadas: {lar:.2f}")              # CH-305: p = 0,30

    def test_CH309_generador(self):
        """CH-309: el generador da una raíz válida y distinta a cada entrada, y es reproducible."""
        lex = LEXICO_EJEMPLO.drop(columns="raiz")
        r1, estad = generar_raices(lex)
        self.assertEqual(r1, generar_raices(lex)[0])
        self.assertNotEqual(r1, generar_raices(lex, semilla=1)[0])
        self.assertEqual(sorted(r1), sorted(lex["id"]))
        self.assertEqual(len(set(r1.values())), len(r1))
        for i, r in r1.items():
            otras = {x for j, x in r1.items() if j != i}
            self.assertEqual(motivos_rechazo(r, otras), [], f"{i}: {r}")
        verbos = lex.loc[lex["cat"].isin(VERBOS), "id"]
        self.assertTrue(all(r1[i].endswith("a") for i in verbos))
        print(f"\n   (raíces de prueba para §4.1: {', '.join(list(r1.values())[:8])} …)", end=" ")


correr_pruebas(PruebasRaices, titulo="raíces (§3)")

## 7. Español

**Qué hace esta sección.** Escribe la oración **española** de una idea. El español sale de la idea, no de traducir la oración aymara. Así cada idea tiene una sola traducción (CH-1013).

**Qué necesita.**
- **Conjugaciones** de cada verbo: presente, pretérito ("fui"), futuro, participio ("ido") y gerundio ("yendo"). Salen de `conjugaciones.csv` (en `datos/` o en `data/lexicon/`). La celda 7.2 escribe un **borrador** de ese archivo con un conjugador automático; una persona debe revisarlo (PREGUNTAS.md, P-07).
- Las formas de *haber*, *estar*, *ser* y *tener*.

**Reglas principales.**

| Regla | Qué decide | Ejemplo |
|---|---|---|
| CH-1001 a CH-1005 | El tiempo español de cada tiempo aymara | SIMP + "ayer" → "fui"; REM.C → "he ido"; FUT → "iré" |
| CH-1006 | La persona del verbo | 4.ª persona → "nosotros" |
| CH-1007 | Pronombre sujeto solo si la oración aymara lo tiene | "Yo voy." / "Voy." |
| CH-1008 | Artículos y determinantes | "come pan", "la casa del maestro", "tengo una casa" |
| CH-1009 | "me / te / nos / le" delante del verbo | "Él me da pan." |
| CH-1010 | El género de las personas de 3.ª persona | "Él duerme." / "Ella duerme." |
| CH-1011 | El orden de las palabras | sujeto – verbo – OD – OI – lugar – compañía – tiempo |

**Dos léxicos** (PREGUNTAS.md, P-05).
- `LEX`: el léxico de la sección 6 (el que se usa para generar).
- `LEX_PRUEBA`: el de ejemplo (§4.1) más el **léxico de validación** de §12.1 (ids `LV01`–`LV18`, con raíces reales del aymara). Solo sirve para los 40 casos de prueba: nunca para generar datos.

**Qué entra.** Una idea y un léxico. **Qué sale.** La oración española.

**Ejemplo** (idea de CH-1101):
- B: `Simllaxa jaskhi pinqarana lisquru q'aspi tulphiwa.`
- A: `Simllaxu jaskhi pinqaramu lisqullu q'aspi tulphuyi.`
- ES: "La mujer le dio pan al niño en el mercado ayer."

In [ ]:
# 7.1 · Los dos léxicos de las oraciones y las tablas del español.
# §12.1: léxico de validación (raíces reales; solo pruebas, no pasa por los filtros CH-306 a CH-310).
CSV_LV = """id,cat,subcat,clase_sem,es_m,es_f,es_pl_m,es_pl_f,genero,contable,raiz
LV01,N_HUM,-,persona,niño,niña,niños,niñas,var,s,wawa
LV02,N_NOHUM,-,comida,pan,,panes,,m,n,t'ant'a
LV03,V_DITR,-,-,dar,,,,-,-,chura
LV04,N_HUM,-,persona,padre,,padres,,m,s,tata
LV05,N_NOHUM,-,lugar,pueblo,,pueblos,,m,s,marka
LV06,V_MOV,-,-,venir,,,,-,-,juta
LV07,V_MOV,-,-,ir,,,,-,-,sara
LV08,V_INTR,-,-,comer,,,,-,-,manq'a
LV09,V_TR,-,-,comer,,,,-,-,manq'a
LV10,N_NOHUM,-,animal,perro,,perros,,m,s,anu
LV11,N_NOHUM,-,vivienda,casa,,casas,,f,s,uta
LV12,N_HUM,-,persona,hermano,hermana,hermanos,hermanas,var,s,jila
LV13,N_HUM,-,persona,autoridad,,autoridades,,f,s,jilaqata
LV14,N_HUM,-,persona,mamá,,mamás,,f,s,mama
LV15,N_NOHUM,-,animal,zorro,,zorros,,m,s,tiwula
LV16,N_NOHUM,-,animal,ganso,,gansos,,m,s,wallata
LV17,V_INTR,-,-,encontrarse,,,,-,-,jikisi
LV18,V_TR,-,-,querer,,,,-,-,muna
"""


def tabla_lex(df):
    """Pasa un léxico (DataFrame) a un diccionario {id: fila}.

    Parámetros:
        df: DataFrame con las columnas de CH-402 (al menos cat, subcat, clase_sem, es_*, genero, contable, raiz).
    Devuelve:
        Diccionario {id: {columna: valor}}.
    Reglas: CH-402.
    Ejemplo:
        >>> tabla_lex(LEXICO_EJEMPLO)["L0013"]["raiz"]
        'tisqa'
    """
    return {f["id"]: f for f in df.to_dict("records")}


LEX = tabla_lex(LEXICO)                                                       # para generar (sección 6)
LEX_PRUEBA = tabla_lex(LEXICO_EJEMPLO) | tabla_lex(leer_csv(io.StringIO(CSV_LV)))   # P-05 (b): solo pruebas

REFL = ["me", "te", "se", "nos", "se"]
HABER = dict(pres="he has ha hemos han".split(), imp="había habías había habíamos habían".split())
ESTAR = "estoy estás está estamos están".split()
SER = "soy eres es somos son".split()
TENER = "tengo tienes tiene tenemos tienen".split()

DEM_ES = {"DEM.1": ("este", "esta", "estos", "estas"), "DEM.2": ("ese", "esa", "esos", "esas"),
          "DEM.3": ("aquel", "aquella", "aquellos", "aquellas")}
DEICT_ES = {"DEM.1": "aquí", "DEM.2": "ahí", "DEM.3": "allá"}                 # CH-916
NUM_ES = {"NUM.3": "tres", "NUM.4": "cuatro", "NUM.6": "seis", "NUM.10": "diez"}   # CH-1012
CLIT = {(1, 2): "te", (2, 1): "me", (3, 1): "me", (3, 2): "te", (3, 4): "nos"}   # CH-1009
PREGUNTA_ES = {"quien_suj": "Quién", "que_od": "Qué", "donde_all": "A dónde",
               "quien_oi": "A quién", "donde_estar": "Dónde"}                  # CH-1011

print(f"LEX: {len(LEX)} entradas · LEX_PRUEBA: {len(LEX_PRUEBA)} entradas.")


### Capa de realismo (R23)

**Qué hace.** Las clases de CH-406 son gruesas: `abstracto` reúne 776 nombres y `objeto` mezcla un libro con la lluvia. Con solo las clases salen oraciones gramaticales pero absurdas ("Dirán cuatro desarrollos", "Compré la lluvia"). Esta celda lee `data/lexicon/realismo.yaml` y define la regla **R23**, que afina palabra por palabra:
- qué abstractos acepta cada verbo como objeto directo, como causa (CAUSE) o en "Esa es una…";
- qué documentos, sonidos y fenómenos naturales acepta cada verbo;
- qué instrumentos (COM) sirven para cada verbo o marco, y qué casos de lugar admite cada verbo de movimiento;
- qué adjetivos modifican a cada abstracto;
- con qué verbos un objeto animal lleva "a" en español ("Cuido al perro").

Los abstractos añadidos automáticamente (nota `AUTO`) salen del muestreo, salvo una lista que se conserva. **No cambia ninguna raíz** (CH-309): solo decide qué ideas son válidas. Con el léxico de ejemplo (§4.1) no hace nada.


In [ ]:
# 7.1b · Capa de realismo (R23): data/lexicon/realismo.yaml.
# Afina, palabra por palabra, qué combinaciones de palabras se generan. No cambia raíces (CH-309).
RUTA_REALISMO = ruta("realismo.yaml")
REALISMO_ACTIVO = LEXICO_PRODUCCION and RUTA_REALISMO.exists()
EXCLUIDAS = set()                  # ids que no se muestrean (abstractos AUTO no conservados, inalcanzables)
OD_ABSTRACTO = {}                  # {id verbo: ids de abstractos aceptados como OD}
CONCRETOS = {}                     # {(id verbo, función, clase): ids aceptados}
FENOMENOS = set()                  # ids de los fenómenos naturales
FENOMENO_OK = {}                   # {(id verbo, función): ids de fenómenos aceptados}
INSTRUMENTOS = {}                  # {id verbo: ids aceptados como instrumento (COM)}
CASOS_MOV = {}                     # {id verbo: casos de lugar aceptados}
CAUSA_ABS = set()                  # abstractos aceptados con CAUSE
SER_DEM_ABS = set()                # abstractos aceptados en "Esa es una…"
ADJ_ABS = {}                       # {id abstracto: ids de adjetivos aceptados}
A_PERSONAL = set()                 # ids de verbos con "a" ante un OD animal (solo español)
VERBOS_PARA = {}                   # {id nombre: ids de verbos que lo admiten} (para la cobertura, CH-404)
NO_LUGAR, NOMBRES_UNICOS, ABS_CONTABLES = set(), set(), set()   # secciones 10, 11 y 12
COMPANIA, COMPANIA_ANIMAL, CAUSA_SIN, CAUSA_PERSONA = set(), set(), set(), set()   # secciones 13 y 14
BEN_VERBOS, ADJ_EXCLUIDOS = None, set()                         # secciones 15 y 16 (None = sin límite)


def es_produccion(i):
    """True si el id es del léxico de producción (L + 5 cifras o más). R23 solo se aplica a esos ids."""
    return bool(re.fullmatch(r"L\d{5,}", i or ""))


if REALISMO_ACTIVO:
    RCFG = yaml.safe_load(RUTA_REALISMO.read_text(encoding="utf-8"))
    _CONJ = RCFG["conjuntos"]
    _NOMBRES, _VERBOS, _ADJS = {}, {}, {}
    for _i, _e in LEX.items():
        if _e["cat"] in NOMBRES:
            _NOMBRES.setdefault(_e["es_m"], []).append(_i)
        elif _e["cat"] in VERBOS:
            _VERBOS.setdefault(_e["es_m"], []).append(_i)
        elif _e["cat"] == "ADJ":
            _ADJS.setdefault(_e["es_m"], []).append(_i)
    _FALTAN = set()

    def _palabras(lista, ctx=""):
        """Expande una lista con "@conjunto" a un conjunto de formas es_m."""
        out = set()
        for x in [lista] if isinstance(lista, str) else lista:
            if x.startswith("@"):
                if x[1:] not in _CONJ:
                    _FALTAN.add(f"conjunto {x} ({ctx})")
                    continue
                out |= _palabras(_CONJ[x[1:]], x[1:])
            else:
                out.add(x)
        return out

    def _ids(lista, tabla, ctx, clase=None):
        """Ids de las formas de una lista (de nombres, verbos o adjetivos). Avisa de las que no existen."""
        out = set()
        for p in _palabras(lista, ctx):
            ids = [i for i in tabla.get(p, []) if clase is None or LEX[i]["clase_sem"] == clase]
            if not ids:
                _FALTAN.add(f"{p} ({ctx}{', clase ' + clase if clase else ''})")
            out.update(ids)
        return out

    def _verbo(v):
        if v not in _VERBOS:
            _FALTAN.add(f"verbo {v}")
        return _VERBOS.get(v, [])

    for v, lista in RCFG["od_abstracto"].items():
        for vid in _verbo(v):
            OD_ABSTRACTO[vid] = _ids(lista, _NOMBRES, f"od_abstracto.{v}", "abstracto")
    for v, funciones in RCFG["concretos_por_verbo"].items():
        for vid in _verbo(v):
            for f, por_clase in funciones.items():
                for cl, lista in por_clase.items():
                    CONCRETOS[(vid, f, cl)] = _ids(lista, _NOMBRES, f"concretos.{v}", cl)
    FENOMENOS = _ids("@fenomeno", _NOMBRES, "fenomeno")
    for f, por_verbo in RCFG["fenomenos"].items():
        for v, lista in por_verbo.items():
            for vid in _verbo(v):
                FENOMENO_OK[(vid, f)] = _ids(lista, _NOMBRES, f"fenomenos.{f}.{v}") & FENOMENOS
    _marco = lambda e: (re.search(r"marco (\w+)", e.get("notas", "")) or [None, None])[1]
    for k, lista in RCFG["instrumentos"].items():          # primero los marcos, después los verbos (mandan)
        if k.startswith("marco:"):
            ids = _ids(lista, _NOMBRES, k)
            for vid in (i for i, e in LEX.items() if e["cat"] in VERBOS and _marco(e) == k[6:]):
                INSTRUMENTOS[vid] = ids
    for k, lista in RCFG["instrumentos"].items():
        if not k.startswith("marco:"):
            for vid in _verbo(k):
                INSTRUMENTOS[vid] = _ids(lista, _NOMBRES, f"instrumentos.{k}")
    for v, casos in RCFG["casos_movimiento"].items():
        for vid in _verbo(v):
            CASOS_MOV[vid] = set(casos)
    CAUSA_ABS = _ids(RCFG["causa_abstracto"], _NOMBRES, "causa_abstracto", "abstracto")
    SER_DEM_ABS = _ids(RCFG["ser_demostrativo_abstracto"], _NOMBRES, "ser_demostrativo_abstracto", "abstracto")
    for conj, adjs in RCFG["adjetivos_abstracto"].items():
        a_ids = _ids(adjs, _ADJS, f"adjetivos_abstracto.{conj}")
        for n in _ids("@" + conj, _NOMBRES, conj, "abstracto"):
            ADJ_ABS.setdefault(n, set()).update(a_ids)
    for v in RCFG["a_personal_animal"]:
        A_PERSONAL.update(_verbo(v))
    NO_LUGAR = _ids(RCFG["no_lugar"], _NOMBRES, "no_lugar")
    NOMBRES_UNICOS = _ids(RCFG["nombres_unicos"], _NOMBRES, "nombres_unicos")
    ABS_CONTABLES = _ids(RCFG["abstractos_contables"], _NOMBRES, "abstractos_contables", "abstracto")
    COMPANIA = {vid for v in RCFG["compania"]["verbos"] for vid in _verbo(v)}
    COMPANIA_ANIMAL = {vid for v in RCFG["compania"]["animal"] for vid in _verbo(v)}
    CAUSA_SIN = {vid for v in RCFG["causa_sin_verbos"] for vid in _verbo(v)}
    CAUSA_PERSONA = {vid for v in RCFG["causa_persona_verbos"] for vid in _verbo(v)}
    BEN_VERBOS = {vid for v in RCFG["ben_verbos"] for vid in _verbo(v) if LEX[vid]["cat"] in ("V_TR", "V_DITR")}
    ADJ_EXCLUIDOS = _ids(RCFG["adjetivos_excluidos"], _ADJS, "adjetivos_excluidos")
    if _FALTAN:
        raise ValueError(f"realismo.yaml nombra {len(_FALTAN)} palabras que no están en el léxico: {sorted(_FALTAN)}")

    # Exclusiones: abstractos AUTO no conservados y abstractos que ninguna regla deja usar.
    conservar = _palabras(RCFG["abstractos_auto_conservados"])
    auto = {i for i, e in LEX.items() if "AUTO" in e.get("notas", "") and e["es_m"] not in conservar}
    alcanzables = set().union(*OD_ABSTRACTO.values(), CAUSA_ABS, SER_DEM_ABS)
    inalcanzables = {i for i, e in LEX.items() if e.get("clase_sem") == "abstracto" and i not in alcanzables} - auto
    # Adjetivos que solo modifican abstractos y que ninguna lista de la sección 8 usa: tampoco caben en ninguna idea.
    adj_con_uso = set().union(*ADJ_ABS.values()) if ADJ_ABS else set()
    adj_sin_uso = {i for i, e in LEX.items() if e["cat"] == "ADJ" and set(e["mod_clases"].split("|")) <= {"abstracto"}
                   and i not in adj_con_uso}
    # V26: un verbo sin objeto (V_INTR, V_MOV) cuya 3.ª persona (B raíz-i, A raíz-u) es igual a la raíz de un
    # predicado de "ser" (nombre de persona o adjetivo) da dos traducciones a la misma oración
    # ("Jupaxa ujsiwa." = "Ella es ayudante." y "Ella transita."). Sale el verbo.
    raices_predicado = {e["raiz"] for e in LEX.values() if e["cat"] in ("N_HUM", "ADJ")}
    verbos_ambiguos = {i for i, e in LEX.items() if e["cat"] in ("V_INTR", "V_MOV")
                       and {e["raiz"][:-1] + "i", e["raiz"][:-1] + "u"} & raices_predicado}
    EXCLUIDAS = auto | inalcanzables | adj_sin_uso | ADJ_EXCLUIDOS | verbos_ambiguos
    LEX = {i: e for i, e in LEX.items() if i not in EXCLUIDAS}
    # Para la cobertura (CH-404): qué verbos admiten cada abstracto o fenómeno (así se fuerza un verbo compatible).
    VERBOS_PARA = {}
    for vid, ids in OD_ABSTRACTO.items():
        for n in ids:
            VERBOS_PARA.setdefault(n, set()).add(vid)
    for (vid, f), ids in FENOMENO_OK.items():
        for n in ids:
            VERBOS_PARA.setdefault(n, set()).add(vid)
    print(f"Realismo (R23) activo: {len(auto)} abstractos AUTO, {len(inalcanzables)} abstractos y {len(adj_sin_uso)} "
          f"adjetivos sin uso posible quedan fuera del muestreo; LEX: {len(LEX)} entradas.")
    _es = LEXICO.set_index("id")["es_m"]
    if inalcanzables:
        print("  Abstractos sin uso:", ", ".join(sorted(_es[list(inalcanzables)])))
    if adj_sin_uso:
        print("  Adjetivos sin uso:", ", ".join(sorted(_es[list(adj_sin_uso)])))
    if verbos_ambiguos:
        print("  Verbos ambiguos con un nombre o adjetivo (V26):", ", ".join(sorted(_es[list(verbos_ambiguos)])))
    print(f"  {len(OD_ABSTRACTO)} verbos con objeto abstracto, {len(CONCRETOS)} restricciones de nombres concretos, "
          f"{len(FENOMENOS)} fenómenos, {len(INSTRUMENTOS)} verbos con instrumentos fijos, {len(CASOS_MOV)} verbos de "
          f"movimiento con casos fijos, {len(A_PERSONAL)} verbos con 'a' personal.")
else:
    print("Realismo (R23) inactivo: falta realismo.yaml o se usa el léxico de ejemplo.")


def nombre_ok(vid, funcion, e):
    """¿Puede el nombre `e` cumplir la función `funcion` ("od", "suj" u otra) del verbo `vid`?

    Reglas: R23 (realismo.yaml, secciones 2, 3 y 4). Sin R23 activo, o fuera del léxico de producción, siempre True.
    Ejemplo:
        >>> nombre_ok(id_de("comprar"), "od", LEX[id_de("lluvia")])
        False
    """
    if not REALISMO_ACTIVO or not es_produccion(e["id"]):
        return True
    if e["id"] in FENOMENOS:
        return e["id"] in FENOMENO_OK.get((vid, funcion), ())
    if funcion == "od" and e["clase_sem"] == "abstracto":
        return e["id"] in OD_ABSTRACTO.get(vid, ())
    permitidos = CONCRETOS.get((vid, funcion, e["clase_sem"]))
    return permitidos is None or e["id"] in permitidos


def adj_realista(e, a):
    """¿Puede el adjetivo `a` modificar al nombre `e`? (R23, sección 8: solo restringe a los abstractos)."""
    if not REALISMO_ACTIVO or not es_produccion(e["id"]) or e["clase_sem"] != "abstracto":
        return True
    return a["id"] in ADJ_ABS.get(e["id"], ())


def instrumento_ok(vid, e):
    """¿Puede `e` ser instrumento (COM) del verbo `vid`? (R23, sección 5; los fenómenos nunca)."""
    if not REALISMO_ACTIVO or not es_produccion(e["id"]):
        return True
    return e["id"] not in FENOMENOS and (vid not in INSTRUMENTOS or e["id"] in INSTRUMENTOS[vid])


def casos_mov(vid):
    """Casos de lugar que admite un verbo de movimiento (R23, sección 6)."""
    return CASOS_MOV.get(vid, {"ALL", "ABL", "UNTIL"}) if REALISMO_ACTIVO else {"ALL", "ABL", "UNTIL"}


def errores_realismo(I, lex):
    """Revisa una idea con la capa de realismo (R23).

    Parámetros:
        I: la idea. lex: léxico.
    Devuelve:
        Lista de errores "R23: …". Vacía si la idea es realista o si R23 no está activo.
    Reglas: R23 (realismo.yaml); P-13 (la compañía animal solo con verbos que aceptan sujetos animales).
    """
    if not REALISMO_ACTIVO:
        return []
    E = []
    mal = lambda t: E.append(f"R23: {t}")
    fila = lambda i: lex.get(i) if es_produccion(i) else None

    def frases(np, funcion):
        """(fila del núcleo, frase) de una frase nominal y de su poseedor."""
        out = []
        if isinstance(np, dict) and fila(np.get("n")):
            out.append((fila(np["n"]), np, funcion))
            if np.get("poseedor"):
                out += frases(np["poseedor"], "poseedor")
        return out

    S, P = I.get("sujeto") or {}, I.get("pnom") or {}
    todas = frases(S.get("np"), "suj") + [x for np in S.get("coord") or [] for x in frases(np, "suj")]
    todas += frases(I.get("od"), "od") + [x for np in I.get("od_coord") or [] for x in frases(np, "od")]
    for k in ("oi", "lugar", "comcausa"):
        todas += frases((I.get(k) or {}).get("np"), k)
    todas += frases(P.get("np"), "pnom")
    for e, np, funcion in todas:                                       # adjetivos (sección 8)
        a = fila(np.get("adj"))
        if a and not adj_realista(e, a):
            mal(f"'{a['es_m']}' no modifica a '{e['es_m']}'")
        varios = np.get("plural") or np.get("num") or (funcion == "suj" and S.get("plural") and S.get("expreso") == "nombre")
        if e["id"] in NOMBRES_UNICOS and (varios or any(np.get(k) for k in ("adj", "dem", "poss", "poseedor"))):
            mal(f"'{e['es_m']}' es único: sin plural, numeral ni modificadores")          # sección 11
        if e["clase_sem"] == "abstracto" and e["id"] not in ABS_CONTABLES and varios:   # sección 12
            mal(f"'{e['es_m']}' no se cuenta: va en singular")
        if funcion in ("lugar", "pnom") and e["id"] in NO_LUGAR and I.get("predicado") in ("verbo", "estar", "tener"):
            mal(f"'{e['es_m']}' no es un lugar")                                          # sección 10
    if I.get("predicado") == "ser" and S.get("plural") and S.get("expreso") == "demostrativo":
        e = fila((P.get("np") or {}).get("n"))
        if e and (e["id"] in NOMBRES_UNICOS or (e["clase_sem"] == "abstracto" and e["id"] not in ABS_CONTABLES)):
            mal(f"'{e['es_m']}' no va en plural ('Esas son …')")
    if P.get("adj") and fila(P["adj"]):
        for e, np, funcion in todas:
            if funcion == "suj" and not adj_realista(e, fila(P["adj"])):
                mal(f"'{fila(P['adj'])['es_m']}' no se dice de '{e['es_m']}'")
    pred = I.get("predicado")
    if pred == "verbo":
        v = fila(I.get("verbo"))
        if v is None:
            return E
        vid = v["id"]
        for e, np, funcion in todas:
            if funcion in ("suj", "od") and not nombre_ok(vid, funcion, e):
                mal(f"'{e['es_m']}' no vale como {funcion} de '{v['es_m']}'")
            elif funcion not in ("suj", "od") and e["id"] in FENOMENOS:
                mal(f"el fenómeno '{e['es_m']}' no va en {funcion}")
        Lg = I.get("lugar") or {}
        if v["cat"] == "V_MOV" and Lg.get("caso") and Lg["caso"] not in casos_mov(vid):
            mal(f"'{v['es_m']}' no admite {Lg['caso']}")
        od = fila((I.get("od") or {}).get("n"))
        if Lg.get("caso") == "LOC" and od and od["clase_sem"] in ("lugar", "vivienda"):
            mal(f"'{od['es_m']}' ya es un lugar: la oración no lleva otro lugar (LOC)")
        if (I.get("oi") or {}).get("funcion") == "BEN" and (I.get("objeto_persona", 3) != 3 or vid not in BEN_VERBOS):
            mal(f"'{v['es_m']}' no lleva beneficiario (BEN) aquí")                           # sección 15
        # Clase del sujeto, para la compañía y la causa (los pronombres y las personas 1, 2 y 4 son persona).
        if S.get("expreso") in ("nombre", "coordinado"):
            clase_suj = {e["clase_sem"] for e, np, f in todas if f == "suj" and np in [S.get("np")] + (S.get("coord") or [])}
        elif S.get("expreso") == "pronombre" or S.get("persona") != 3:
            clase_suj = {"persona"}
        else:
            clase_suj = clases_de(v, "suj_clases") or {"persona"}
        C = I.get("comcausa") or {}
        c = fila((C.get("np") or {}).get("n"))
        if c and C.get("caso") == "COM":
            instrumento = c["clase_sem"] in (clases_de(v, "instr_clases") or set())
            if instrumento:
                if not instrumento_ok(vid, c):
                    mal(f"'{c['es_m']}' no es instrumento de '{v['es_m']}'")
                if C["np"].get("plural") or C["np"].get("num"):
                    mal("el instrumento va en singular")
            elif vid not in COMPANIA:                                                  # sección 13
                mal(f"'{v['es_m']}' no lleva compañía (COM)")
            elif c["clase_sem"] == "animal" and (vid not in COMPANIA_ANIMAL or clase_suj != {"persona"}):
                mal(f"compañía animal con '{v['es_m']}'")
        if c and C.get("caso") == "CAUSE":                                             # secciones 7 y 14
            if c["clase_sem"] == "abstracto" and c["id"] not in CAUSA_ABS:
                mal(f"'{c['es_m']}' no vale como causa")
            if clase_suj != {"persona"} or vid in CAUSA_SIN:
                mal(f"causa (CAUSE) con '{v['es_m']}' y sujeto {sorted(clase_suj)}")
            if c["clase_sem"] == "persona" and (vid not in CAUSA_PERSONA or C["np"].get("plural") or C["np"].get("num")):
                mal(f"causa persona ('por {c['es_m']}') con '{v['es_m']}'")
    else:
        for e, np, funcion in todas:
            if e["id"] in FENOMENOS and not (pred == "ser" and funcion == "pnom"):
                mal(f"el fenómeno '{e['es_m']}' no va con '{pred}'")
            if pred == "ser" and funcion == "pnom" and e["clase_sem"] == "abstracto" and e["id"] not in SER_DEM_ABS:
                mal(f"'{e['es_m']}' no vale en 'Esa es una …'")
    return E


def id_de(es_m, cat=None):
    """Id de una palabra del léxico de producción por su forma española (para pruebas y revisiones)."""
    return next(i for i, e in LEX.items() if e["es_m"] == es_m and es_produccion(i) and (cat is None or e["cat"] == cat))

### Conjugaciones del español (P-07)

El español necesita cinco formas de cada verbo: **presente**, **pretérito** ("fui"), **futuro**, **participio** ("ido") y **gerundio** ("yendo"). Están en `datos/conjugaciones.csv`.

**De dónde salen.** Esta celda las calcula con un **conjugador automático**. Funciona así:
1. Conjuga como un verbo **regular** (-ar, -er, -ir): `comprar` → `compro`, `compré`, `compraré`.
2. Aplica los **cambios de vocal** de una lista de verbos comunes: `dormir` → `duermo`, `durmió`.
3. Aplica los **cambios de escritura**, que siguen reglas fijas: `buscar` → `busqué`, `conocer` → `conozco`, `leer` → `leyó`.
4. Usa una **tabla de irregulares**: `ir`, `ser`, `tener`, `hacer`… Sus compuestos se conjugan igual (`mantener` como `tener`).

No se usa una biblioteca: la más completa (`verbecc`) necesita instalar scikit-learn, scipy y lxml. Un conjugador propio no necesita nada y da siempre el mismo resultado.

**Es un borrador.** Un conjugador automático se puede equivocar con un irregular que no está en la tabla. Por eso cada fila nueva lleva `revisado = n`. Cuando una persona la revisa, cambia el valor a `s`.
- Si el archivo ya existe, **solo se añaden los verbos que faltan**. Las filas que ya están (revisadas o corregidas a mano) no se tocan nunca.
- Si quedan filas sin revisar, se avisa, sin detener el notebook. El informe final dice cuántas son.

**Formato** (una fila por verbo y tiempo):

| infinitivo | tiempo | p1sg | p2sg | p3sg | p1pl | p3pl | reflexivo | revisado |
|---|---|---|---|---|---|---|---|---|
| ir | presente | voy | vas | va | vamos | van | n | n |
| ir | participio | ido | | | | | n | n |

- `tiempo` es `presente`, `preterito`, `futuro`, `participio` o `gerundio`. El participio y el gerundio van en la columna `p1sg`.
- `p2sg` es "tú" y `p3pl` sirve para "ellos" y "ustedes" (CH-1006).
- `reflexivo = s` en verbos como `encontrarse`. Las formas van sin el pronombre: "me", "te", "se" los pone la oración.

In [ ]:
# 7.2 · Conjugaciones (P-07): borrador automático en datos/conjugaciones.csv.
TERMINACIONES = {
    "ar": {"presente": ["o", "as", "a", "amos", "an"], "preterito": ["é", "aste", "ó", "amos", "aron"]},
    "er": {"presente": ["o", "es", "e", "emos", "en"], "preterito": ["í", "iste", "ió", "imos", "ieron"]},
    "ir": {"presente": ["o", "es", "e", "imos", "en"], "preterito": ["í", "iste", "ió", "imos", "ieron"]},
}
FUTURO = ["é", "ás", "á", "emos", "án"]
PRET_FUERTE = ["e", "iste", "o", "imos", "ieron"]          # tuve, tuviste, tuvo… (dije → dijeron)
FUERTES = (0, 1, 2, 4)                                      # personas con la vocal tónica en la raíz

# Cambio de vocal de la raíz. "ie" / "ue": solo en presente. "i", "ie-i", "ue-u": también en el pretérito
# (3sg y 3pl) y en el gerundio, con e → i u o → u.
CAMBIOS = {
    "ie": "pensar empezar comenzar cerrar despertar sentar calentar negar nevar gobernar recomendar "
          "atravesar confesar fregar regar temblar tropezar merendar apretar encerrar entender perder "
          "encender defender tender atender extender pretender querer",
    "ue": "contar encontrar mostrar demostrar recordar costar volar sonar soñar probar comprobar aprobar "
          "almorzar acostar colgar rogar tostar mover volver llover doler morder poder soler torcer cocer "
          "resolver oler",
    "i": "pedir servir repetir seguir conseguir perseguir vestir medir elegir corregir competir despedir "
         "impedir rendir gemir",
    "ie-i": "sentir mentir preferir divertir convertir herir hervir advertir sugerir consentir referir",
    "ue-u": "dormir morir",
    "u-ue": "jugar",
}
CAMBIO_DE = {v: tipo for tipo, lista in CAMBIOS.items() for v in lista.split()}
VOCALES_CAMBIO = {"ie": ("e", "ie"), "ue": ("o", "ue"), "i": ("e", "i"), "ie-i": ("e", "ie"),
                  "ue-u": ("o", "ue"), "u-ue": ("u", "ue")}
DEBIL = {"i": ("e", "i"), "ie-i": ("e", "i"), "ue-u": ("o", "u")}
ACENTO = set("enviar confiar variar vaciar esquiar continuar actuar graduar evaluar situar acentuar".split())   # envío, continúo
# -cer / -cir tras vocal que NO hacen -zco.
SIN_ZC = {"hacer", "decir", "cocer", "mecer", "satisfacer"}

IRREGULARES = {
    "ser": {"presente": "soy eres es somos son", "preterito": "fui fuiste fue fuimos fueron"},
    "ir": {"presente": "voy vas va vamos van", "preterito": "fui fuiste fue fuimos fueron", "gerundio": "yendo"},
    "estar": {"presente": "estoy estás está estamos están", "pret_raiz": "estuv"},
    "haber": {"presente": "he has ha hemos han", "pret_raiz": "hub", "fut_raiz": "habr"},
    "tener": {"presente": "tengo tienes tiene tenemos tienen", "pret_raiz": "tuv", "fut_raiz": "tendr"},
    "venir": {"presente": "vengo vienes viene venimos vienen", "pret_raiz": "vin", "fut_raiz": "vendr",
              "gerundio": "viniendo"},
    "poner": {"p1sg": "pongo", "pret_raiz": "pus", "fut_raiz": "pondr", "participio": "puesto"},
    "hacer": {"p1sg": "hago", "preterito": "hice hiciste hizo hicimos hicieron", "fut_raiz": "har", "participio": "hecho"},
    "decir": {"presente": "digo dices dice decimos dicen", "pret_raiz": "dij", "fut_raiz": "dir",
              "participio": "dicho", "gerundio": "diciendo"},
    "traer": {"p1sg": "traigo", "pret_raiz": "traj"},
    "caer": {"p1sg": "caigo"},
    "salir": {"p1sg": "salgo", "fut_raiz": "saldr"},
    "valer": {"p1sg": "valgo", "fut_raiz": "valdr"},
    "oír": {"presente": "oigo oyes oye oímos oyen", "preterito": "oí oíste oyó oímos oyeron", "fut_raiz": "oir",
            "participio": "oído", "gerundio": "oyendo"},
    "reír": {"presente": "río ríes ríe reímos ríen", "preterito": "reí reíste rio reímos rieron", "fut_raiz": "reir",
             "participio": "reído", "gerundio": "riendo"},
    "ver": {"presente": "veo ves ve vemos ven", "preterito": "vi viste vio vimos vieron", "participio": "visto"},
    "dar": {"presente": "doy das da damos dan", "preterito": "di diste dio dimos dieron"},
    "saber": {"p1sg": "sé", "pret_raiz": "sup", "fut_raiz": "sabr"},
    "caber": {"p1sg": "quepo", "pret_raiz": "cup", "fut_raiz": "cabr"},
    "poder": {"pret_raiz": "pud", "fut_raiz": "podr", "gerundio": "pudiendo"},
    "querer": {"pret_raiz": "quis", "fut_raiz": "querr"},
    "andar": {"pret_raiz": "anduv"},
    "abrir": {"participio": "abierto"}, "cubrir": {"participio": "cubierto"}, "escribir": {"participio": "escrito"},
    "romper": {"participio": "roto"}, "morir": {"participio": "muerto"}, "volver": {"participio": "vuelto"},
    "resolver": {"participio": "resuelto"}, "imprimir": {"participio": "impreso"},
}
# Bases cuyos compuestos se conjugan igual (mantener, suponer, devolver…). Las bases cortas (ver, dar, ir…)
# no se usan: "ver" daría formas falsas para "volver" o "mover".
DERIVABLES = ["tener", "poner", "venir", "traer", "salir", "volver", "mover", "hacer", "decir",
              "cubrir", "escribir", "abrir", "seguir", "sentir", "reír"]


def cambiar_ultima(raiz, de, a):
    """Cambia la última vocal `de` de la raíz por `a` (pens → piens)."""
    i = raiz.rfind(de)
    return raiz if i < 0 else raiz[:i] + a + raiz[i + 1:]


def conjugar(infinitivo):
    """Borrador de conjugación de un verbo español.

    Parámetros:
        infinitivo: el verbo, como en es_m ("ir", "encontrarse").
    Devuelve:
        {"presente": [5 formas], "preterito": [5], "futuro": [5], "participio": str, "gerundio": str,
         "reflexivo": bool}. Personas: 1sg, 2sg, 3sg, 1pl, 3pl (CH-1006).
    Lanza:
        ValueError si no termina en -ar, -er o -ir (ni en -ír).
    Reglas: §10 (conjugador), P-07 (borrador: alguien debe revisarlo).
    Ejemplo:
        >>> conjugar("dormir")["preterito"]
        ['dormí', 'dormiste', 'durmió', 'dormimos', 'durmieron']
    """
    reflexivo = infinitivo.endswith("rse")
    v = infinitivo[:-2] if reflexivo else infinitivo
    if v not in IRREGULARES and v not in CAMBIO_DE:
        base = next((b for b in DERIVABLES if v.endswith(b) and v != b), None)
        if base:                                            # compuesto: prefijo + formas de la base
            pref, f = v[:-len(base)], conjugar(base)
            return {t: ([pref + x for x in f[t]] if isinstance(f[t], list) else pref + f[t]) if t != "reflexivo"
                    else reflexivo for t in f}
    t = {"ír": "ir"}.get(v[-2:], v[-2:])
    if t not in TERMINACIONES:
        raise ValueError(f"P-07: '{infinitivo}' no termina en -ar, -er ni -ir"
                         f"{' (es una expresión de varias palabras: el léxico debe dar un solo verbo)' if ' ' in infinitivo else ''}.")
    r = v[:-2]
    pres = [r + e for e in TERMINACIONES[t]["presente"]]
    pret = [r + e for e in TERMINACIONES[t]["preterito"]]
    fut = [v + e for e in FUTURO]
    part = r + ("ado" if t == "ar" else "ido")
    ger = r + ("ando" if t == "ar" else "iendo")

    cambio = CAMBIO_DE.get(v)                               # cambio de vocal de la raíz
    if cambio:
        rf = cambiar_ultima(r, *VOCALES_CAMBIO[cambio])
        rf = "h" + rf if rf.startswith("ue") else rf        # oler → huelo
        for i in FUERTES:
            pres[i] = rf + TERMINACIONES[t]["presente"][i]
        if cambio in DEBIL:
            rd = cambiar_ultima(r, *DEBIL[cambio])
            pret[2], pret[4] = rd + "ió", rd + "ieron"
            ger = rd + "iendo"
    if v in ACENTO:                                         # envío, continúo
        for i in FUERTES:
            s = pres[i][:-len(TERMINACIONES[t]["presente"][i])]
            k = max(s.rfind("i"), s.rfind("u"))
            pres[i] = s[:k] + {"i": "í", "u": "ú"}[s[k]] + s[k + 1:] + TERMINACIONES[t]["presente"][i]

    # Cambios ortográficos (no son irregularidades: mantienen el sonido).
    p1 = pres[0][:-1]
    if v.endswith("guir"):
        pres[0] = p1[:-1] + "o"                             # sigo, distingo
    elif v.endswith(("ger", "gir")):
        pres[0] = p1[:-1] + "jo"                            # cojo, elijo
    elif v.endswith(("cer", "cir")) and v not in SIN_ZC:
        pres[0] = p1[:-1] + ("zco" if p1[-2] in "aeiou" else "zo")   # conozco, venzo
    if t == "ar" and v.endswith(("car", "gar", "zar")):
        pret[0] = {"c": r[:-1] + "qué", "g": r + "ué", "z": r[:-1] + "cé"}[r[-1]]   # busqué, pagué, empecé
    if v.endswith("uir") and not v.endswith("guir"):        # construyo, construyó, construyendo
        for i in FUERTES:
            pres[i] = r + "y" + TERMINACIONES[t]["presente"][i]
        pret[2], pret[4], ger = r + "yó", r + "yeron", r + "yendo"
    elif t in ("er", "ir") and r[-1:] in ("a", "e", "o"):   # leyó, leído, leyendo, cayó
        pret = [r + "í", r + "íste", r + "yó", r + "ímos", r + "yeron"]
        part, ger = r + "ído", r + "yendo"
    if v.endswith("ducir"):                                 # conduje, condujeron
        pret = [r[:-1] + "j" + e for e in PRET_FUERTE]
        pret[4] = r[:-1] + "jeron"

    irr = IRREGULARES.get(v, {})                            # irregularidades propias del verbo
    if "presente" in irr:
        pres = irr["presente"].split()
    if "p1sg" in irr:
        pres[0] = irr["p1sg"]
    if "pret_raiz" in irr:
        pret = [irr["pret_raiz"] + e for e in PRET_FUERTE]
        if irr["pret_raiz"].endswith("j"):
            pret[4] = irr["pret_raiz"] + "eron"
    if "preterito" in irr:
        pret = irr["preterito"].split()
    if "fut_raiz" in irr:
        fut = [irr["fut_raiz"] + e for e in FUTURO]
    part, ger = irr.get("participio", part), irr.get("gerundio", ger)
    return {"presente": pres, "preterito": pret, "futuro": fut, "participio": part, "gerundio": ger,
            "reflexivo": reflexivo}



COLUMNAS_CONJ = ["infinitivo", "tiempo", "p1sg", "p2sg", "p3sg", "p1pl", "p3pl", "reflexivo", "revisado"]
TIEMPOS_CONJ = ["presente", "preterito", "futuro", "participio", "gerundio"]


def filas_conjugacion(infinitivo):
    """Filas de conjugaciones.csv de un verbo, con revisado = n.

    Parámetros:
        infinitivo: el verbo, como en es_m.
    Devuelve:
        Lista de 5 filas (listas con las columnas de COLUMNAS_CONJ).
    Reglas: P-07 (formato del archivo).
    Ejemplo:
        >>> filas_conjugacion("ir")[3]
        ['ir', 'participio', 'ido', '', '', '', '', 'n', 'n']
    """
    c = conjugar(infinitivo)
    refl = "s" if c["reflexivo"] else "n"
    return ([[infinitivo, t, *c[t], refl, "n"] for t in TIEMPOS_CONJ[:3]]
            + [[infinitivo, t, c[t], "", "", "", "", refl, "n"] for t in TIEMPOS_CONJ[3:]])


def errores_conjugaciones(df):
    """Revisa conjugaciones.csv (puede venir editado a mano).

    Parámetros:
        df: la tabla leída con leer_csv().
    Devuelve:
        Lista de errores. Vacía si el archivo es correcto.
    Reglas: P-07 (formato).
    Ejemplo:
        >>> errores_conjugaciones(pd.DataFrame(filas_conjugacion("ir"), columns=COLUMNAS_CONJ))
        []
    """
    if list(df.columns) != COLUMNAS_CONJ:
        return [f"columnas {list(df.columns)}; deben ser {COLUMNAS_CONJ}"]
    errores = []
    for inf, g in df.groupby("infinitivo", sort=False):
        if sorted(g["tiempo"]) != sorted(TIEMPOS_CONJ):
            errores.append(f"{inf}: debe tener una fila por tiempo {TIEMPOS_CONJ}; tiene {list(g['tiempo'])}")
        for f in g.itertuples():
            personas = [f.p1sg, f.p2sg, f.p3sg, f.p1pl, f.p3pl]
            if f.tiempo in TIEMPOS_CONJ[:3] and not all(personas):
                errores.append(f"{inf}, {f.tiempo}: falta alguna de las 5 formas")
            if f.tiempo in TIEMPOS_CONJ[3:] and (not f.p1sg or any(personas[1:])):
                errores.append(f"{inf}, {f.tiempo}: la forma va solo en p1sg")
        for col in ("reflexivo", "revisado"):
            if not set(g[col]) <= {"s", "n"}:
                errores.append(f"{inf}: {col} debe ser s o n")
        if g["reflexivo"].nunique() > 1:
            errores.append(f"{inf}: reflexivo distinto entre sus filas")
    return errores


def completar_conjugaciones(previa, verbos):
    """Añade a la tabla los verbos que faltan. Nunca cambia las filas que ya están.

    Parámetros:
        previa: tabla actual (puede estar vacía). verbos: infinitivos que deben estar.
    Devuelve:
        (tabla completa, lista de verbos añadidos).
    Reglas: P-07 (las filas revisadas a mano se conservan).
    Ejemplo:
        >>> completar_conjugaciones(pd.DataFrame(columns=COLUMNAS_CONJ), ["ir"])[1]
        ['ir']
    """
    nuevos = [v for v in verbos if v not in set(previa["infinitivo"])]
    filas = [f for v in nuevos for f in filas_conjugacion(v)]
    return pd.concat([previa, pd.DataFrame(filas, columns=COLUMNAS_CONJ)], ignore_index=True), nuevos


RUTA_CONJ = ruta("conjugaciones.csv")
VERBOS_ES = sorted({e["es_m"] for lex in (LEX, LEX_PRUEBA) for e in lex.values() if e["cat"] in VERBOS})
previa = leer_csv(RUTA_CONJ) if RUTA_CONJ.exists() else pd.DataFrame(columns=COLUMNAS_CONJ)
errores = errores_conjugaciones(previa)
if errores:                       # no se reescribe un archivo con errores: se perderían las correcciones a mano
    raise ValueError(f"{RUTA_CONJ.name} tiene {len(errores)} errores. Corrígelos a mano:\n" + "\n".join(errores[:30]))
TABLA_CONJ, añadidos = completar_conjugaciones(previa, VERBOS_ES)
if añadidos:
    TABLA_CONJ.to_csv(RUTA_CONJ, index=False, encoding="utf-8")
    print(f"{RUTA_CONJ.name}: añadidos {len(añadidos)} verbos con el conjugador automático: {', '.join(añadidos[:15])}"
          f"{' …' if len(añadidos) > 15 else ''}")

CONJUGACIONES = {}
for inf, g in TABLA_CONJ.groupby("infinitivo"):
    t = {f.tiempo: [f.p1sg, f.p2sg, f.p3sg, f.p1pl, f.p3pl] for f in g.itertuples()}
    CONJUGACIONES[inf] = dict(pres=" ".join(t["presente"]), pret=" ".join(t["preterito"]), fut=" ".join(t["futuro"]),
                              part=t["participio"][0], ger=t["gerundio"][0], refl=g["reflexivo"].iloc[0] == "s")

N_CONJ_SIN_REVISAR = int((TABLA_CONJ["revisado"] == "n").sum())    # lo usa el informe final
print(f"{RUTA_CONJ.name}: {len(CONJUGACIONES)} verbos, {len(TABLA_CONJ)} filas.")
if N_CONJ_SIN_REVISAR:
    print(f"AVISO P-07: {N_CONJ_SIN_REVISAR} filas de {len(TABLA_CONJ)} sin revisar (revisado = n). "
          "Revísalas y cambia revisado a s.")
display(TABLA_CONJ[TABLA_CONJ["infinitivo"] == "dormir"])

In [ ]:
# 7.3 · Frase nominal y oración en español (§10).
def hueco(persona, plural):
    """Elige la forma del verbo español para una persona aymara.

    Parámetros:
        persona: 1, 2, 3 o 4.
        plural: True si el sujeto es plural.
    Devuelve:
        0 = 1sg, 1 = 2sg (tú), 2 = 3sg, 3 = 1pl (nosotros), 4 = 3pl (ellos / ustedes).
    Reglas: CH-1006.
    Ejemplo:
        >>> hueco(4, False)
        3
    """
    if persona == 4 or (persona == 1 and plural):
        return 3
    if persona in (2, 3) and plural:
        return 4
    return {1: 0, 2: 1, 3: 2}[persona]


def genero_np(np, lex):
    """Género español de una frase nominal.

    Parámetros:
        np: frase nominal de la idea ({"n": id, "genero": …}).
        lex: léxico ({id: fila}).
    Devuelve:
        "m" o "f".
    Reglas: CH-1010 (el campo genero solo cuenta si el nombre es "var"; los de género fijo lo ignoran).
    Ejemplo:
        >>> genero_np({"n": "L0004", "genero": "m"}, LEX_PRUEBA)   # "mujer": género fijo
        'f'
    """
    g = lex[np["n"]]["genero"]
    return (np.get("genero") or "m") if g == "var" else g   # el prototipo ponía mal los paréntesis: daba "el mujer"


def es_plural(np):
    """Dice si una frase nominal es plural (con plural o con numeral).

    Parámetros:
        np: frase nominal de la idea.
    Devuelve:
        True o False.
    Reglas: CH-913, R12 (el numeral ya es plural).
    Ejemplo:
        >>> es_plural({"n": "L0008", "num": "NUM.3"})
        True
    """
    return bool(np.get("plural")) or bool(np.get("num"))


def articulo(g, pl, definido=True):
    """Artículo español.

    Parámetros:
        g: "m" o "f". pl: plural. definido: True = el/la/los/las; False = un/una (nada en plural).
    Devuelve:
        El artículo, o "" (indefinido plural).
    Reglas: CH-1008.
    Ejemplo:
        >>> articulo("f", False, False)
        'una'
    """
    if definido:
        return {("m", False): "el", ("f", False): "la", ("m", True): "los", ("f", True): "las"}[(g, pl)]
    return "" if pl else ("un" if g == "m" else "una")


def nombre_es(np, lex):
    """Forma española del nombre (masculino o femenino, singular o plural).

    Parámetros:
        np: frase nominal. lex: léxico.
    Devuelve:
        El nombre: "niña", "perros"…
    Reglas: CH-405, CH-1010.
    Ejemplo:
        >>> nombre_es({"n": "L0001", "genero": "f", "plural": True}, LEX_PRUEBA)
        'niñas'
    """
    e, g, pl = lex[np["n"]], genero_np(np, lex), es_plural(np)
    if e["genero"] == "var":
        return (e["es_pl_m"] if g == "m" else e["es_pl_f"]) if pl else (e["es_m"] if g == "m" else e["es_f"])
    return e["es_pl_m"] if pl else e["es_m"]


def adj_es(a, g, pl, lex):
    """Forma española del adjetivo, concordada.

    Parámetros:
        a: id del adjetivo. g: "m" o "f". pl: plural. lex: léxico.
    Devuelve:
        El adjetivo: "roja", "grandes"…
    Reglas: CH-405, CH-1008 (concuerda en género y número).
    Ejemplo:
        >>> adj_es("L0022", "f", True, LEX_PRUEBA)
        'rojas'
    """
    e = lex[a]
    return (e["es_pl_m"] if g == "m" else e["es_pl_f"]) if pl else (e["es_m"] if g == "m" else e["es_f"])


def prep(p, s):
    """Pone una preposición y contrae "a el" → "al" y "de el" → "del".

    Parámetros:
        p: preposición. s: frase que sigue.
    Devuelve:
        La frase con la preposición.
    Reglas: CH-1008 (regla 8).
    Ejemplo:
        >>> prep("a", "el niño")
        'al niño'
    """
    if p in ("a", "de") and s.startswith("el "):
        return {"a": "al ", "de": "del "}[p] + s[3:]
    return p + " " + s


def np_es(np, lex, art="def"):
    """Frase nominal española: determinante + nombre + adjetivo (+ "de" + poseedor).

    Parámetros:
        np: frase nominal de la idea. lex: léxico.
        art: "def" (el/la), "indef" (un/una) o "nada".
    Devuelve:
        La frase: "la casa del maestro", "tres perros"…
    Reglas: CH-1008 (en su orden de prioridad), CH-1010.
    Ejemplo:
        >>> np_es({"n": "L0011", "adj": "L0022"}, LEX_PRUEBA, "indef")
        'un cuchillo rojo'
    """
    g, pl, e = genero_np(np, lex), es_plural(np), lex[np["n"]]
    g_art = "m" if (g == "f" and not pl and "ART_EL" in (e.get("notas") or "")) else g      # CH-1008.9: el agua
    out = []
    if np.get("dem"):                                            # 1
        out.append(DEM_ES[np["dem"]][(2 if pl else 0) + (1 if g == "f" else 0)])
    elif np.get("num"):                                          # 2
        out.append(NUM_ES[np["num"]])
    elif np.get("poss"):                                         # 3
        p = np["poss"]
        if p == 4:
            out.append(("nuestras" if pl else "nuestra") if g == "f" else ("nuestros" if pl else "nuestro"))
        else:
            out.append({1: ("mi", "mis"), 2: ("tu", "tus"), 3: ("su", "sus")}[p][1 if pl else 0])
    elif np.get("poseedor") or art == "def":                     # 4 y 8
        out.append(articulo(g_art, pl, True))
    elif art == "indef" and articulo(g_art, pl, False):          # 5 y 6 (quien llama decide si es contable)
        out.append(articulo(g_art, pl, False))
    out.append(nombre_es(np, lex))
    if np.get("adj"):
        out.append(adj_es(np["adj"], g, pl, lex))
    if np.get("poseedor"):
        out.append(prep("de", np_es(np["poseedor"], lex, "def")))
    return " ".join(out)


def pron_es(p, pl, g):
    """Pronombre sujeto español.

    Parámetros:
        p: persona aymara (1 a 4). pl: plural. g: "m" o "f" (solo cuenta en 3.ª persona).
    Devuelve:
        "yo", "tú", "él", "ella", "nosotros", "ustedes", "ellos", "ellas".
    Reglas: CH-1006, CH-1009 ("ustedes", nunca "vosotros"), CH-1010.
    Ejemplo:
        >>> pron_es(3, True, "f")
        'ellas'
    """
    if p in (1, 4):
        return "nosotros" if (pl or p == 4) else "yo"
    if p == 2:
        return "ustedes" if pl else "tú"
    return ("ellas" if g == "f" else "ellos") if pl else ("ella" if g == "f" else "él")


def oracion_espanol(I, lex):
    """Escribe la oración española de una idea.

    Parámetros:
        I: la idea (esquema de CH-1101). lex: léxico.
    Devuelve:
        La oración, con mayúscula y puntuación.
    Lanza:
        KeyError si el verbo no está en conjugaciones.csv (P-07).
    Reglas: CH-1001 a CH-1013, CH-908 (ser, estar, tener), CH-916 (deícticos).
    Ejemplo:
        >>> oracion_espanol({"predicado": "verbo", "modalidad": "afirmativa", "tiempo": "SIMP", "verbo": "L0015",
        ...                  "sujeto": {"persona": 3, "expreso": "pronombre", "genero": "f"}}, LEX_PRUEBA)
        'Ella duerme.'
    """
    pred, mod, S = I["predicado"], I["modalidad"], I["sujeto"]
    sp, pl, g = S["persona"], bool(S.get("plural")), S.get("genero") or "m"
    inter, expreso = I.get("interrogativo"), S.get("expreso", "ninguno")
    sujeto = ""
    if inter == "quien_suj":
        sp, pl = 3, False
    elif expreso == "pronombre":
        sujeto = pron_es(sp, pl, g)                                    # CH-1007
    elif expreso == "nombre":
        sujeto = np_es(S["np"], lex, "def")
        g = genero_np(S["np"], lex)                                    # CH-405: el adjetivo de "ser" concuerda con el sujeto
    elif expreso == "coordinado":
        sujeto = " y ".join(np_es(x, lex, "def") for x in S["coord"])  # CH-907, CH-1011
        pl = True
    h = hueco(sp, pl)
    resto = []
    if pred == "verbo":
        v = lex[I["verbo"]]["es_m"]
        if v not in CONJUGACIONES:
            raise KeyError(f"P-07: el verbo '{v}' ({I['verbo']}) no está en conjugaciones.csv.")
        c, t, adv = CONJUGACIONES[v], I.get("tiempo", "SIMP"), I.get("adv_t")
        if I.get("aspecto") == "progresivo":                           # CH-1005
            vf = ESTAR[h] + " " + c["ger"]
        elif t == "SIMP":                                              # CH-1001
            vf = c["pret" if (adv and lex[adv]["subcat"] in ("pasado", "pasado_remoto")) else "pres"].split()[h]
        elif t == "FUT":                                               # CH-1004
            vf = c["fut"].split()[h]
        elif t == "REM.C":                                             # CH-1002
            vf = HABER["pres"][h] + " " + c["part"]
        else:                                                          # CH-1003
            vf = HABER["imp"][h] + " " + c["part"]
        if c.get("refl"):
            vf = REFL[h] + " " + vf
        op, cl = I.get("objeto_persona", 3), ""
        if op != 3:                                                    # CH-1009: transiciones
            cl = CLIT[(S["persona"], op)] + " "
        OI = I.get("oi")
        if (OI and OI["funcion"] == "COMPL.RU" and op == 3) or inter == "quien_oi":
            cl = "les " if (OI and OI.get("np") and es_plural(OI["np"])) else "le "      # el numeral también es plural
        if inter != "que_od":
            if I.get("od_coord"):
                miembros = [np_es(x, lex, "nada" if lex[x["n"]]["contable"] == "n" else "def") for x in I["od_coord"]]
                if I["verbo"] in A_PERSONAL and lex[I["od_coord"][0]["n"]]["clase_sem"] == "animal":   # R23 (sección 9)
                    miembros[0] = prep("a", miembros[0])
                if I["verbo"] in A_PERSONAL and lex[I["od_coord"][1]["n"]]["clase_sem"] == "animal":
                    miembros[1] = prep("a", miembros[1])
                resto.append(" y ".join(miembros))
            elif I.get("od"):
                D = I["od"]
                simple = not any(D.get(k) for k in ("plural", "dem", "poss", "poseedor", "num"))
                od_es = np_es(D, lex, "nada" if (lex[D["n"]]["contable"] == "n" and simple) else "def")   # CH-1008.7
                a_personal = I["verbo"] in A_PERSONAL and lex[D["n"]]["clase_sem"] == "animal"            # R23 (sección 9)
                resto.append(prep("a", od_es) if a_personal else od_es)
        if OI and inter != "quien_oi" and not I.get("oi_pronombre_expreso") and OI.get("np"):
            resto.append(prep("a" if OI["funcion"] == "COMPL.RU" else "para", np_es(OI["np"], lex, "def")))
        Lg = I.get("lugar")
        if Lg and inter != "donde_all":
            if Lg.get("deictico"):
                resto.append(DEICT_ES[Lg["deictico"]])
            else:
                resto.append(prep({"ALL": "a", "ABL": "de", "LOC": "en", "UNTIL": "hasta"}[Lg["caso"]],
                                  np_es(Lg["np"], lex, "def")))
        C = I.get("comcausa")
        if C:
            resto.append(prep("con" if C["caso"] == "COM" else "por", np_es(C["np"], lex, "def")))
        if adv:
            resto.append(lex[adv]["es_m"])
        verbo = cl + vf
    else:
        P = I["pnom"]
        if pred == "ser":
            verbo = SER[h]
            pl_pred = pl or h in (3, 4)                 # "nosotros" (4.ª persona) es plural en español: "Somos grandes"
            if "adj" in P:
                resto.append(adj_es(P["adj"], g, pl_pred, lex))
            else:
                npp = dict(P["np"], plural=pl_pred)
                # CH-1008.5 (corregida, P-08): "un/una" si es no humano, contable y singular; si no, nada.
                e = lex[npp["n"]]
                resto.append(np_es(npp, lex, "indef" if e["cat"] != "N_HUM" and e["contable"] == "s" else "nada"))
            if expreso == "demostrativo":
                sujeto = DEM_ES[S["dem"]][(2 if pl else 0) + (1 if genero_np(P["np"], lex) == "f" else 0)]
        elif pred == "estar":
            verbo = ESTAR[h]
            if inter != "donde_estar":
                resto.append(DEICT_ES[P["deictico"]] if P.get("deictico") else prep("en", np_es(P["np"], lex, "def")))
        else:
            verbo = TENER[h]
            np = P["np"]
            resto.append(np_es(np, lex, "indef" if lex[np["n"]]["contable"] == "s" else "nada"))   # CH-1008.6
    neg = "no " if mod == "negativa" else ""
    if mod == "pregunta_info":                                         # CH-1011
        partes = [PREGUNTA_ES[inter], verbo] + ([sujeto] if sujeto else []) + resto
    else:
        partes = ([sujeto] if sujeto else []) + [neg + verbo] + resto
    s = " ".join(x for x in partes if x)
    s = s[0].upper() + s[1:]                                           # CH-1012
    return "¿" + s + "?" if mod in ("pregunta_si_no", "pregunta_info") else s + "."


IDEA_CH1101 = {"predicado": "verbo", "modalidad": "afirmativa", "tiempo": "SIMP", "verbo": "L0018",
               "sujeto": {"persona": 3, "expreso": "nombre", "np": {"n": "L0004"}, "genero": "f"},
               "oi": {"funcion": "COMPL.RU", "np": {"n": "L0001", "genero": "m"}}, "od": {"n": "L0006"},
               "lugar": {"caso": "LOC", "np": {"n": "L0009"}}, "adv_t": "L0023", "posicion_wa": "verbo"}
print(oracion_espanol(IDEA_CH1101, LEX_PRUEBA))

### Pruebas de la sección 7

- **CH-1006:** la tabla de personas.
- **CH-1008:** los ejemplos de la especificación ("la casa del maestro", "un cuchillo rojo"…).
- **CH-1010 y CH-405:** el género (dos errores del prototipo, corregidos; ver PREGUNTAS.md).
- **CH-1008.5:** el predicado de "ser" con un nombre no contable va sin artículo (P-08).
- **P-07:** el conjugador da las mismas formas que el prototipo en sus 10 verbos, y acierta irregulares y cambios de escritura conocidos. Las filas que ya están en `conjugaciones.csv` no se cambian.

Las oraciones completas se prueban en la sección 8, con los 40 casos.

In [ ]:
# 7.4 · Pruebas del español.
class PruebasEspanol(unittest.TestCase):
    def idea_ser(self, sujeto, pnom):
        return {"predicado": "ser", "modalidad": "afirmativa", "sujeto": sujeto, "pnom": pnom}

    def test_CH1006_personas(self):
        """CH-1006: 1 → yo; 1 pl y 4 → nosotros; 2 pl → ustedes (3.ª plural del verbo)."""
        self.assertEqual([hueco(1, False), hueco(1, True), hueco(2, False), hueco(2, True),
                          hueco(3, False), hueco(3, True), hueco(4, False), hueco(4, True)], [0, 3, 1, 4, 2, 4, 3, 3])
        self.assertEqual([pron_es(1, True, "f"), pron_es(4, False, "f"), pron_es(2, True, "m")],
                         ["nosotros", "nosotros", "ustedes"])

    def test_CH1008_determinantes(self):
        """CH-1008: demostrativo, numeral, posesivo, poseedor, OD no contable y contracciones."""
        L = LEX_PRUEBA
        self.assertEqual(np_es({"n": "L0005", "dem": "DEM.1"}, L), "esta casa")
        self.assertEqual(np_es({"n": "L0008", "num": "NUM.3"}, L), "tres perros")
        self.assertEqual(np_es({"n": "L0002", "poss": 1, "genero": "f"}, L), "mi hermana")
        self.assertEqual(np_es({"n": "L0005", "poss": 4}, L), "nuestra casa")
        self.assertEqual(np_es({"n": "L0005", "poseedor": {"n": "L0003", "genero": "m"}}, L), "la casa del maestro")
        self.assertEqual(np_es({"n": "L0011", "adj": "L0022"}, L, "indef"), "un cuchillo rojo")
        self.assertEqual(np_es({"n": "L0005", "plural": True}, L, "indef"), "casas")
        self.assertEqual(prep("a", "el niño"), "al niño")

    def test_CH1008_ejemplo_tener(self):
        """CH-1008: "Mi hermana tiene un cuchillo rojo." (reglas 3 y 6)."""
        idea = {"predicado": "tener", "modalidad": "afirmativa",
                "sujeto": {"persona": 3, "expreso": "nombre", "np": {"n": "L0002", "poss": 1, "genero": "f"}},
                "pnom": {"np": {"n": "L0011", "adj": "L0022"}}}
        self.assertEqual(oracion_espanol(idea, LEX_PRUEBA), "Mi hermana tiene un cuchillo rojo.")

    def test_CH1010_genero_fijo(self):
        """CH-1010: "mujer" tiene género fijo; el campo genero de la idea se ignora (error del prototipo)."""
        self.assertEqual(np_es({"n": "L0004", "genero": "m"}, LEX_PRUEBA), "la mujer")

    def test_CH405_adjetivo_predicado(self):
        """CH-405: el adjetivo de "ser" concuerda con un sujeto nombre: "La casa es roja." (error del prototipo)."""
        idea = self.idea_ser({"persona": 3, "expreso": "nombre", "np": {"n": "L0005"}}, {"adj": "L0022"})
        self.assertEqual(oracion_espanol(idea, LEX_PRUEBA), "La casa es roja.")

    def test_CH1008_5_ser_no_humano(self):
        """CH-1008.5: "Esta es una casa." / "Eres maestra." (sin artículo si es humano) / "Ese es pan." (no contable)."""
        idea = self.idea_ser({"persona": 3, "expreso": "demostrativo", "dem": "DEM.1"}, {"np": {"n": "L0005"}})
        self.assertEqual(oracion_espanol(idea, LEX_PRUEBA), "Esta es una casa.")
        idea = self.idea_ser({"persona": 2, "expreso": "ninguno"}, {"np": {"n": "L0003", "genero": "f"}})
        self.assertEqual(oracion_espanol(idea, LEX_PRUEBA), "Eres maestra.")
        idea = self.idea_ser({"persona": 3, "expreso": "demostrativo", "dem": "DEM.2"}, {"np": {"n": "L0006"}})
        self.assertEqual(oracion_espanol(idea, LEX_PRUEBA), "Ese es pan.")                  # P-08: "pan" no es contable
        idea = self.idea_ser({"persona": 3, "plural": True, "expreso": "demostrativo", "dem": "DEM.2"}, {"np": {"n": "L0005"}})
        self.assertEqual(oracion_espanol(idea, LEX_PRUEBA), "Esas son casas.")

    def test_CH1101_ejemplo(self):
        """CH-1101 / CH-1011: "La mujer le dio pan al niño en el mercado ayer."."""
        self.assertEqual(oracion_espanol(IDEA_CH1101, LEX_PRUEBA), "La mujer le dio pan al niño en el mercado ayer.")

    def test_CH1009_les_con_numeral(self):
        """CH-1009 (revisión): COMPL.RU con numeral es plural: "les"."""
        idea = {"predicado": "verbo", "modalidad": "afirmativa", "tiempo": "SIMP", "verbo": "L0018",
                "sujeto": {"persona": 1, "expreso": "pronombre"}, "od": {"n": "L0006"},
                "oi": {"funcion": "COMPL.RU", "np": {"n": "L0003", "num": "NUM.3", "genero": "m"}}}
        self.assertEqual(oracion_espanol(idea, LEX_PRUEBA), "Yo les doy pan a tres maestros.")

    def test_CH1008_9_el_agua(self):
        """CH-1008, regla 9: femenino con a- tónica (ART_EL): el agua, un agua, del agua; las aguas; esta agua."""
        agua = {"cat": "N_NOHUM", "clase_sem": "bebida", "es_m": "agua", "es_f": "", "es_pl_m": "aguas", "es_pl_f": "",
                "genero": "f", "contable": "s", "raiz": "x", "notas": "ART_EL"}
        L = {**LEX_PRUEBA, "LX01": agua}
        self.assertEqual(np_es({"n": "LX01"}, L), "el agua")
        self.assertEqual(np_es({"n": "LX01"}, L, "indef"), "un agua")
        self.assertEqual(prep("de", np_es({"n": "LX01"}, L)), "del agua")
        self.assertEqual(np_es({"n": "LX01", "plural": True}, L), "las aguas")
        self.assertEqual(np_es({"n": "LX01", "dem": "DEM.1"}, L), "esta agua")
        self.assertEqual(np_es({"n": "LX01", "adj": "L0022"}, L), "el agua roja")

    def test_P07_conjugaciones_prueba(self):
        """P-07: todos los verbos de los dos léxicos están en conjugaciones.csv, con sus 5 personas."""
        for e in list(LEX_PRUEBA.values()) + list(LEX.values()):
            if e["cat"] in VERBOS:
                c = CONJUGACIONES[e["es_m"]]
                self.assertTrue(all(len(c[t].split()) == 5 for t in ("pres", "pret", "fut")), e["es_m"])
                self.assertTrue(c["part"] and c["ger"], e["es_m"])

    def test_P07_control_prototipo(self):
        """P-07: para los 10 verbos del prototipo, el conjugador da exactamente sus formas."""
        PROTOTIPO = {   # tabla CONJ de datos/chulupi_oraciones.py, copiada tal cual
            "ir": dict(pres="voy vas va vamos van", pret="fui fuiste fue fuimos fueron", fut="iré irás irá iremos irán", part="ido", ger="yendo"),
            "venir": dict(pres="vengo vienes viene venimos vienen", pret="vine viniste vino vinimos vinieron", fut="vendré vendrás vendrá vendremos vendrán", part="venido", ger="viniendo"),
            "dormir": dict(pres="duermo duermes duerme dormimos duermen", pret="dormí dormiste durmió dormimos durmieron", fut="dormiré dormirás dormirá dormiremos dormirán", part="dormido", ger="durmiendo"),
            "comer": dict(pres="como comes come comemos comen", pret="comí comiste comió comimos comieron", fut="comeré comerás comerá comeremos comerán", part="comido", ger="comiendo"),
            "ver": dict(pres="veo ves ve vemos ven", pret="vi viste vio vimos vieron", fut="veré verás verá veremos verán", part="visto", ger="viendo"),
            "dar": dict(pres="doy das da damos dan", pret="di diste dio dimos dieron", fut="daré darás dará daremos darán", part="dado", ger="dando"),
            "comprar": dict(pres="compro compras compra compramos compran", pret="compré compraste compró compramos compraron", fut="compraré comprarás comprará compraremos comprarán", part="comprado", ger="comprando"),
            "cortar": dict(pres="corto cortas corta cortamos cortan", pret="corté cortaste cortó cortamos cortaron", fut="cortaré cortarás cortará cortaremos cortarán", part="cortado", ger="cortando"),
            "querer": dict(pres="quiero quieres quiere queremos quieren", pret="quise quisiste quiso quisimos quisieron", fut="querré querrás querrá querremos querrán", part="querido", ger="queriendo"),
            "encontrarse": dict(pres="encuentro encuentras encuentra encontramos encuentran", pret="encontré encontraste encontró encontramos encontraron", fut="encontraré encontrarás encontrará encontraremos encontrarán", part="encontrado", ger="encontrando", refl=True),
        }
        for v, esperado in PROTOTIPO.items():
            c = conjugar(v)
            with self.subTest(verbo=v):
                self.assertEqual(dict(pres=" ".join(c["presente"]), pret=" ".join(c["preterito"]), fut=" ".join(c["futuro"]),
                                      part=c["participio"], ger=c["gerundio"], refl=c["reflexivo"]),
                                 {**esperado, "refl": esperado.get("refl", False)})

    def test_P07_reglas_conjugador(self):
        """P-07: cambios de vocal, de escritura, irregulares y compuestos conocidos del español."""
        casos = [("buscar", "preterito", 0, "busqué"), ("pagar", "preterito", 0, "pagué"), ("empezar", "presente", 0, "empiezo"),
                 ("conocer", "presente", 0, "conozco"), ("vencer", "presente", 0, "venzo"), ("construir", "preterito", 2, "construyó"),
                 ("leer", "preterito", 4, "leyeron"), ("pedir", "preterito", 2, "pidió"), ("seguir", "presente", 0, "sigo"),
                 ("elegir", "presente", 0, "elijo"), ("mantener", "preterito", 0, "mantuve"), ("devolver", "presente", 2, "devuelve"),
                 ("suponer", "futuro", 0, "supondré"), ("conducir", "preterito", 4, "condujeron"), ("oler", "presente", 0, "huelo"),
                 ("enviar", "presente", 4, "envían"), ("continuar", "presente", 0, "continúo"), ("hacer", "preterito", 2, "hizo"),
                 ("sentir", "preterito", 4, "sintieron"), ("decir", "preterito", 4, "dijeron"), ("saber", "presente", 0, "sé")]
        for v, t, i, forma in casos:
            self.assertEqual(conjugar(v)[t][i], forma, v)
        for v, t, forma in [("devolver", "participio", "devuelto"), ("leer", "participio", "leído"), ("leer", "gerundio", "leyendo"),
                            ("pedir", "gerundio", "pidiendo"), ("componer", "participio", "compuesto"), ("describir", "participio", "descrito")]:
            self.assertEqual(conjugar(v)[t], forma, v)

    def test_P07_conserva_filas(self):
        """P-07: completar la tabla no cambia una fila corregida a mano y solo añade los verbos que faltan."""
        previa = pd.DataFrame(filas_conjugacion("ir"), columns=COLUMNAS_CONJ)
        previa.loc[0, ["p1sg", "revisado"]] = ["VOY", "s"]
        tabla, nuevos = completar_conjugaciones(previa, ["ir", "comer"])
        self.assertEqual(nuevos, ["comer"])
        self.assertEqual(tabla.loc[0, "p1sg"], "VOY")
        self.assertEqual(len(tabla), 10)
        self.assertEqual(errores_conjugaciones(tabla), [])
        self.assertTrue(errores_conjugaciones(tabla.drop(index=3)))       # falta el participio de "ir"


correr_pruebas(PruebasEspanol, titulo="español (§10)")

## 8. Oraciones aymaras (modos B y A)

**Qué hace esta sección.** Convierte una idea en la oración aymara, en los dos modos. Cada palabra se construye con el motor de la sección 3.

**Orden de las partes** (CH-901): sujeto – `jani` – tiempo – lugar – compañía o causa – objeto indirecto – objeto directo – verbo.

**Los sufijos de oración** (oracionales):
- `-wa` (afirmación, CH-902) va en **una sola** palabra: el verbo, el sujeto o un complemento, según `posicion_wa`.
- `-xa` (tema, CH-903) marca al sujeto expreso, si no lleva `-wa`. Si no hay sujeto, lo lleva el adverbio de tiempo.
- `-ti` (pregunta sí/no o negación, CH-904, CH-905) y `-sa` (pregunta con "qué", "quién", "dónde", CH-906).

**Reglas derivadas** (`DV-01` a `DV-13`). Algunas formas no están atestiguadas en la guía: se deducen de un patrón (Anexo A). El constructor anota cuáles usa cada oración, para poder revisarlas después.

**Qué entra.** Una idea y un léxico. **Qué sale.** `generar(idea, lex)` devuelve un diccionario con:
- `b`, `a`, `es`: las tres oraciones;
- `palabras`: la base y los ids de cada palabra (para el verificador de la sección 11);
- `dv`: las reglas derivadas que usa.

**Ejemplo** (CH-901):
- B: `Simlla-xa jaskhi pinqara-na lisqu-ru q'aspi tulph-i-wa.` → `Simllaxa jaskhi pinqarana lisquru q'aspi tulphiwa.` — mujer-TOP ayer mercado-LOC niño-COMPL.RU pan dar-3>3.SIMP-AFF.
- A: `Simllaxu jaskhi pinqaramu lisqullu q'aspi tulphuyi.`
- ES: "La mujer le dio pan al niño en el mercado ayer."

**Al final** se prueban los 40 casos de §12 (CH-1201). Si uno falla, el notebook se detiene.

In [ ]:
# 8.1 · Palabras gramaticales, ids del verbo y reglas derivadas (CH-702, Anexo A).
B_DE = lambda i: GRAMATICALES[i]["b"]                         # forma B de una palabra gramatical (CH-505)
PRON_B = {p: B_DE(f"PRON.{p}") for p in (1, 2, 3, 4)}
DEM_B = {d: B_DE(d) for d in ("DEM.1", "DEM.2", "DEM.3")}
NUM_B = {n: B_DE(n) for n in ("NUM.3", "NUM.4", "NUM.6", "NUM.10")}
PERS = {1: "P1", 2: "P2", 3: "P3", 4: "P4"}
FUTP = {1: "FUT.1", 2: "FUT.2", 3: "FUT.3", 4: "FUT.4"}
TRANS = {(1, 2): "T12", (2, 1): "T21", (3, 1): "T31", (3, 2): "T32", (3, 4): "T34"}   # §3.1, R2
PERS_IDS = set(PERS.values()) | set(FUTP.values()) | set(TRANS.values()) | {"P3.REM"}
ORAC_DV = {"AFF", "Q.YN", "NEG.TI", "Q.INF"}                     # oracionales que cuentan para las DV (sin TOP)


def reglas_derivadas(ids, tipo="VERBO"):
    """Dice qué reglas [DERIVADA] usa una palabra.

    Parámetros:
        ids: ids de la palabra. tipo: "VERBO", "ESTAR", "TENER" o "SER".
    Devuelve:
        Lista ordenada de DV ("DV-01"…).
    Reglas: CH-702, Anexo A (DV-01 a DV-13).
    Ejemplo:
        >>> reglas_derivadas(["T32", "AFF"])
        ['DV-01']
    """
    s = []
    o = ids[-1] if ids and ids[-1] in ORAC_DV else None
    p = next((i for i in ids if i in PERS_IDS), None)
    rem = "REM.C" in ids or "REM.L" in ids
    if tipo == "VERBO":
        if p == "T32" and o: s.append("DV-01")
        if p in ("T12", "T21", "T31", "T34") and o: s.append("DV-02")
        if o in ("Q.YN", "NEG.TI") and p in ("P2", "P3", "P4") and not rem: s.append("DV-03")
        if (p in ("FUT.1", "FUT.2") and o) or (p == "FUT.4" and o in ("Q.YN", "NEG.TI")): s.append("DV-04")
        if rem and o: s.append("DV-05")
        if "PL.V" in ids and (p in ("FUT.1", "FUT.3", "FUT.4") or "REM.L" in ids): s.append("DV-06")
        if "PL.V" in ids and p and p.startswith("T"): s.append("DV-07")
        if "NEG" in ids and p not in ("P1", "FUT.3"): s.append("DV-08")
        if "PROG" in ids and p != "P3": s.append("DV-09")
    else:
        if tipo == "ESTAR" and p != "P1": s.append("DV-10")
        if tipo == "TENER" and p in ("P2", "P4"): s.append("DV-11")
        if tipo == "SER" and p in ("P2", "P4"): s.append("DV-12")
        if o == "Q.YN": s.append("DV-13")
    return sorted(set(s))


def ids_verbo(I, orac):
    """Ids del verbo de una idea: [NEG | PL.V | PROG] + tiempo y persona + oracional.

    Parámetros:
        I: la idea. orac: id del oracional del verbo, o None.
    Devuelve:
        Lista de ids, en el orden de CH-602.
    Reglas: CH-602 a CH-606, CH-511 (P3.REM en los remotos), §3.1 (transiciones).
    Ejemplo:
        >>> ids_verbo({"modalidad": "afirmativa", "tiempo": "REM.C", "sujeto": {"persona": 3, "plural": True}}, "AFF")
        ['PL.V', 'REM.C', 'P3.REM', 'AFF']
    """
    S = I["sujeto"]
    sp, op, t = S["persona"], I.get("objeto_persona", 3), I.get("tiempo", "SIMP")
    ids = []
    if I["modalidad"] == "negativa":
        ids.append("NEG")
    elif S.get("plural"):
        ids.append("PL.V")
    elif I.get("aspecto") == "progresivo":
        ids.append("PROG")
    if op != 3:
        ids.append(TRANS[(sp, op)])
    elif t == "SIMP":
        ids.append(PERS[sp])
    elif t == "FUT":
        ids.append(FUTP[sp])
    else:
        ids += [t, "P3.REM" if sp == 3 else PERS[sp]]
    if orac:
        ids.append(orac)
    return ids


print(reglas_derivadas(["T32", "AFF"]), ids_verbo({"modalidad": "negativa", "tiempo": "FUT", "sujeto": {"persona": 1}}, "NEG.TI"))

In [ ]:
# 8.2 · Constructor de la oración aymara (§9) y generar() (B, A y español).
class Oracion:
    """Guarda las palabras de una oración mientras se construye, en un modo."""

    def __init__(self, modo, lex):
        self.modo, self.lex, self.palabras, self.dv = modo, lex, [], []
        self.const = "S"                     # constituyente que se está escribiendo (CH-901); lo usa la sección 11

    def w(self, base, ids, tipo=None):
        """Construye una palabra (sección 3), la anota y devuelve su forma. tipo = para las reglas DV."""
        forma = construir_palabra(base, list(ids), self.modo)
        self.palabras.append({"base": base, "ids": list(ids), "forma": forma, "const": self.const})
        if tipo:
            self.dv += reglas_derivadas(ids, tipo)
        return forma

    def raiz(self, i):
        """Raíz de una entrada del léxico."""
        return self.lex[i]["raiz"]


def np_aymara(O, np, caso=None, orac=None):
    """Frase nominal: [poseedor-GEN] [DEM | NUM] [ADJ] NÚCLEO-sufijos.

    Parámetros:
        O: la Oracion. np: frase nominal de la idea. caso: id de caso o None. orac: oracional o None.
    Devuelve:
        La frase (palabras separadas por espacio).
    Reglas: CH-912 (orden), CH-911 (poseedor con GEN y POSS.3), CH-913 (PL, salvo con numeral).
    Ejemplo:
        >>> np_aymara(Oracion("B", LEX_PRUEBA), {"n": "L0005", "plural": True, "poss": 1}, "LOC")
        'chimulanakajana'
    """
    out = []
    if np.get("poseedor"):
        out.append(np_aymara(O, np["poseedor"], "GEN"))
    if np.get("dem"):
        out.append(O.w(DEM_B[np["dem"]], []))
    if np.get("num"):
        out.append(O.w(NUM_B[np["num"]], []))
    if np.get("adj"):
        out.append(O.w(O.raiz(np["adj"]), []))
    ids = ["PL"] if np.get("plural") and not np.get("num") else []
    if np.get("poseedor"):
        ids.append("POSS.3")
    elif np.get("poss"):
        ids.append(f"POSS.{np['poss']}")
    ids += [x for x in (caso, orac) if x]
    out.append(O.w(O.raiz(np["n"]), ids))
    return " ".join(out)


def pron_aymara(O, persona, plural, caso=None, orac=None):
    """Pronombre personal con sus sufijos.

    Parámetros:
        O: la Oracion. persona: 1 a 4. plural: bool. caso, orac: ids o None.
    Devuelve:
        La palabra.
    Reglas: CH-505, CH-517 (1.ª plural = nänaka, sin PL).
    Ejemplo:
        >>> pron_aymara(Oracion("B", LEX_PRUEBA), 2, True, None, "TOP")
        'jumanakaxa'
    """
    base, ids = PRON_B[persona], []
    if plural:
        if persona == 1:
            base = B_DE("PRON.1.PL")
        else:
            ids.append("PL")
    return O.w(base, ids + [x for x in (caso, orac) if x])


def destino_wa(I, suj_expreso):
    """Decide qué parte lleva -wa en una afirmativa.

    Parámetros:
        I: la idea. suj_expreso: True si el sujeto aparece (y no es khiti).
    Devuelve:
        "verbo", "sujeto", "lugar", "comcausa", "oi", "pred" o None (sin -wa).
    Reglas: CH-902 (posicion_wa; complemento = el más cercano al verbo; respaldo = verbo), CH-908.
    Ejemplo:
        >>> destino_wa({"modalidad": "afirmativa", "predicado": "ser"}, True)
        'pred'
    """
    if I["modalidad"] != "afirmativa":
        return None
    if I["predicado"] != "verbo":
        return "pred"
    d = I.get("posicion_wa", "verbo")
    if d == "ninguna":
        return None
    if d == "sujeto" and not suj_expreso:
        return "verbo"
    if d == "complemento":
        cands = [k for k in ("lugar", "comcausa", "oi") if I.get(k)
                 and (k != "lugar" or (I[k].get("caso") in ("ALL", "UNTIL") and not I[k].get("deictico")))]
        return cands[-1] if cands else "verbo"
    return d


def oracion_aymara(I, modo, lex):
    """Construye la oración aymara de una idea en un modo.

    Parámetros:
        I: la idea (CH-1101). modo: "B" o "A". lex: léxico.
    Devuelve:
        (oración, palabras, reglas DV).
    Reglas: CH-901 a CH-918, CH-106 (mayúscula y puntuación).
    Ejemplo:
        >>> oracion_aymara(IDEA_CH1101, "B", LEX_PRUEBA)[0]
        "Simllaxa jaskhi pinqarana lisquru q'aspi tulphiwa."
    """
    O = Oracion(modo, lex)
    pred, mod, S = I["predicado"], I["modalidad"], I["sujeto"]
    inter = I.get("interrogativo")
    expreso = S.get("expreso", "ninguno")
    suj_expreso = expreso != "ninguno" and inter != "quien_suj"
    destino = destino_wa(I, suj_expreso)
    partes = []
    # 1. Sujeto (CH-903, CH-906, CH-907)
    if inter == "quien_suj":
        partes.append(O.w(B_DE("INT.QUIEN"), ["Q.INF"]))
    elif suj_expreso:
        if expreso == "coordinado":
            a, b = S["coord"]
            partes.append(np_aymara(O, a, "COM"))
            partes.append(np_aymara(O, b, "COM", "AFF" if destino == "sujeto" else None))
        else:
            orac = "AFF" if destino == "sujeto" else "TOP"
            if expreso == "pronombre":
                partes.append(pron_aymara(O, S["persona"], S.get("plural"), None, orac))
            elif expreso == "demostrativo":
                partes.append(O.w(DEM_B[S["dem"]], (["PL"] if S.get("plural") else []) + [orac]))
            else:
                partes.append(np_aymara(O, S["np"], None, orac))
    # 2. jani-wa (CH-905)
    O.const = "JANI"
    if mod == "negativa":
        partes.append(O.w(B_DE("NEG.PAL"), ["AFF"]))
    # 3. Tiempo: lleva -xa si no hay sujeto expreso (CH-903)
    O.const = "T"
    if I.get("adv_t"):
        top = expreso == "ninguno" and inter != "quien_suj"     # CH-903.2: sin sujeto expreso (ni khiti), -xa en el adverbio
        partes.append(O.w(O.raiz(I["adv_t"]), ["TOP"] if top else []))
    if pred == "verbo":
        O.const = "L"
        Lg = I.get("lugar")                                       # 4. Lugar (CH-916, CH-918)
        if Lg:
            o = "AFF" if destino == "lugar" else None
            if inter == "donde_all":
                partes.append(O.w(B_DE("INT.DONDE"), ["ALL", "Q.INF"]))
            elif Lg.get("deictico"):
                partes.append(O.w(DEM_B[Lg["deictico"]], ["LOC"]))
            else:
                partes.append(np_aymara(O, Lg["np"], Lg["caso"], o))
        O.const = "COMCAUSE"
        C = I.get("comcausa")                                     # 5. COM o CAUSE
        if C:
            partes.append(np_aymara(O, C["np"], C["caso"], "AFF" if destino == "comcausa" else None))
        O.const = "OI"
        OI = I.get("oi")                                          # 6. Objeto indirecto (CH-909, CH-1009)
        if OI or inter == "quien_oi":
            o = "AFF" if destino == "oi" else None
            if inter == "quien_oi":
                partes.append(O.w(B_DE("INT.QUIEN"), ["COMPL.RU", "Q.INF"]))
            elif I.get("oi_pronombre_expreso"):
                partes.append(pron_aymara(O, I["objeto_persona"], False, "COMPL.RU", o))
            else:
                partes.append(np_aymara(O, OI["np"], OI["funcion"], o))
        O.const = "OD"
        if inter == "que_od":                                     # 7. Objeto directo (CH-907, CH-915)
            partes.append(O.w(B_DE("INT.QUE"), ["Q.INF"]))
        elif I.get("od_coord"):
            a, b = I["od_coord"]
            partes.append(np_aymara(O, a, "COM"))
            partes.append(np_aymara(O, b, "COM"))
        elif I.get("od"):
            partes.append(np_aymara(O, I["od"]))
        O.const = "V"
        orac = {"afirmativa": "AFF" if destino == "verbo" else None, "negativa": "NEG.TI",
                "pregunta_si_no": "Q.YN", "pregunta_info": None}[mod]
        partes.append(O.w(O.raiz(I["verbo"]), ids_verbo(I, orac), "VERBO"))   # 8. Verbo
    else:                                                         # 8. Predicado nominal (CH-908)
        O.const = "V"
        P, sp = I["pnom"], S["persona"]
        orac = {"afirmativa": "AFF", "pregunta_si_no": "Q.YN", "pregunta_info": "Q.INF"}[mod]
        if pred == "ser":
            base = O.raiz(P["adj"]) if "adj" in P else O.raiz(P["np"]["n"])
            if sp == 3:
                partes.append(O.w(base, (["PL"] if (S.get("plural") and "np" in P) else []) + [orac]))
            else:
                partes.append(O.w(base, ["VBLZ.LONG", PERS[sp], orac], "SER"))
        elif pred == "estar":
            if inter == "donde_estar":
                partes.append(O.w(B_DE("INT.DONDE"), ["LOC", "VBLZ.KA", PERS[sp], "Q.INF"], "ESTAR"))
            elif P.get("deictico"):
                partes.append(O.w(DEM_B[P["deictico"]], ["LOC", "VBLZ.KA", PERS[sp], orac], "ESTAR"))
            else:
                np = P["np"]
                for k, tabla in (("dem", DEM_B), ("num", NUM_B)):
                    if np.get(k):
                        partes.append(O.w(tabla[np[k]], []))
                if np.get("adj"):
                    partes.append(O.w(O.raiz(np["adj"]), []))
                ids = (["PL"] if np.get("plural") else []) + ([f"POSS.{np['poss']}"] if np.get("poss") else []) \
                    + ["LOC", "VBLZ.KA", PERS[sp], orac]
                partes.append(O.w(O.raiz(np["n"]), ids, "ESTAR"))
        elif pred == "tener":
            np = P["np"]
            for k, tabla in (("dem", DEM_B), ("num", NUM_B)):
                if np.get(k):
                    partes.append(O.w(tabla[np[k]], []))
            if np.get("adj"):
                partes.append(O.w(O.raiz(np["adj"]), []))
            ids = (["PL"] if (np.get("plural") and not np.get("num")) else []) + ["POSR"] \
                + ([] if sp == 3 else ["VBLZ.LONG", PERS[sp]]) + [orac]
            partes.append(O.w(O.raiz(np["n"]), ids, "TENER"))
    s = " ".join(partes)
    s = s[0].upper() + s[1:]                                      # CH-106
    s = "¿" + s + "?" if mod in ("pregunta_si_no", "pregunta_info") else s + "."
    return s, O.palabras, sorted(set(O.dv))


def generar(idea, lex):
    """Genera las tres versiones de una idea.

    Parámetros:
        idea: la idea (CH-1101). lex: léxico ({id: fila}); LEX para generar, LEX_PRUEBA para los casos.
    Devuelve:
        {"b", "a", "es", "palabras" (B), "palabras_a", "dv"}.
    Reglas: §9, §10, CH-702.
    Ejemplo:
        >>> generar(IDEA_CH1101, LEX_PRUEBA)["a"]
        "Simllaxu jaskhi pinqaramu lisqullu q'aspi tulphuyi."
    """
    b, palabras_b, dv = oracion_aymara(idea, "B", lex)
    a, palabras_a, _ = oracion_aymara(idea, "A", lex)
    return {"b": b, "a": a, "es": oracion_espanol(idea, lex), "palabras": palabras_b, "palabras_a": palabras_a, "dv": dv}


r = generar(IDEA_CH1101, LEX_PRUEBA)
print("B: ", r["b"]); print("A: ", r["a"]); print("ES:", r["es"]); print("DV:", r["dv"] or "ninguna")

### Los 40 casos de prueba (CH-1201)

`casos_prueba.json` trae 40 ideas con su salida esperada en B, A y español. Cubren todas las personas, tiempos, casos, preguntas y predicados (§12.3).

Se comprueba, para cada caso:
1. B, A y español, **carácter por carácter**;
2. las reglas derivadas (`dv`) y los ids de cada palabra;
3. en los 17 casos **[ATESTIGUADO]**, que B es la oración de la guía;
4. que cada palabra B y A es válida (CH-709);
5. CH-1013: si dos ideas solo cambian el género, la oración aymara es la misma y el español solo cambia en palabras de género.

Los casos usan siempre `LEX_PRUEBA` (P-05).

In [ ]:
# 8.3 · Los 40 casos de CH-1201. Si uno falla, el notebook se detiene.
import copy

CASOS = json.loads(ruta("casos_prueba.json").read_text(encoding="utf-8"))
RESULTADOS = {c["id"]: generar(c["idea"], LEX_PRUEBA) for c in CASOS}
PALABRAS_GENERO = {"el", "la", "los", "las", "un", "una", "él", "ella", "ellos", "ellas",
                   "este", "esta", "estos", "estas", "ese", "esa", "esos", "esas", "aquel", "aquella",
                   "aquellos", "aquellas", "nuestro", "nuestra", "nuestros", "nuestras", "al", "del"}


class PruebasCasos(unittest.TestCase):
    def test_CH903_adverbio_en_pregunta_sin_sujeto(self):
        """CH-903.2 (revisión): pregunta de información sin sujeto expreso: el adverbio lleva -xa; con khiti, no."""
        I = {"predicado": "verbo", "modalidad": "pregunta_info", "tiempo": "REM.C", "verbo": "L0019",
             "sujeto": {"persona": 2, "expreso": "ninguno"}, "interrogativo": "que_od", "adv_t": "L0026"}
        r = generar(I, LEX_PRUEBA)
        self.assertEqual((r["b"], r["a"]), ("¿Lasmiq'axa kunasa nirqayäta?", "¿Lasmiq'axu sipukhu nirqajüku?"))
        J = {"predicado": "verbo", "modalidad": "pregunta_info", "tiempo": "REM.C", "verbo": "L0013",
             "sujeto": {"persona": 3}, "interrogativo": "quien_suj", "adv_t": "L0026"}
        self.assertTrue(generar(J, LEX_PRUEBA)["b"].startswith("¿Khitisa lasmiq'a "))

    def test_CH1201_cantidad(self):
        """CH-1201: hay 40 casos, 17 de ellos [ATESTIGUADO]."""
        self.assertEqual(len(CASOS), 40)
        self.assertEqual(sum(bool(c["atestiguado"]) for c in CASOS), 17)

    def test_CH1201_salidas(self):
        """CH-1201: B, A y español exactos en los 40 casos."""
        for c in CASOS:
            r = RESULTADOS[c["id"]]
            with self.subTest(caso=c["id"]):
                self.assertEqual((r["b"], r["a"], r["es"]), (c["esperado"]["b"], c["esperado"]["a"], c["esperado"]["es"]))

    def test_CH702_dv_e_ids(self):
        """CH-702: reglas derivadas e ids de cada palabra iguales a los del archivo."""
        for c in CASOS:
            r = RESULTADOS[c["id"]]
            with self.subTest(caso=c["id"]):
                self.assertEqual(r["dv"], c["dv"])
                self.assertEqual([[p["base"], p["ids"]] for p in r["palabras"]], c["palabras"])

    def test_CH1201_atestiguados(self):
        """CH-1201: los 17 casos [ATESTIGUADO] dan en B la oración de la guía."""
        for c in CASOS:
            if c["atestiguado"]:
                with self.subTest(caso=c["id"], fuente=c["fuente"]):
                    self.assertEqual(RESULTADOS[c["id"]]["b"], c["atestiguado"])

    def test_CH709_palabras_validas(self):
        """CH-709: cada palabra de B y de A es válida."""
        for c in CASOS:
            r = RESULTADOS[c["id"]]
            for w in re.findall(r"[^\s¿?.]+", (r["b"] + " " + r["a"]).lower()):
                self.assertEqual(validar_palabra(w), [], (c["id"], w))

    def test_CH1013_genero(self):
        """CH-1013: C12 con genero = f: misma oración B y A; el español solo cambia en palabras de género."""
        base = next(c for c in CASOS if c["id"] == "C12")["idea"]
        f = copy.deepcopy(base)
        f["sujeto"]["genero"] = "f"
        rm, rf = generar(base, LEX_PRUEBA), generar(f, LEX_PRUEBA)
        self.assertEqual((rm["b"], rm["a"]), (rf["b"], rf["a"]))
        self.assertNotEqual(rm["es"], rf["es"])
        for x, y in zip(rm["es"].split(), rf["es"].split()):
            if x != y:
                self.assertTrue(x.strip("¿?.").lower() in PALABRAS_GENERO or x[:-1] == y[:-1], (x, y))


correr_pruebas(PruebasCasos, titulo="40 casos de CH-1201")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([{"caso": c["id"], "B": RESULTADOS[c["id"]]["b"], "A": RESULTADOS[c["id"]]["a"],
                           "español": RESULTADOS[c["id"]]["es"], "DV": " ".join(RESULTADOS[c["id"]]["dv"]),
                           "atestiguado": c["fuente"] if c["atestiguado"] else ""} for c in CASOS]).set_index("caso"))

## 9. Ideas

**Qué es una idea.** Es la descripción, en datos, de lo que dice una oración: el verbo, quién lo hace, el tiempo, los complementos… (CH-1101). De cada idea salen las tres versiones (B, A y español) con `generar()` (sección 8).

Esta sección hace dos cosas:
1. **Revisar** una idea (`validar_idea`). Comprueba que sus valores existen (CH-1102) y que cumple las 22 restricciones R1–R22 (CH-1103). Por ejemplo: con objeto "me / te / nos", el tiempo debe ser SIMP (R1).
2. **Sortear** ideas nuevas con los pesos de `pesos.yaml` (CH-1104 a CH-1113).

**Cómo se sortea** (CH-1104):
1. Se reparte el total entre las combinaciones de **tipo de oración × tiempo × persona** del sujeto. Cada una recibe una **cuota**, según sus pesos. Por ejemplo, "verbo afirmativo, FUT, 3.ª persona" recibe 0,42 × 0,25 × 0,40 del total.
2. Si una combinación es imposible con el léxico, su cuota se reparte entre las demás.
3. Dentro de cada idea se sortea el resto: el verbo, el sujeto, los complementos… Cada palabra se elige con Zipf (CH-403): las de rango bajo salen más.
4. Si algo sorteado no es posible (por ejemplo, "¿a dónde?" con un verbo que no es de movimiento), se vuelve a sortear entre lo posible.
5. Cada idea se revisa con `validar_idea`. Si falla, se descarta y se sortea otra.

**Qué entra.** El léxico y los pesos. **Qué sale.** Una lista de ideas válidas.

**Ejemplo** (CH-1103):
- ✗ Idea con `tiempo = FUT` y `objeto_persona = 1`: R1 falla y se descarta.
- ✓ La misma idea en SIMP: B `Kirpuxa ch'ixrituwa.` / A `Kirpuxu ch'ixruquyi.` / ES "El perro me ve."

In [ ]:
# 9.1 · Validador de ideas: valores permitidos (CH-1102) y restricciones R1–R22 (CH-1103).
ESQ = CONFIG["idea.schema"]
PESOS = CONFIG["pesos"]
TIEMPOS_ADV = CONFIG["sintaxis"]["adv_tiempo_compatibilidad"]                      # CH-917
LUGAR_GENERAL = {"lugar", "vivienda"}                                               # CH-406, tabla 3
CLASES_ESTAR_LUGAR = {"lugar", "vivienda", "recipiente"}
CLASES_ESTAR_NO_SUJETO = {"lugar", "tiempo", "abstracto"}
CLASES_TENER = {"animal", "comida", "bebida", "ropa", "herramienta", "recipiente", "planta", "vivienda", "objeto"}
MODS_NP = ("adj", "dem", "num", "poss", "poseedor")


def clases_de(e, col):
    """Clases de una columna del léxico, como conjunto.

    Parámetros:
        e: fila del léxico. col: suj_clases, obj_clases, instr_clases, loc_clases o mod_clases.
    Devuelve:
        Conjunto de clases ("-" y "NINGUNA" dan un conjunto vacío). None si la fila no trae la
        columna: el léxico de validación (LV01–LV18) no tiene clases, y esas comprobaciones se saltan.
    Reglas: CH-402, CH-406.
    Ejemplo:
        >>> clases_de(LEX_PRUEBA["L0016"], "obj_clases")
        {'comida'}
    """
    v = e.get(col)
    if v is None:
        return None
    return set() if v in ("-", "", "NINGUNA") else set(v.split("|"))


def ids_de_idea(I):
    """Lista de los ids del léxico que usa una idea (con repeticiones).

    Parámetros:
        I: la idea.
    Devuelve:
        Lista de ids: verbo, adverbio, núcleos, adjetivos y poseedores de todas las frases nominales.
    Reglas: R21, R22.
    Ejemplo:
        >>> ids_de_idea(IDEA_CH1101)
        ['L0018', 'L0023', 'L0004', 'L0006', 'L0001', 'L0009']
    """
    out = [I[k] for k in ("verbo", "adv_t") if I.get(k)]

    def np_ids(np):
        if isinstance(np, dict) and np.get("n"):
            out.append(np["n"])
            if np.get("adj"):
                out.append(np["adj"])
            np_ids(np.get("poseedor"))

    S = I.get("sujeto") or {}
    np_ids(S.get("np"))
    for x in (S.get("coord") or []) + (I.get("od_coord") or []):
        np_ids(x)
    np_ids(I.get("od"))
    for k in ("oi", "lugar", "comcausa"):
        np_ids((I.get(k) or {}).get("np"))
    P = I.get("pnom") or {}
    if P.get("adj"):
        out.append(P["adj"])
    np_ids(P.get("np"))
    return out


def validar_idea(I, lex, solo_produccion=True):
    """Revisa una idea.

    Parámetros:
        I: la idea (CH-1101). lex: léxico ({id: fila}).
        solo_produccion: True (lo normal) aplica R22. Solo los casos de prueba (§12) usan False,
            porque usan el léxico de validación LV01–LV18.
    Devuelve:
        Lista de errores, cada uno con su restricción (R1–R22) o su regla. Vacía si la idea es válida.
    Reglas: CH-1102 (valores), CH-1103 (R1–R22), R23 (realismo.yaml, dev/Camilo), CH-406, CH-603, CH-901, CH-907, CH-908, CH-909, CH-911,
        CH-912, CH-913, CH-915, CH-916, CH-917.
    Ejemplo:
        >>> validar_idea({**IDEA_CH1101, "tiempo": "FUT", "objeto_persona": 1}, LEX_PRUEBA)[0]
        'R1: con objeto 1, 2 o 4 el tiempo es SIMP'
    """
    E = []
    mal = lambda regla, texto: E.append(f"{regla}: {texto}")
    pred, mod = I.get("predicado"), I.get("modalidad")
    S = I.get("sujeto") or {}
    sp, pl, expreso = S.get("persona"), bool(S.get("plural")), S.get("expreso", "ninguno")
    t, asp = I.get("tiempo", "SIMP"), I.get("aspecto", "simple")
    op, inter = I.get("objeto_persona", 3), I.get("interrogativo")

    # Valores permitidos (CH-1102). Si alguno falla, no se sigue: las demás reglas lo necesitan.
    for campo, valor, permitidos in [
            ("predicado", pred, ESQ["predicado"]), ("modalidad", mod, ESQ["modalidad"]), ("tiempo", t, ESQ["tiempo"]),
            ("aspecto", asp, ESQ["aspecto"]), ("sujeto.persona", sp, ESQ["sujeto"]["persona"]),
            ("sujeto.expreso", expreso, ESQ["sujeto"]["expreso"]), ("sujeto.genero", S.get("genero"), ESQ["sujeto"]["genero"]),
            ("objeto_persona", op, ESQ["objeto_persona"]), ("interrogativo", inter, ESQ["interrogativo"]),
            ("posicion_wa", I.get("posicion_wa", "verbo"), ESQ["posicion_wa"])]:
        if valor not in permitidos:
            mal("CH-1102", f"{campo} = {valor!r} no es un valor permitido")
    if E:
        return E
    usados = ids_de_idea(I)
    for i in sorted({i for i in usados if usados.count(i) > 1}):            # R21
        nombre = lex[i]["es_m"] if i in lex else i
        mal("R21", f"el lexema '{nombre}' ({i}) aparece {usados.count(i)} veces en la idea")
    if solo_produccion:                                                     # R22
        for i in sorted({i for i in usados if not re.fullmatch(r"L\d{4,}", i)}):
            mal("R22", f"{i} no es una entrada L… del léxico de producción (§12.1: LV01–LV18 solo para pruebas)")

    def ent(i, cats, campo):
        e = lex.get(i)
        if e is None or e["cat"] not in cats:
            mal("CH-1102", f"{campo} = {i!r} no es un id de {'/'.join(cats)} del léxico")
            return None
        return e

    def revisar_np(np, campo, clases_ok=None, mods=MODS_NP, regla="R20"):
        """Revisa una frase nominal. Devuelve la fila del núcleo, o None."""
        if not isinstance(np, dict) or "n" not in np:
            mal("CH-1102", f"{campo}: falta la frase nominal")
            return None
        e = ent(np["n"], NOMBRES, campo + ".n")
        if e is None:
            return None
        for k in MODS_NP:
            if np.get(k) and k not in mods:
                mal("CH-908/CH-912", f"{campo}: aquí el nombre no admite {k}"
                    + (" (las frases de lugar no llevan numeral, v1.1.2)" if k == "num" and campo in ("lugar.np", "pnom.np") else ""))
        if np.get("num") not in ESQ["NP"]["num"]:
            mal("R13", f"{campo}: numeral {np.get('num')!r} no permitido (solo 3, 4, 6 y 10)")
        for k in ("dem", "poss", "genero"):
            if np.get(k) not in ESQ["NP"][k]:
                mal("CH-1102", f"{campo}.{k} = {np.get(k)!r} no es un valor permitido")
        if np.get("dem") and np.get("num"):
            mal("CH-912", f"{campo}: demostrativo y numeral a la vez")
        if np.get("dem") and (np.get("poss") or np.get("poseedor")):
            mal("CH-912", f"{campo}: el demostrativo no va con posesivo ni poseedor")
        if np.get("poss") and np.get("poseedor"):
            mal("CH-911", f"{campo}: posesivo y poseedor a la vez")
        if np.get("num") and np.get("plural"):
            mal("R12", f"{campo}: con numeral, plural = false")
        if (np.get("num") or np.get("plural")) and e["contable"] == "n":
            mal("CH-912", f"{campo}: '{e['es_m']}' no es contable: sin numeral ni plural (decisión 20-11-2026)")
        if e["genero"] == "var" and np.get("genero") not in ("m", "f"):
            mal("R19", f"{campo}: '{e['es_m']}' necesita genero (m o f)")
        if clases_ok is not None and e["clase_sem"] not in clases_ok:
            mal(regla, f"{campo}: la clase '{e['clase_sem']}' ({e['es_m']}) no vale aquí; valen {sorted(clases_ok)}")
        if np.get("adj"):
            a = ent(np["adj"], ["ADJ"], campo + ".adj")
            m = clases_de(a, "mod_clases") if a else None
            if m is not None and e["clase_sem"] not in m:
                mal("R20", f"{campo}: '{a['es_m']}' no modifica la clase '{e['clase_sem']}'")
        if np.get("poseedor"):
            if e["clase_sem"] == "tiempo":
                mal("R20", f"{campo}: un nombre de tiempo no se posee")
            revisar_np(np["poseedor"], campo + ".poseedor", {"persona"}, mods=())
        return e

    # Sujeto: forma y clase (CH-406: los pronombres de 1.ª, 2.ª y 4.ª persona y jupa cuentan como persona).
    clase_suj = None
    if inter == "quien_suj" or expreso == "pronombre" or (expreso == "ninguno" and sp != 3):
        clase_suj = {"persona"}
    if expreso in ("nombre", "coordinado", "demostrativo") and sp != 3:
        mal("CH-901", f"un sujeto '{expreso}' es de 3.ª persona")
    if expreso == "pronombre" and sp == 3 and S.get("genero") not in ("m", "f"):
        mal("R19", "el pronombre de 3.ª persona necesita genero")
    if expreso == "nombre":
        e = revisar_np(S.get("np"), "sujeto.np")
        if e:
            clase_suj = {e["clase_sem"]}
            if es_plural(S["np"]) != pl:
                mal("R11", "sujeto.plural debe coincidir con el plural (o el numeral) del nombre (CH-913)")
    elif expreso == "coordinado":
        coord = S.get("coord")
        if not (isinstance(coord, list) and len(coord) == 2):
            mal("R18", "la coordinación tiene exactamente 2 miembros (CH-907)")
        else:
            filas = [revisar_np(x, f"sujeto.coord[{k}]", mods=("adj",)) for k, x in enumerate(coord)]
            clase_suj = {f["clase_sem"] for f in filas if f}
            if any(x.get("plural") for x in coord):
                mal("R18", "los miembros coordinados van sin PL (CH-907)")
            if not pl:
                mal("R11", "un sujeto coordinado es plural (CH-907, CH-913)")
    elif expreso == "demostrativo" and S.get("dem") not in ("DEM.1", "DEM.2", "DEM.3"):
        mal("CH-1102", "sujeto demostrativo sin dem válido")

    if pred == "verbo":
        v = ent(I.get("verbo"), VERBOS, "verbo")
        if v is None:
            return E
        cat = v["cat"]
        suj = clases_de(v, "suj_clases")
        if suj is not None and clase_suj and not clase_suj <= suj:
            mal("R20", f"el sujeto ({sorted(clase_suj)}) no vale para '{v['es_m']}' (suj_clases {sorted(suj)})")
        if expreso == "demostrativo":
            mal("CH-901", "el demostrativo solo es sujeto de 'ser'")
        if op != 3:                                                       # R1, R2
            if t != "SIMP":
                mal("R1", "con objeto 1, 2 o 4 el tiempo es SIMP")
            if v.get("obj_persona") not in (None, "s"):
                mal("R1", f"'{v['es_m']}' no admite objeto de persona (obj_persona = {v.get('obj_persona')})")
            if (sp, op) not in TRANS:
                mal("R2", f"transición {sp}>{op} no permitida (solo 1>2, 2>1, 3>1, 3>2, 3>4)")
        if mod == "negativa" and (t not in ("SIMP", "FUT") or asp != "simple"):
            mal("R3", "la negativa va en SIMP o FUT, sin progresivo")
        if asp == "progresivo" and t != "SIMP":
            mal("R4", "el progresivo solo va en SIMP")
        if asp == "progresivo" and pl:
            mal("CH-603", "PROG y PL.V se excluyen: un sujeto plural no lleva progresivo")
        OI = I.get("oi")                                                  # R5, R6, CH-909
        if OI:
            f = OI.get("funcion")
            if f not in ("COMPL.RU", "BEN"):
                mal("R6", f"oi.funcion = {f!r}: es COMPL.RU o BEN (uno solo)")
            if f == "BEN" and cat not in ("V_TR", "V_DITR"):
                mal("R20", f"BEN solo con V_TR y V_DITR; un {cat} no lleva objeto indirecto (CH-406, v1.1.1)")
            elif f == "BEN" and v.get("admite_ben") not in (None, "s"):
                mal("R20", f"'{v['es_m']}' no admite BEN (admite_ben = {v.get('admite_ben')}, CH-406, v1.1.2)")
            if f == "BEN" and cat == "V_DITR" and op != 3:
                mal("R6", "con objeto 1, 2 o 4 el destinatario (COMPL.RU) es esa persona: no cabe BEN (CH-910)")
            if f == "COMPL.RU" and v.get("rige_ru") not in (None, "s"):
                mal("R5", f"'{v['es_m']}' no rige COMPL.RU (rige_ru = {v.get('rige_ru')})")
            sin_np = f == "COMPL.RU" and (op != 3 or inter == "quien_oi")
            if sin_np and OI.get("np"):
                mal("R5", "el destinatario es la persona de la transición o el interrogativo: COMPL.RU sin np (CH-909)")
            elif f == "COMPL.RU" and op != 3 and not I.get("oi_pronombre_expreso"):
                mal("CH-909", "COMPL.RU sin np solo con oi_pronombre_expreso = true o con quien_oi")
            elif not sin_np:
                revisar_np(OI.get("np"), "oi.np", {"persona"}, regla="R5" if f == "COMPL.RU" else "R20")
        if I.get("oi_pronombre_expreso") and not (op != 3 and cat == "V_DITR" and OI and OI.get("funcion") == "COMPL.RU"):
            mal("CH-909", "oi_pronombre_expreso solo con V_DITR, objeto 1, 2 o 4 y oi = COMPL.RU")
        od, odc = I.get("od"), I.get("od_coord")                         # R7, R8
        oc = clases_de(v, "obj_clases")
        oc = None if oc is None else oc - {"persona"}
        if cat in ("V_TR", "V_DITR"):
            if (op == 3 or cat == "V_DITR") and not (od or odc or inter == "que_od"):
                mal("R8", f"'{v['es_m']}' necesita objeto directo (CH-915)")
            if cat == "V_TR" and op != 3 and (od or odc):
                mal("R8", "con objeto 1, 2 o 4, un V_TR no lleva objeto directo (CH-915)")
        elif od or odc:
            mal("R8", f"un {cat} no lleva objeto directo (CH-401)")
        if od and odc:
            mal("R8", "od y od_coord a la vez")
        if inter == "que_od" and (od or odc):
            mal("R17", "que_od reemplaza al objeto directo: sin od")
        for k, x in ([("od", od)] if od else []) + ([(f"od_coord[{k}]", x) for k, x in enumerate(odc)] if odc else []):
            e = revisar_np(x, k, oc, mods=MODS_NP if k == "od" else ("adj",), regla="R7")
            if e and e["cat"] == "N_HUM":
                mal("R7", f"{k}: el objeto directo nunca es humano (CH-914)")
        if odc and (len(odc) != 2 or any(x.get("plural") for x in odc)):
            mal("R18", "od_coord: 2 miembros sin PL (CH-907)")
        Lg = I.get("lugar")                                               # R9, R10, CH-916
        if Lg:
            caso = Lg.get("caso")
            if caso not in ("ALL", "ABL", "LOC", "UNTIL"):
                mal("CH-1102", f"lugar.caso = {caso!r}")
            elif caso != "LOC" and cat != "V_MOV":
                mal("R9", f"{caso} solo con V_MOV")
            elif caso == "LOC" and (cat == "V_MOV" or v.get("loc_clases") == "NINGUNA"):
                mal("R9", f"'{v['es_m']}' no admite LOC")
            if Lg.get("deictico"):
                if caso != "LOC" or Lg.get("np") or Lg["deictico"] not in ("DEM.1", "DEM.2", "DEM.3"):
                    mal("CH-916", "el deíctico es DEM.x + LOC, sin np")
            elif inter == "donde_all":
                if caso != "ALL" or Lg.get("np"):
                    mal("R17", "donde_all: lugar = {caso: ALL}, sin np")
            elif caso in ("ALL", "ABL", "UNTIL", "LOC"):
                if caso == "LOC":
                    lc = v.get("loc_clases")
                    clases_ok = None if lc is None else LUGAR_GENERAL if lc == "-" else clases_de(v, "loc_clases")
                else:
                    clases_ok = LUGAR_GENERAL | ({"tiempo"} if caso == "UNTIL" else set())
                e = revisar_np(Lg.get("np"), "lugar.np", clases_ok, mods=tuple(m for m in MODS_NP if m != "num"))   # v1.1.2
                if caso == "UNTIL" and e:
                    if silabas(e["raiz"]) != 2:
                        mal("R10", f"UNTIL solo con raíces de 2 sílabas ('{e['raiz']}', CH-514)")
                    if Lg["np"].get("plural") or Lg["np"].get("poss") or Lg["np"].get("poseedor"):
                        mal("R10", "UNTIL: el nombre va sin plural ni posesivo (el poseedor añade POSS.3, CH-601)")
        elif inter == "donde_all":
            mal("R17", "donde_all necesita lugar = {caso: ALL}")
        C = I.get("comcausa")                                             # CH-406, CH-918
        if C:
            e = revisar_np(C.get("np"), "comcausa.np")
            if C.get("caso") not in ("COM", "CAUSE"):
                mal("CH-1102", f"comcausa.caso = {C.get('caso')!r}")
            elif C["caso"] == "CAUSE" and cat not in ("V_MOV", "V_INTR"):
                mal("R20", f"CAUSE solo con V_MOV y V_INTR, no con {cat} (CH-406, v1.1.2)")
            elif e and C["caso"] == "CAUSE" and e["clase_sem"] not in ("abstracto", "persona"):
                mal("R20", f"CAUSE: '{e['es_m']}' no es abstracto ni persona")
            elif e and C["caso"] == "COM":
                ic = clases_de(v, "instr_clases")
                instrumento = ic is None or e["clase_sem"] in ic
                compania = (e["clase_sem"] in ("persona", "animal") and cat in ("V_MOV", "V_INTR")
                            and clase_suj in (None, {"persona"}))
                if not (instrumento or compania):
                    mal("R20", f"COM: '{e['es_m']}' no es instrumento de '{v['es_m']}' ni compañía posible")
        if I.get("adv_t"):                                                # R14
            a = ent(I["adv_t"], ["ADV_T"], "adv_t")
            if a and (t not in TIEMPOS_ADV[a["subcat"]] or (a["subcat"].startswith("pasado") and asp == "progresivo")):
                mal("R14", f"'{a['es_m']}' ({a['subcat']}) no va con {t}{' progresivo' if asp == 'progresivo' else ''}")
        if mod == "pregunta_info":                                        # R17, CH-906
            if inter in (None, "donde_estar"):
                mal("R17", "una pregunta de información verbal necesita quien_suj, que_od, donde_all o quien_oi")
            elif inter == "quien_suj":
                if sp != 3 or pl:
                    mal("R17", "quien_suj: sujeto de 3.ª persona singular (CH-913)")
                if expreso != "ninguno":
                    mal("R17", "quien_suj reemplaza al sujeto: sin sujeto expreso")
            elif inter == "que_od" and (cat not in ("V_TR", "V_DITR") or op != 3):
                mal("R17", "que_od: V_TR o V_DITR con objeto de 3.ª persona")
            elif inter == "donde_all" and cat != "V_MOV":
                mal("R17", "donde_all: solo con V_MOV")
            elif inter == "quien_oi" and (cat != "V_DITR" or op != 3 or v.get("rige_ru") not in (None, "s")):
                mal("R17", "quien_oi: V_DITR con COMPL.RU y objeto de 3.ª persona")
        elif inter is not None:
            mal("R17", "interrogativo solo en pregunta_info")
        if (expreso == "coordinado" or odc) and (mod in ("negativa", "pregunta_info") or (C and C.get("caso") == "COM")):
            mal("R18", "coordinación prohibida en negativas, preguntas de información y con COM")
        if pl and mod == "negativa":                                      # R11 (CH-913)
            if expreso not in ("pronombre", "nombre"):
                mal("R11", "negativa con sujeto plural: sujeto expreso obligatorio (pronombre o nombre)")
            elif expreso == "nombre" and S["np"].get("num"):
                mal("R11", "negativa con sujeto plural: sin numeral")
    else:
        if t != "SIMP" or asp != "simple":                               # R15
            mal("R15", f"'{pred}' va siempre en SIMP simple")
        if not (mod in ("afirmativa", "pregunta_si_no") or (mod == "pregunta_info" and pred == "estar")):
            mal("R15", f"'{pred}' no admite la modalidad {mod}")
        if mod == "pregunta_info" and inter != "donde_estar":
            mal("R17", "la pregunta de información con 'estar' es donde_estar")
        elif mod != "pregunta_info" and inter is not None:
            mal("R17", "interrogativo solo en pregunta_info")
        for campo in ("verbo", "od", "od_coord", "oi", "lugar", "comcausa", "adv_t"):
            if I.get(campo):
                mal("CH-908", f"'{pred}' no lleva {campo} (ver PREGUNTAS.md, P-12 para adv_t)")
        if op != 3:
            mal("CH-908", f"'{pred}' no lleva objeto de persona")
        if expreso == "coordinado":
            mal("CH-907", "la coordinación de sujeto solo va con verbos")
        if expreso == "demostrativo" and pred != "ser":
            mal("CH-901", "el demostrativo solo es sujeto de 'ser'")
        if pl and (expreso == "ninguno" or (expreso == "nombre" and not S["np"].get("plural"))):
            mal("R11", f"'{pred}' con sujeto plural: sujeto expreso con -naka (CH-913)")
        P = I.get("pnom")
        if not isinstance(P, dict):
            mal("CH-1102", "falta pnom")
            return sorted(set(E))
        if pred == "ser":
            if ("adj" in P) == ("np" in P):
                mal("CH-1102", "ser: pnom es {adj} o {np}")
            elif "adj" in P:
                a = ent(P["adj"], ["ADJ"], "pnom.adj")
                m = clases_de(a, "mod_clases") if a else None
                if expreso == "demostrativo":
                    mal("R16", "con sujeto demostrativo, el predicado de 'ser' es un nombre")
                elif m is not None and clase_suj and not clase_suj <= m:
                    mal("R20", f"'{a['es_m']}' no se dice de la clase {sorted(clase_suj)}")
            else:
                e = revisar_np(P["np"], "pnom.np", mods=())
                if e and pl and e["contable"] == "n":
                    mal("CH-912", f"'{e['es_m']}' no es contable: no puede ser predicado plural (decisión 20-11-2026)")
                if e and expreso != "demostrativo" and not (e["cat"] == "N_HUM" and clase_suj in (None, {"persona"})):
                    mal("R16", "ser + nombre: sujeto demostrativo, o sujeto y nombre de persona")
        elif pred == "estar":
            if inter == "donde_estar":
                if P:
                    mal("R17", "donde_estar: pnom vacío ({})")
            elif P.get("deictico"):
                if P["deictico"] not in ("DEM.1", "DEM.2", "DEM.3") or P.get("np"):
                    mal("CH-916", "estar: deíctico DEM.x, sin np")
            else:
                revisar_np(P.get("np"), "pnom.np", CLASES_ESTAR_LUGAR, mods=("adj", "dem", "poss"))   # lugar sin numeral (v1.1.2)
            if clase_suj and clase_suj & CLASES_ESTAR_NO_SUJETO:
                mal("R20", f"'estar en': el sujeto no puede ser {sorted(clase_suj & CLASES_ESTAR_NO_SUJETO)}")
        else:
            revisar_np(P.get("np"), "pnom.np", CLASES_TENER, mods=("adj", "num"))
            if clase_suj not in (None, {"persona"}):
                mal("R20", "'tener': el poseedor es persona")
    E += errores_realismo(I, lex)                                   # R23 (realismo.yaml)
    return sorted(set(E))


print(validar_idea(IDEA_CH1101, LEX_PRUEBA) or "La idea de CH-1101 es válida.")
print(validar_idea({**IDEA_CH1101, "tiempo": "FUT", "objeto_persona": 1}, LEX_PRUEBA))

### Pruebas del validador

- Las 40 ideas de `casos_prueba.json` son válidas.
- Para cada restricción R1–R22 hay una idea estropeada a propósito, y el validador la rechaza citando esa restricción.
- El ejemplo de CH-1103: con FUT y objeto 1 falla R1; en SIMP da "El perro me ve.".

Las ideas que usan el léxico de validación (`LV01`–`LV18`) no tienen clases semánticas: para ellas no se revisa CH-406 (R20).

In [ ]:
# 9.2 · Pruebas del validador.
CASO = {c["id"]: c["idea"] for c in CASOS}


def cambiar(base, **campos):
    """Copia una idea y cambia algunos campos (None = quitar el campo)."""
    I = copy.deepcopy(CASO[base] if isinstance(base, str) else base)
    for k, v in campos.items():
        if v is None:
            I.pop(k, None)
        else:
            I[k] = v
    return I


def idea_v(verbo, persona=1, **extra):
    """Idea verbal afirmativa simple, con sujeto pronombre, para las pruebas."""
    S = {"persona": persona, "expreso": "pronombre", **({"genero": "m"} if persona == 3 else {})}
    return {"predicado": "verbo", "modalidad": "afirmativa", "tiempo": "SIMP", "verbo": verbo, "sujeto": S, **extra}


class PruebasValidador(unittest.TestCase):
    def falla(self, regla, idea):
        errores = validar_idea(idea, LEX_PRUEBA)
        self.assertTrue(tiene_regla(errores, regla), f"se esperaba {regla}; errores: {errores}")

    def test_CH1201_casos_validos(self):
        """CH-1103: las 40 ideas de los casos de prueba son válidas."""
        for c in CASOS:
            with self.subTest(caso=c["id"]):
                self.assertEqual(validar_idea(c["idea"], LEX_PRUEBA, solo_produccion=False), [])

    def test_CH1103_ejemplo(self):
        """CH-1103: FUT + objeto 1 → R1; en SIMP: "El perro me ve."."""
        I = {"predicado": "verbo", "modalidad": "afirmativa", "tiempo": "SIMP", "verbo": "L0017",
             "sujeto": {"persona": 3, "expreso": "nombre", "np": {"n": "L0008"}}, "objeto_persona": 1, "posicion_wa": "verbo"}
        self.falla("R1", {**I, "tiempo": "FUT"})
        r = generar(I, LEX_PRUEBA)
        self.assertEqual((r["b"], r["a"], r["es"]), ("Kirpuxa ch'ixrituwa.", "Kirpuxu ch'ixruquyi.", "El perro me ve."))

    def test_CH1102_valores(self):
        self.falla("CH-1102", cambiar("C01", tiempo="PASADO"))
        self.falla("CH-1102", cambiar("C18", verbo="L0001"))

    def test_R1(self):
        self.falla("R1", cambiar("C18", tiempo="FUT"))
        self.falla("R1", idea_v("L0016", 1, objeto_persona=2))             # comer: obj_persona = n

    def test_R2(self):
        self.falla("R2", cambiar("C18", objeto_persona=1))                 # 1>1

    def test_R3(self):
        self.falla("R3", cambiar("C22", tiempo="REM.C", objeto_persona=None))

    def test_R4(self):
        self.falla("R4", cambiar("C30", tiempo="FUT", adv_t=None))

    def test_R5(self):
        self.falla("R5", idea_v("L0018", 1, oi={"funcion": "COMPL.RU", "np": {"n": "L0008"}}, od={"n": "L0006"}))
        self.falla("R5", idea_v("L0016", 1, oi={"funcion": "COMPL.RU", "np": {"n": "L0001", "genero": "m"}}, od={"n": "L0006"}))

    def test_R6(self):
        self.falla("R6", cambiar("C01", oi={"funcion": "DAT", "np": {"n": "LV01", "genero": "m"}}))

    def test_R7(self):
        self.falla("R7", idea_v("L0017", 1, od={"n": "L0001", "genero": "m"}))   # OD humano
        self.falla("R7", idea_v("L0016", 1, od={"n": "L0010"}))                  # comer el palo

    def test_R8(self):
        self.falla("R8", idea_v("L0019", 1))                                     # comprar sin OD
        self.falla("R8", cambiar("C18", od={"n": "L0006"}))                      # V_TR con objeto 2 y OD

    def test_R9(self):
        self.falla("R9", idea_v("L0015", 1, lugar={"caso": "ALL", "np": {"n": "L0007"}}))
        self.falla("R9", idea_v("L0013", 1, lugar={"caso": "LOC", "np": {"n": "L0005"}}))

    def test_R10(self):
        self.falla("R10", idea_v("L0013", 1, lugar={"caso": "UNTIL", "np": {"n": "L0009"}}))   # pinqara: 3 sílabas
        self.falla("R10", idea_v("L0013", 1, lugar={"caso": "UNTIL", "np": {"n": "L0007", "poseedor": {"n": "L0003", "genero": "m"}}}))

    def test_R11(self):
        self.falla("R11", cambiar("C29", sujeto={"persona": 2, "plural": True, "expreso": "ninguno"}))

    def test_R12(self):
        self.falla("R12", idea_v("L0017", 1, od={"n": "L0008", "num": "NUM.3", "plural": True}))

    def test_R13(self):
        self.falla("R13", idea_v("L0017", 1, od={"n": "L0008", "num": "NUM.2"}))

    def test_R14(self):
        self.falla("R14", cambiar("C25", adv_t="L0024"))                         # mañana con REM.L
        self.falla("R14", cambiar("C25", adv_t="L0023"))                         # ayer (pasado) solo con SIMP (v1.1.2)
        self.assertEqual(validar_idea(cambiar("C25", adv_t="L0026"), LEX_PRUEBA), [])   # antes (pasado_remoto) con REM.L

    def test_R15(self):
        self.falla("R15", cambiar("C35", tiempo="FUT"))
        self.falla("R15", cambiar("C35", modalidad="negativa"))

    def test_R16(self):
        self.falla("R16", cambiar("C37", sujeto={"persona": 3, "expreso": "nombre", "np": {"n": "L0008"}}))

    def test_R17(self):
        self.falla("R17", cambiar("C32", verbo="L0013"))                         # que_od con V_MOV
        self.falla("R17", cambiar("C01", interrogativo="que_od"))                # interrogativo en afirmativa

    def test_R18(self):
        self.falla("R18", cambiar("C11", modalidad="negativa", posicion_wa=None))

    def test_R19(self):
        self.falla("R19", idea_v("L0015", 3, sujeto={"persona": 3, "expreso": "pronombre"}))
        self.falla("R19", cambiar("C26", oi={"funcion": "BEN", "np": {"n": "L0001"}}))

    def test_R20(self):
        self.falla("R20", idea_v("L0016", 3, sujeto={"persona": 3, "expreso": "nombre", "np": {"n": "L0005"}},
                                 od={"n": "L0006"}))                              # la casa come
        self.falla("R20", cambiar("C25", comcausa={"caso": "CAUSE", "np": {"n": "L0010"}}))
        self.falla("R20", idea_v("L0013", 1, oi={"funcion": "BEN", "np": {"n": "L0001", "genero": "m"}}))   # BEN con V_MOV
        self.falla("R20", idea_v("L0019", 1, od={"n": "L0006"}, comcausa={"caso": "COM", "np": {"n": "L0008"}}))  # comprar: sin COM
        self.falla("R20", idea_v("L0017", 1, od={"n": "L0006"}, comcausa={"caso": "CAUSE", "np": {"n": "L0012"}}))  # CAUSE con V_TR
        self.falla("R20", idea_v("L0016", 1, od={"n": "L0006"}, oi={"funcion": "BEN", "np": {"n": "L0001", "genero": "m"}}))  # comer: admite_ben = n

    def test_no_contable(self):
        """CH-912 (decisión 20-11-2026): un nombre no contable no lleva numeral ni plural."""
        self.falla("CH-912", idea_v("L0016", 1, od={"n": "L0006", "num": "NUM.3"}))     # tres panes
        self.falla("CH-912", idea_v("L0016", 1, od={"n": "L0006", "plural": True}))
        self.assertFalse(tiene_regla(validar_idea(idea_v("L0016", 1, od={"n": "L0006"}), LEX_PRUEBA), "CH-912"))

    def test_R22_ids_largos(self):
        """R22: los ids de 5 cifras (L10001) son de producción."""
        lex = {**LEX_PRUEBA, "L10001": {**LEX_PRUEBA["L0016"]}}
        self.assertFalse(tiene_regla(validar_idea(idea_v("L10001", 1, od={"n": "L0006"}), lex), "R22"))

    def test_lugar_sin_numeral(self):
        """CH-912 (v1.1.2): las frases de lugar (ALL, ABL, LOC, UNTIL) y el lugar de "estar" no llevan numeral."""
        self.falla("CH-912", idea_v("L0013", 1, lugar={"caso": "ALL", "np": {"n": "L0007", "num": "NUM.3"}}))
        self.falla("CH-912", cambiar("C38", pnom={"np": {"n": "L0005", "num": "NUM.4"}}))

    def test_R21(self):
        """R21: el mismo lexema dos veces ("para los niños con el niño")."""
        self.falla("R21", idea_v("L0013", 1, oi=None, comcausa={"caso": "COM", "np": {"n": "L0001", "genero": "m"}},
                                 lugar={"caso": "ALL", "np": {"n": "L0007", "adj": "L0021"}},
                                 sujeto={"persona": 3, "expreso": "nombre", "np": {"n": "L0001", "genero": "f"}}))
        self.falla("R21", cambiar("C11", sujeto={"persona": 3, "plural": True, "expreso": "coordinado",
                                                 "coord": [{"n": "LV15"}, {"n": "LV15"}]}))
        self.assertFalse(tiene_regla(validar_idea(IDEA_CH1101, LEX_PRUEBA), "R21"))

    def test_R22(self):
        """R22: una idea con una entrada LV no vale como dato de producción; los casos de prueba sí la usan."""
        self.falla("R22", CASO["C01"])
        self.assertEqual(validar_idea(CASO["C01"], LEX_PRUEBA, solo_produccion=False), [])


correr_pruebas(PruebasValidador, titulo="validador de ideas (R1–R22)")

In [ ]:
# 9.2b · Pruebas de la capa de realismo (R23). Solo con el léxico de producción y realismo.yaml.
@unittest.skipUnless(REALISMO_ACTIVO, "R23 inactivo: falta realismo.yaml o se usa el léxico de ejemplo")
class PruebasRealismo(unittest.TestCase):
    def idea(self, verbo, **extra):
        return {"predicado": "verbo", "modalidad": "afirmativa", "tiempo": "SIMP", "verbo": id_de(verbo, None),
                "sujeto": {"persona": 1, "expreso": "pronombre"}, "posicion_wa": "verbo", **extra}

    def test_fenomenos(self):
        """R23 §4: "Compro la lluvia" ✗; "Veo la lluvia" ✓; "Tengo un sol" ✗."""
        lluvia, sol = id_de("lluvia"), id_de("sol")
        self.assertIn("R23", " ".join(validar_idea(self.idea("comprar", od={"n": lluvia}), LEX)))
        self.assertEqual(validar_idea(self.idea("ver", od={"n": lluvia}), LEX), [])
        tener = {"predicado": "tener", "modalidad": "afirmativa", "sujeto": {"persona": 1, "expreso": "pronombre"},
                 "pnom": {"np": {"n": sol}}}
        self.assertIn("R23", " ".join(validar_idea(tener, LEX)))

    def test_abstractos(self):
        """R23 §1–2: "Digo la verdad" ✓; "Digo la libertad" ✗; los abstractos AUTO no conservados no se muestrean."""
        self.assertEqual(validar_idea(self.idea("decir", od={"n": id_de("verdad")}), LEX), [])
        self.assertIn("R23", " ".join(validar_idea(self.idea("decir", od={"n": id_de("libertad")}), LEX)))
        self.assertFalse(any(e["es_m"] == "cohesión" for e in LEX.values()))

    def test_movimiento_e_instrumentos(self):
        """R23 §5–6: "asistir" solo con ALL; "cortar el pan con la escoba" ✗, "con el cuchillo" ✓."""
        self.assertEqual(casos_mov(id_de("asistir")), {"ALL"})
        cortar = self.idea("cortar", od={"n": id_de("pan")})
        self.assertIn("R23", " ".join(validar_idea({**cortar, "comcausa": {"caso": "COM", "np": {"n": id_de("escoba")}}}, LEX)))
        self.assertEqual(validar_idea({**cortar, "comcausa": {"caso": "COM", "np": {"n": id_de("cuchillo")}}}, LEX), [])

    def test_espanol(self):
        """R23 §9: "Yo cuido al perro."; CH-1006: "Nosotros somos grandes." también con la 4.ª persona singular."""
        cuidar = {**self.idea("cuidar", od={"n": id_de("perro")}), "sujeto": {"persona": 1, "expreso": "pronombre"}}
        self.assertEqual(oracion_espanol(cuidar, LEX), "Yo cuido al perro.")
        ser = {"predicado": "ser", "modalidad": "afirmativa", "sujeto": {"persona": 4, "expreso": "pronombre"},
               "pnom": {"adj": id_de("grande", "ADJ")}}
        self.assertEqual(oracion_espanol(ser, LEX), "Nosotros somos grandes.")


correr_pruebas(PruebasRealismo, titulo="capa de realismo (R23)")

### Generador de ideas

`generar_ideas(n, lex, semilla)` devuelve `n` ideas válidas y una tabla con las cuotas (CH-1104).

**Decisiones que la especificación no fija** (todas en PREGUNTAS.md):
- **P-10.** La persona del sujeto la fija la cuota. Por eso la transición (CH-1109) se sortea solo entre las que tienen esa persona de sujeto.
- **P-11** (corregida en la v1.1.1). El rango es dentro de cada categoría. Para el verbo, primero se sortea la categoría con `categoria_verbo` y después se aplica Zipf dentro de ella. Para el sujeto, si el verbo acepta `persona` y otras clases, es `N_HUM` con probabilidad 0,60 (`sujeto_clase_humana`, v1.1.2). En las demás funciones con las dos categorías (compañía, CAUSE), las dos pesan lo mismo (P-11).
- **R21 y R22** (v1.1.1). Un lexema no se repite en la idea, y nunca se usan las entradas `LV01`–`LV18`.
- **P-12.** "Ser", "estar" y "tener" no llevan adverbio de tiempo: el español de §10 no lo escribe.
- **P-13.** COM es instrumento si el verbo acepta instrumentos (`instr_clases`); si no, es compañía (solo `V_MOV` o `V_INTR` con sujeto `persona`). Si no cabe ninguna de las dos, no hay COM.
- **BEN** solo va con `V_TR` y `V_DITR` con `admite_ben = s`; **CAUSE** solo con `V_MOV` y `V_INTR`; las frases de lugar no llevan numeral (v1.1.2).

La cobertura mínima de cada palabra (CH-404) se aplica al generar el conjunto completo (sección 10).

In [ ]:
# 9.3 · Generador de ideas (CH-1104 a CH-1113).
EXP_ZIPF = 0.9                                     # CH-403
PC = PESOS["complementos"]
_INDICES = {}
FORZAR = {"id": None, "verbos": None}              # CH-404: entrada que se fuerza al completar la cobertura (sección 10)
                                                   # y, con R23, los verbos que la admiten


def indice(lex):
    """Índice {(categoría, clase): [ids]} de un léxico (se calcula una vez por léxico)."""
    if id(lex) not in _INDICES:
        ix = {}
        for i, e in lex.items():
            ix.setdefault((e["cat"], e.get("clase_sem", "-")), []).append(i)
        _INDICES[id(lex)] = ix
    return _INDICES[id(lex)]


def elegir(rng, lex, cats, clases=None, filtro=None, pesos_cat=None):
    """Elige una palabra del léxico: primero la categoría, después Zipf sobre su rango.

    Parámetros:
        rng: random.Random. lex: léxico. cats: categorías aceptadas. clases: clases semánticas aceptadas (None = todas).
        filtro: función opcional sobre la fila.
        pesos_cat: pesos de las categorías (categoria_verbo, sujeto_clase_humana). Si se omite y hay varias
            categorías con candidatas, todas pesan lo mismo (P-11).
    Devuelve:
        Un id, o None si no hay candidatas.
    Reglas: CH-402 (el rango es dentro de la categoría), CH-403 (Zipf sobre las candidatas, en su orden de rango),
        CH-406 (las clases filtran antes), P-11 (categoria_verbo).
    Ejemplo:
        >>> elegir(random.Random(1), LEX, ["V_MOV"])
        'L0013'
    """
    grupos = {}
    for (c, k), lst in indice(lex).items():
        if c in cats and (clases is None or k in clases):
            grupos.setdefault(c, []).extend(i for i in lst if not filtro or filtro(lex[i]))
    grupos = {c: g for c, g in grupos.items() if g}
    if FORZAR.get("verbos") and grupos and set(grupos) <= set(VERBOS):     # CH-404 + R23: un verbo que admita la entrada
        compatibles = {c: [i for i in g if i in FORZAR["verbos"]] for c, g in grupos.items()}
        grupos = {c: g for c, g in compatibles.items() if g} or grupos
    if not grupos:
        return None
    if FORZAR["id"] and any(FORZAR["id"] in g for g in grupos.values()):
        return FORZAR["id"]                                         # CH-404: si la entrada cabe aquí, sale
    if len(grupos) == 1:
        ids = next(iter(grupos.values()))
    elif pesos_cat:
        cat = sortear_entre(pesos_cat, grupos, rng)
        if cat is None:
            return None
        ids = grupos[cat]
    else:
        ids = grupos[rng.choice(sorted(grupos))]                    # P-11: categorías con el mismo peso
    ids = sorted(ids, key=lambda i: (int(lex[i]["rango"]), i))
    return rng.choices(ids, weights=[1 / (k + 1) ** EXP_ZIPF for k in range(len(ids))])[0]


def sortear_entre(pesos, posibles, rng):
    """Sortea una clave de `pesos` entre las `posibles`, renormalizando. None si no queda ninguna."""
    p = {k: w for k, w in pesos.items() if k in posibles and w > 0}
    return sortear(p, rng) if p else None


def np_al_azar(rng, lex, clases, rol, plural=False, mods=MODS_NP, plural_ok=True, filtro=None, pesos_cat=None):
    """Sortea una frase nominal: el nombre, su modificador y su número.

    Parámetros:
        rng, lex: generador y léxico. clases: clases aceptadas del nombre (None = todas).
        rol: "sujeto" (el plural lo decide `plural`) u otro (plural con probabilidad plural_no_sujeto).
        mods: modificadores permitidos. plural_ok: False prohíbe el plural. filtro: función opcional sobre el nombre.
        pesos_cat: pesos de N_HUM / N_NOHUM (para el sujeto: sujeto_clase_humana).
    Devuelve:
        La frase nominal (dict), o None si no hay nombre posible.
    Reglas: CH-1112 (pesos), CH-912 (un modificador), CH-911 (poseedor N_HUM sin modificadores),
        CH-913 (sujeto plural: -naka o numeral), R12, R19 (género de los N_HUM "var").
    Ejemplo:
        >>> np_al_azar(random.Random(3), LEX, {"animal"}, "od")
        {'n': 'L0008'}
    """
    n = elegir(rng, lex, NOMBRES, clases, filtro, pesos_cat)
    if n is None:
        return None
    e, np = lex[n], {"n": n}
    if e["genero"] == "var":
        np["genero"] = sortear(PESOS["sujeto"]["genero"], rng)
    posibles = {"ninguno"} | set(mods)
    if e["clase_sem"] == "tiempo":
        posibles -= {"poss", "poseedor"}
    if (rol == "sujeto" and not plural) or e["contable"] != "s":     # CH-912: no contable, sin numeral
        posibles.discard("num")
    adj_ok = lambda a: e["clase_sem"] in (clases_de(a, "mod_clases") or set()) and adj_realista(e, a)   # R23
    if elegir(rng, lex, ["ADJ"], filtro=adj_ok) is None:
        posibles.discard("adj")
    m = sortear_entre(PESOS["np"]["modificador"], posibles, rng)
    if m == "adj":
        np["adj"] = elegir(rng, lex, ["ADJ"], filtro=adj_ok)
    elif m == "dem":
        np["dem"] = sortear(PESOS["np"]["dem"], rng)
    elif m == "num":
        np["num"] = sortear(PESOS["np"]["num"], rng)
    elif m == "poss":
        np["poss"] = int(sortear(PESOS["np"]["poss"], rng))
    elif m == "poseedor":
        np["poseedor"] = np_al_azar(rng, lex, {"persona"}, "poseedor", mods=())
    if rol == "sujeto":
        if plural and m != "num":
            np["plural"] = True
    elif plural_ok and m != "num" and e["contable"] == "s" and rng.random() < PESOS["np"]["plural_no_sujeto"]:
        np["plural"] = True
    return np


def sujeto_al_azar(rng, lex, sp, plural, opciones_3, clases_nombre, mods=MODS_NP):
    """Sortea el sujeto de una idea.

    Parámetros:
        sp: persona. plural: bool. opciones_3: formas posibles en 3.ª persona (nombre, pronombre, ninguno, coordinado).
        clases_nombre: clases aceptadas para un sujeto nombre. mods: modificadores del nombre.
    Devuelve:
        (sujeto, clase del sujeto) o (None, None) si no es posible.
    Reglas: CH-1110, CH-913 (un sujeto plural se expresa si hace falta: lo decide quien llama), CH-907, R19.
    """
    pesos = PESOS["sujeto"]["expreso_3"] if sp == 3 else PESOS["sujeto"]["expreso_1_2_4"]
    humano = {"N_HUM": PESOS["sujeto_clase_humana"], "N_NOHUM": 1 - PESOS["sujeto_clase_humana"]}   # P-11 (v1.1.2)
    expreso = sortear_entre(pesos, opciones_3 if sp == 3 else {"pronombre", "ninguno"}, rng)
    if expreso is None:
        return None, None
    S = {"persona": sp, "expreso": expreso}
    if expreso == "coordinado":
        plural = True
    if plural:
        S["plural"] = True
    clase = {"persona"} if expreso == "pronombre" or (expreso == "ninguno" and sp != 3) else None
    if expreso == "pronombre" and sp == 3:
        S["genero"] = sortear(PESOS["sujeto"]["genero"], rng)
    elif expreso == "nombre":
        contable = (lambda e: e["contable"] == "s") if plural else None                # CH-912: plural solo si es contable
        S["np"] = np_al_azar(rng, lex, clases_nombre, "sujeto", plural=plural, mods=mods, pesos_cat=humano, filtro=contable)
        if S["np"] is None:
            return None, None
        clase = {lex[S["np"]["n"]]["clase_sem"]}
    elif expreso == "coordinado":
        S["coord"] = [np_al_azar(rng, lex, clases_nombre, "coord", mods=("adj",), plural_ok=False, pesos_cat=humano)
                      for _ in range(2)]
        if None in S["coord"]:
            return None, None
        clase = {lex[x["n"]]["clase_sem"] for x in S["coord"]}
    return S, clase


def idea_verbal(rng, lex, mod, t, sp):
    """Sortea una idea verbal para la combinación (modalidad, tiempo, persona). Puede salir inválida."""
    I = {"predicado": "verbo", "modalidad": mod, "tiempo": t}
    acepta_persona = lambda e: "persona" in clases_de(e, "suj_clases")
    vid = elegir(rng, lex, VERBOS, filtro=None if sp == 3 else acepta_persona, pesos_cat=PESOS["categoria_verbo"])   # P-11
    if vid is None:
        return None
    I["verbo"], v = vid, lex[vid]
    cat, suj = v["cat"], clases_de(v, "suj_clases")
    op = 3
    if t == "SIMP" and v["obj_persona"] == "s":                          # CH-1109 (P-10)
        T = sortear_entre(PESOS["transicion"], {"objeto_3"} | {x for (s, o), x in TRANS.items() if s == sp}, rng)
        if T != "objeto_3":
            op = next(o for (s, o), x in TRANS.items() if x == T)
            I["objeto_persona"] = op
    inter = None
    if mod == "pregunta_info":                                           # CH-1113, CH-906
        pos = set()
        if sp == 3 and "persona" in suj:
            pos.add("quien_suj")
        if cat in ("V_TR", "V_DITR") and op == 3:
            pos.add("que_od")
        if cat == "V_MOV" and "ALL" in casos_mov(vid):                  # R23 (sección 6)
            pos.add("donde_all")
        if cat == "V_DITR" and op == 3 and v["rige_ru"] == "s":
            pos.add("quien_oi")
        inter = sortear_entre(PESOS["interrogativo"]["verbo"], pos, rng)
        if inter is None:
            return None
        I["interrogativo"] = inter
    if inter == "quien_suj":
        S, clase_suj = {"persona": 3}, {"persona"}
    else:
        plural = rng.random() < PESOS["sujeto"]["plural"]
        opciones = {"nombre", "ninguno"} | ({"pronombre"} if "persona" in suj else set())
        if mod not in ("negativa", "pregunta_info"):
            opciones.add("coordinado")                                   # R18
        if plural and mod == "negativa":
            opciones.discard("ninguno")                                  # R11
        S, clase_suj = sujeto_al_azar(rng, lex, sp, plural, opciones, suj,
                                      mods=MODS_NP if mod != "negativa" else tuple(m for m in MODS_NP if m != "num"))
        if S is None or (plural and mod == "negativa" and S["expreso"] == "ninguno"):
            return None
    I["sujeto"] = S
    if t == "SIMP" and mod != "negativa" and not S.get("plural") \
            and rng.random() < PESOS["aspecto"]["progresivo_si_SIMP_y_no_negativa"]:   # CH-1107, CH-603
        I["aspecto"] = "progresivo"
    coord = S.get("expreso") == "coordinado"
    if cat in ("V_TR", "V_DITR") and inter != "que_od" and (op == 3 or cat == "V_DITR"):   # CH-915
        oc = clases_de(v, "obj_clases") - {"persona"}
        if mod not in ("negativa", "pregunta_info") and rng.random() < PC["od_coord"]:
            I["od_coord"] = [np_al_azar(rng, lex, oc, "coord", mods=("adj",), plural_ok=False,
                                        filtro=lambda e: nombre_ok(vid, "od", e)) for _ in range(2)]   # R23
            if None in I["od_coord"]:
                return None
            coord = True
        else:
            I["od"] = np_al_azar(rng, lex, oc, "od", filtro=lambda e: nombre_ok(vid, "od", e))   # R23
            if I["od"] is None:
                return None
    if op != 3 and cat == "V_DITR" and rng.random() < PC["oi_pronombre_expreso"]:   # CH-909
        I["oi"], I["oi_pronombre_expreso"] = {"funcion": "COMPL.RU"}, True
    if inter == "donde_all":
        I["lugar"] = {"caso": "ALL"}
    if inter == "quien_oi":
        I["oi"] = {"funcion": "COMPL.RU"}

    tipos = {"tiempo", "comcausa"}                                        # CH-1111: complementos opcionales
    if "lugar" not in I and (cat == "V_MOV" or v["loc_clases"] != "NINGUNA"):
        tipos.add("lugar")
    ben = v["admite_ben"] == "s" and (BEN_VERBOS is None or vid in BEN_VERBOS)   # CH-406 (v1.1.2), R23 (sección 15)
    if "oi" not in I and ((cat == "V_TR" and ben) or (cat == "V_DITR" and op == 3)):   # BEN solo con V_TR y V_DITR
        tipos.add("oi")
    for _ in range(int(sortear(PC["cuantos_opcionales"], rng))):
        tipo = sortear_entre(PC["tipo"], tipos, rng)
        if tipo is None:
            break
        tipos.discard(tipo)
        if tipo == "tiempo":
            ok = [s for s, ts in TIEMPOS_ADV.items() if t in ts and not (s.startswith("pasado") and I.get("aspecto") == "progresivo")]
            adv = elegir(rng, lex, ["ADV_T"], filtro=lambda e: e["subcat"] in ok)
            if adv:
                I["adv_t"] = adv
        elif tipo == "lugar" and cat == "V_MOV":
            caso = sortear_entre(PC["lugar_V_MOV"], casos_mov(vid), rng)                  # R23 (sección 6)
            if caso is None:
                continue
            dos = (lambda e: silabas(e["raiz"]) == 2) if caso == "UNTIL" else None          # CH-514
            np = np_al_azar(rng, lex, LUGAR_GENERAL | ({"tiempo"} if caso == "UNTIL" else set()), "lugar",
                            mods=tuple(m for m in MODS_NP if m != "num" and not (caso == "UNTIL" and m in ("poss", "poseedor"))),   # v1.1.2, R10
                            plural_ok=caso != "UNTIL", filtro=dos)
            if np:
                I["lugar"] = {"caso": caso, "np": np}
        elif tipo == "lugar":
            if sortear(PC["lugar_LOC"], rng) == "deictico":                               # CH-916
                I["lugar"] = {"caso": "LOC", "deictico": sortear(PESOS["np"]["dem"], rng)}
            else:
                lc = LUGAR_GENERAL if v["loc_clases"] == "-" else clases_de(v, "loc_clases")
                np = np_al_azar(rng, lex, lc, "lugar", mods=tuple(m for m in MODS_NP if m != "num"))   # v1.1.2
                if np:
                    I["lugar"] = {"caso": "LOC", "np": np}
        elif tipo == "comcausa":
            ic = clases_de(v, "instr_clases")
            compania = cat in ("V_MOV", "V_INTR") and clase_suj == {"persona"}                 and (vid in COMPANIA or not REALISMO_ACTIVO)                                       # R23 (sección 13)
            suj_persona = (clase_suj or clases_de(v, "suj_clases")) == {"persona"}
            causa = {"CAUSE"} if cat in ("V_MOV", "V_INTR") and (not REALISMO_ACTIVO or (suj_persona and vid not in CAUSA_SIN))                 else set()                                                                         # CH-406 (v1.1.2), R23 (sección 14)
            caso = sortear_entre(PC["comcausa"], causa | ({"COM"} if (ic or compania) and not coord else set()), rng)
            if caso is None:
                continue
            clases = {"abstracto", "persona"} if caso == "CAUSE" else ic or {"persona", "animal"}   # P-13
            if caso == "CAUSE":                                                            # R23 (secciones 5 y 7)
                ok = lambda e: e["clase_sem"] != "abstracto" or e["id"] in CAUSA_ABS or not REALISMO_ACTIVO
            else:
                ok = lambda e: (instrumento_ok(vid, e) if e["clase_sem"] in (ic or set())
                                else e["clase_sem"] != "animal" or vid in COMPANIA_ANIMAL or not REALISMO_ACTIVO)
            np = np_al_azar(rng, lex, clases, "comcausa", filtro=lambda e: ok(e) and e["id"] not in FENOMENOS)
            if np:
                I["comcausa"] = {"caso": caso, "np": np}
        else:
            f = sortear_entre(PC["oi_V_DITR"] if cat == "V_DITR" else PC["oi_V_TR"], {"COMPL.RU"} | ({"BEN"} if ben else set()), rng)
            np = np_al_azar(rng, lex, {"persona"}, "oi")
            if np:
                I["oi"] = {"funcion": f, "np": np}
    if mod == "afirmativa":
        I["posicion_wa"] = sortear(PESOS["posicion_wa"], rng)                             # CH-1113
    return I


def idea_nominal(rng, lex, pred, mod, sp):
    """Sortea una idea de "ser", "estar" o "tener" (CH-908). Puede salir inválida."""
    I = {"predicado": pred, "modalidad": mod}
    plural = rng.random() < PESOS["sujeto"]["plural"]
    opciones = {"nombre", "pronombre"} | (set() if plural else {"ninguno"})              # R11: plural expreso
    sin_num = tuple(m for m in MODS_NP if m != "num")                                     # R11: plural con -naka
    if pred == "ser":
        if sortear(PESOS["ser"]["pnom"], rng) == "adj":
            S, clase = sujeto_al_azar(rng, lex, sp, plural, opciones, None, mods=sin_num)
            if S is None:
                return None
            adj = elegir(rng, lex, ["ADJ"], filtro=lambda e: clase is None or clase <= clases_de(e, "mod_clases"))
            if adj is None:
                return None
            I["pnom"] = {"adj": adj}
        elif sp == 3 and sortear(PESOS["ser"]["sujeto_con_np"], rng) == "demostrativo":
            S = {"persona": 3, "expreso": "demostrativo", "dem": sortear(PESOS["np"]["dem"], rng)}
            if plural:
                S["plural"] = True
            dem_ok = lambda e: e["clase_sem"] != "abstracto" or e["id"] in SER_DEM_ABS or not REALISMO_ACTIVO   # R23
            np = np_al_azar(rng, lex, None, "pnom", mods=(), filtro=lambda e: dem_ok(e) and (e["contable"] == "s" or not plural))
            if np is None:
                return None
            np.pop("plural", None)
            I["pnom"] = {"np": np}
        else:                                                                             # humano + nombre humano
            S, clase = sujeto_al_azar(rng, lex, sp, plural, opciones, {"persona"}, mods=sin_num)
            if S is None:
                return None
            np = np_al_azar(rng, lex, {"persona"}, "pnom", mods=())
            if np is None:
                return None
            np.pop("plural", None)
            if "genero" in np:                                        # el nombre predicado concuerda con el sujeto
                g = S.get("genero") or (genero_np(S["np"], lex) if "np" in S else None)
                np["genero"] = g or np["genero"]
            I["pnom"] = {"np": np}
    elif pred == "estar":
        clases_suj = set(CLASES_SEM) - CLASES_ESTAR_NO_SUJETO
        S, clase = sujeto_al_azar(rng, lex, sp, plural, opciones, clases_suj, mods=sin_num)
        if S is None:
            return None
        if mod == "pregunta_info":
            I["interrogativo"], I["pnom"] = "donde_estar", {}
        elif sortear(PC["lugar_LOC"], rng) == "deictico":
            I["pnom"] = {"deictico": sortear(PESOS["np"]["dem"], rng)}
        else:
            np = np_al_azar(rng, lex, CLASES_ESTAR_LUGAR, "pnom", mods=("adj", "dem", "poss"))   # lugar sin numeral (v1.1.2)
            if np is None:
                return None
            I["pnom"] = {"np": np}
    else:
        S, clase = sujeto_al_azar(rng, lex, sp, plural, opciones, {"persona"}, mods=sin_num)
        if S is None:
            return None
        np = np_al_azar(rng, lex, CLASES_TENER, "pnom", mods=("adj", "num"), filtro=lambda e: e["id"] not in FENOMENOS)   # R23
        if np is None:
            return None
        I["pnom"] = {"np": np}
    I["sujeto"] = S
    return I


def idea_al_azar(rng, lex, combo):
    """Sortea una idea para una combinación (predicado, modalidad, tiempo, persona). Puede ser None o inválida."""
    pred, mod, t, sp = combo
    return idea_verbal(rng, lex, mod, t, sp) if pred == "verbo" else idea_nominal(rng, lex, pred, mod, sp)

In [ ]:
# 9.4 · Cuotas (CH-1104) y generar_ideas().
from collections import Counter

MAX_INTENTOS_IDEA = 500          # intentos por idea antes de dar la combinación por imposible


def combinaciones():
    """Combinaciones predicado × modalidad × tiempo × persona con su peso (producto de los marginales).

    Devuelve:
        Lista [((predicado, modalidad, tiempo, persona), peso)]. Los pesos suman 1.
    Reglas: CH-1104 (1), CH-1105, CH-1106, CH-1108. Los predicados nominales van siempre en SIMP (R15).
    Ejemplo:
        >>> combinaciones()[0]
        (('verbo', 'afirmativa', 'SIMP', 1), 0.03696)
    """
    out = []
    for pm, w in PESOS["predicado_modalidad"].items():
        pred, mod = pm.split(".")
        tiempos = PESOS["tiempo"]["negativa" if mod == "negativa" else "general"] if pred == "verbo" else {"SIMP": 1.0}
        for t, wt in tiempos.items():
            for sp, ws in PESOS["persona_sujeto"].get(t, PESOS["persona_sujeto"]["general"]).items():
                out.append(((pred, mod, t, int(sp)), round(w * wt * ws, 10)))
    return out


def repartir(pesos, n):
    """Reparte n entre claves según sus pesos (método del resto mayor). Las cuotas suman n exactamente."""
    total = sum(pesos.values())
    exactas = {k: n * w / total for k, w in pesos.items()}
    cuotas = {k: int(x) for k, x in exactas.items()}
    for k in sorted(exactas, key=lambda k: exactas[k] - cuotas[k], reverse=True)[:n - sum(cuotas.values())]:
        cuotas[k] += 1
    return cuotas


def intentar_idea(rng, lex, combo, intentos=MAX_INTENTOS_IDEA, aceptar=None):
    """Sortea ideas de una combinación hasta que una sea válida (y, si se da, `aceptar(idea)` sea True).
    Devuelve (idea o None, intentos usados)."""
    for k in range(1, intentos + 1):
        I = idea_al_azar(rng, lex, combo)
        if I is not None and not validar_idea(I, lex) and (aceptar is None or aceptar(I)):
            return I, k
    return None, intentos


def presupuesto_dv(n, semilla, lex):
    """Función aceptar() que limita las reglas [DERIVADA] (DC-68, V30).

    Una idea sin DV se acepta siempre. Una idea con DV se rechaza si alguna de sus DV ya llegó a
    MAX_PROP_POR_DV · n ideas, y si no, con probabilidad PROB_RECHAZO_DV. Al rechazarla, intentar_idea()
    sortea otra idea de la misma combinación, así que las cuotas de CH-1104 no cambian.
    """
    rng_dv, cuenta, tope = random.Random(semilla + 30), Counter(), max(1, int(MAX_PROP_POR_DV * n))

    def aceptar(I):
        dv = generar(I, lex)["dv"]
        if not dv:
            return True
        if any(cuenta[d] >= tope for d in dv) or rng_dv.random() < PROB_RECHAZO_DV:
            return False
        cuenta.update(dv)
        return True
    return aceptar


def generar_ideas(n, lex, semilla=SEMILLA):
    """Genera n ideas válidas con las cuotas de CH-1104.

    Parámetros:
        n: número de ideas. lex: léxico. semilla: semilla de la generación.
    Devuelve:
        (ideas, tabla). ideas: lista de ideas con id "I000001"…, en orden aleatorio.
        tabla: DataFrame con cada combinación, su cuota, las ideas generadas y los intentos.
    Lanza:
        RuntimeError si una combinación posible deja de producir ideas válidas.
    Reglas: CH-1104 (cuotas; las imposibles se quitan y su cuota se reparte), CH-1105 a CH-1113, CH-1103.
    Ejemplo:
        >>> ideas, tabla = generar_ideas(100, LEX, 1)
        >>> len(ideas)
        100
    """
    rng = random.Random(semilla)
    combos = combinaciones()
    primera, intentos = {}, {}
    for combo, _ in combos:                             # 3. ¿es posible la combinación con este léxico?
        primera[combo], intentos[combo] = intentar_idea(rng, lex, combo)
    posibles = {c: w for c, w in combos if primera[c] is not None}
    cuotas = repartir(posibles, n)
    aceptar = presupuesto_dv(n, semilla, lex) if PROB_RECHAZO_DV > 0 else None   # DC-68, V30
    ideas = []
    for combo, q in cuotas.items():
        lista = [primera[combo]][:q]
        while len(lista) < q:
            I, k = intentar_idea(rng, lex, combo, aceptar=aceptar)
            intentos[combo] += k
            if I is None:
                raise RuntimeError(f"CH-1104: la combinación {combo} dejó de producir ideas válidas.")
            lista.append(I)
        ideas += lista
    rng.shuffle(ideas)
    ideas = [{"id": f"I{k:06d}", **I} for k, I in enumerate(ideas, 1)]
    tabla = pd.DataFrame([{"predicado": c[0], "modalidad": c[1], "tiempo": c[2], "persona": c[3], "peso": w,
                           "posible": c in posibles, "cuota": cuotas.get(c, 0), "intentos": intentos[c]} for c, w in combos])
    return ideas, tabla


IDEAS_MUESTRA, TABLA_CUOTAS = generar_ideas(300, LEX, SEMILLA)
imposibles = TABLA_CUOTAS[~TABLA_CUOTAS["posible"]]
print(f"{len(IDEAS_MUESTRA)} ideas de muestra · {len(TABLA_CUOTAS)} combinaciones, {len(imposibles)} imposibles con este léxico"
      f" · {TABLA_CUOTAS['intentos'].sum() / len(IDEAS_MUESTRA):.1f} intentos por idea.")
if len(imposibles):
    display(imposibles[["predicado", "modalidad", "tiempo", "persona", "peso"]])
with pd.option_context("display.max_colwidth", None):
    MUESTRA_30 = random.Random(SEMILLA).sample(IDEAS_MUESTRA, 30)          # para revisar a mano que se lean bien
    display(pd.DataFrame([{"id": I["id"], **{k: v for k, v in generar(I, LEX).items() if k in ("b", "a", "es")}}
                          for I in MUESTRA_30]).set_index("id"))

### Pruebas del generador de ideas

- **CH-1104:** las cuotas suman el total, y la proporción de cada tipo de oración es la de los pesos.
- **CH-1103:** todas las ideas generadas son válidas, y `generar()` las convierte en B, A y español sin errores (CH-709).
- **Reproducible:** con la misma semilla salen las mismas ideas.
- **R21, R22, BEN y CAUSE** (v1.1.1 y v1.1.2): en 10.000 ideas no hay ningún id LV ni lexemas repetidos; BEN solo sale con `V_TR` y `V_DITR` con `admite_ben = s`; CAUSE solo con `V_MOV` y `V_INTR`; ningún lugar lleva numeral; el sujeto es humano cerca del 60 % de las veces que puede no serlo.
- **P-11 y P-13:** la categoría del verbo sale con sus pesos; COM de compañía solo con `V_MOV` o `V_INTR` y sujeto persona.
- **Pesos:** las proporciones de `posicion_wa`, del plural y del progresivo se acercan a las de `pesos.yaml`.

In [ ]:
# 9.5 · Pruebas del generador de ideas.
class PruebasIdeas(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.ideas, cls.tabla = generar_ideas(10_000, LEX, 7)

    def test_CH1104_cuotas(self):
        """CH-1104: las cuotas suman n, y cada predicado.modalidad tiene su peso (sobre las combinaciones posibles)."""
        T = self.tabla[self.tabla["posible"]]
        self.assertEqual(T["cuota"].sum(), 10_000)
        self.assertEqual(len(self.ideas), 10_000)
        real = pd.Series([f"{I['predicado']}.{I['modalidad']}" for I in self.ideas]).value_counts(normalize=True)
        pesos = T.assign(pm=T["predicado"] + "." + T["modalidad"]).groupby("pm")["peso"].sum()
        pesos /= pesos.sum()
        for pm, w in pesos.items():
            self.assertAlmostEqual(real.get(pm, 0), w, delta=0.002, msg=pm)

    def test_CH1103_validas_y_construibles(self):
        """CH-1103 y CH-709: todas las ideas son válidas y generar() da B, A y español."""
        for I in self.ideas:
            self.assertEqual(validar_idea(I, LEX), [], I)
            r = generar(I, LEX)
            self.assertTrue(r["b"] and r["a"] and r["es"])

    def test_reproducible(self):
        """La misma semilla da las mismas ideas; otra semilla, otras."""
        a, _ = generar_ideas(200, LEX, 11)
        b, _ = generar_ideas(200, LEX, 11)
        c, _ = generar_ideas(200, LEX, 12)
        self.assertEqual(a, b)
        self.assertNotEqual(a, c)

    def test_CH1113_posicion_wa(self):
        """CH-1113: posicion_wa de las verbales afirmativas cerca de sus pesos (±0,05)."""
        vals = pd.Series([I["posicion_wa"] for I in self.ideas
                          if I["predicado"] == "verbo" and I["modalidad"] == "afirmativa"]).value_counts(normalize=True)
        for k, w in PESOS["posicion_wa"].items():
            self.assertAlmostEqual(vals.get(k, 0), w, delta=0.05, msg=k)

    def test_CH1110_plural_y_CH1107_progresivo(self):
        """CH-1110: cerca del 30 % de sujetos plurales; CH-1107: progresivo solo en SIMP no negativo, algo menos del 15 %."""
        pl = sum(bool(I["sujeto"].get("plural")) for I in self.ideas) / len(self.ideas)
        self.assertTrue(0.20 < pl < 0.36, pl)
        cand = [I for I in self.ideas if I["predicado"] == "verbo" and I["tiempo"] == "SIMP" and I["modalidad"] != "negativa"]
        prog = sum(I.get("aspecto") == "progresivo" for I in cand) / len(cand)
        self.assertTrue(0.05 < prog <= 0.16, prog)
        self.assertFalse(any(I.get("aspecto") == "progresivo" and I.get("tiempo", "SIMP") != "SIMP" for I in self.ideas))


    def test_R22_sin_LV(self):
        """R22: en 10.000 ideas generadas no aparece ningún id LV (ni ningún id fuera del léxico de producción)."""
        usados = {i for I in self.ideas for i in ids_de_idea(I)}
        self.assertFalse([i for i in usados if i.startswith("LV")])
        self.assertTrue(usados <= set(LEX))

    def test_R21_sin_repeticiones(self):
        """R21: ninguna idea repite un lexema."""
        for I in self.ideas:
            u = ids_de_idea(I)
            self.assertEqual(len(u), len(set(u)), I)

    def test_P11_categoria_verbo(self):
        """P-11: la categoría del verbo sale con los pesos de categoria_verbo (±0,03), en las combinaciones que
        aceptan todos los verbos (sujeto de 3.ª persona, afirmativa o pregunta sí/no)."""
        cats = pd.Series([LEX[I["verbo"]]["cat"] for I in self.ideas if I["predicado"] == "verbo"
                          and I["sujeto"]["persona"] == 3 and I["modalidad"] in ("afirmativa", "pregunta_si_no")])
        frec = cats.value_counts(normalize=True)
        for c, w in PESOS["categoria_verbo"].items():
            self.assertAlmostEqual(frec.get(c, 0), w, delta=0.03, msg=c)

    def test_BEN_y_CAUSE(self):
        """CH-406 (v1.1.2): BEN solo con V_TR y V_DITR con admite_ben = s; CAUSE solo con V_MOV y V_INTR."""
        ben = [LEX[I["verbo"]] for I in self.ideas if (I.get("oi") or {}).get("funcion") == "BEN"]
        self.assertTrue(ben)
        self.assertTrue(all(v["cat"] in ("V_TR", "V_DITR") and v["admite_ben"] == "s" for v in ben))
        causa = {LEX[I["verbo"]]["cat"] for I in self.ideas if (I.get("comcausa") or {}).get("caso") == "CAUSE"}
        self.assertTrue(causa)
        self.assertTrue(causa <= {"V_MOV", "V_INTR"}, causa)

    def test_lugar_sin_numeral(self):
        """v1.1.2: ninguna frase de lugar (ni el lugar de "estar") lleva numeral."""
        for I in self.ideas:
            for np in ((I.get("lugar") or {}).get("np"), (I.get("pnom") or {}).get("np") if I["predicado"] == "estar" else None):
                self.assertFalse((np or {}).get("num"), I)

    def test_P11_sujeto_clase_humana(self):
        """P-11 (v1.1.2): si el verbo acepta persona y otras clases, el sujeto nombre es N_HUM con p ≈ 0,60 (±0,05)."""
        hum = [LEX[I["sujeto"]["np"]["n"]]["cat"] == "N_HUM" for I in self.ideas
               if I["predicado"] == "verbo" and I["sujeto"].get("expreso") == "nombre"
               and len(clases_de(LEX[I["verbo"]], "suj_clases")) > 1 and "persona" in clases_de(LEX[I["verbo"]], "suj_clases")]
        self.assertGreater(len(hum), 300)
        self.assertAlmostEqual(sum(hum) / len(hum), PESOS["sujeto_clase_humana"], delta=0.05)

    def test_P13_COM(self):
        """P-13: COM de compañía solo con V_MOV o V_INTR y sujeto persona; sin instrumento ni compañía, no hay COM."""
        n_compania = 0
        for I in self.ideas:
            C = I.get("comcausa") or {}
            if C.get("caso") != "COM":
                continue
            v = LEX[I["verbo"]]
            instr = clases_de(v, "instr_clases")
            if LEX[C["np"]["n"]]["clase_sem"] not in instr:                    # no es instrumento: es compañía
                n_compania += 1
                S = I["sujeto"]
                quien = I.get("interrogativo") == "quien_suj"                     # khiti es persona (CH-906)
                clase = {"persona"} if quien or S.get("expreso") == "pronombre" or (S.get("expreso", "ninguno") == "ninguno" and S["persona"] != 3) \
                    else {LEX[S["np"]["n"]]["clase_sem"]} if S.get("expreso") == "nombre" else {LEX[x["n"]]["clase_sem"] for x in S.get("coord", [])}
                self.assertIn(v["cat"], ("V_MOV", "V_INTR"), I)
                self.assertEqual(clase, {"persona"}, I)
                self.assertIn(LEX[C["np"]["n"]]["clase_sem"], ("persona", "animal"), I)
        self.assertGreater(n_compania, 0)
        sin_com = [I for I in self.ideas if I["predicado"] == "verbo" and not clases_de(LEX[I["verbo"]], "instr_clases")
                   and LEX[I["verbo"]]["cat"] not in ("V_MOV", "V_INTR")]
        self.assertTrue(sin_com)
        self.assertFalse([I for I in sin_com if (I.get("comcausa") or {}).get("caso") == "COM"])


correr_pruebas(PruebasIdeas, titulo="generador de ideas (CH-1104 a CH-1113)")

## 10. Generación

**Qué hace esta sección.**
1. **Repite todas las pruebas** del notebook (CH-1201, V10). Si una falla, no se genera nada.
2. **Genera las ideas** con las cuotas de la sección 9 (`N_PARES` ideas).
3. **Cobertura mínima** (CH-404). Cuenta cuántas veces aparece cada palabra del léxico. Si alguna aparece menos de 20 veces, genera ideas extra que la incluyan, hasta llegar a 20. Esas ideas se mezclan con las demás.
4. **Construye los pares**: B, A y español de cada idea, en el formato de CH-1301.

**Léxico de ejemplo.** Si no hay `lexico_sin_raices.csv`, el léxico es el de ejemplo de §4.1 (26 palabras). Los pares sirven para **probar el proceso**, no para entrenar. Por eso:
- se guardan en `salida/prueba_lexico_ejemplo/`;
- cada par lleva el campo `uso` = "NO USAR PARA ENTRENAR…";
- la carpeta tiene un `LEEME.txt` con el mismo aviso.

**Tiempo.** Cada paso mide cuánto tarda. El informe de la sección 11 lo recoge.

**`GENERAR`.** Si en la celda de parámetros vale `False`, las secciones 10.2 a 11 no se ejecutan: el notebook solo corre las pruebas, fija las raíces y muestra 30 pares (sección 9.4) para revisarlos.

**Qué entra.** `LEX`, los pesos, `N_PARES`, `SEMILLA`. **Qué sale.** `PARES` (en memoria). Los archivos se escriben en la sección 11, después de verificar.

In [ ]:
# 10.1 · Todas las pruebas antes de generar (CH-1201, V10).
import time
from collections import Counter

TIEMPOS = {}
t0 = time.perf_counter()
CLASES_PRUEBA = [c for c in list(globals().values())
                 if isinstance(c, type) and issubclass(c, unittest.TestCase) and c.__module__ == "__main__"]
correr_pruebas(*CLASES_PRUEBA, titulo=f"todas las pruebas ({len(CLASES_PRUEBA)} grupos)")
TIEMPOS["pruebas"] = time.perf_counter() - t0

In [ ]:
# 10.2 · Ideas y cobertura mínima (CH-1104, CH-404).
if not GENERAR:
    print("GENERAR = False en la celda de parámetros: esta celda no se ejecuta.")
else:
    MIN_COBERTURA = 20                       # CH-404
    MAX_INTENTOS_COBERTURA = 200             # por idea extra; si se agotan, la entrada queda en el informe (V29)
    LEXICO_DE_EJEMPLO = not LEXICO_PRODUCCION
    AVISO_EJEMPLO = "NO USAR PARA ENTRENAR: generado con el léxico de ejemplo de §4.1 (prueba del proceso)."


    def cobertura(ideas):
        """Cuántas ideas usan cada entrada del léxico.

        Parámetros:
            ideas: lista de ideas.
        Devuelve:
            Counter {id: número de ideas}.
        Reglas: CH-404.
        Ejemplo:
            >>> cobertura([IDEA_CH1101])["L0018"]
            1
        """
        return Counter(i for I in ideas for i in set(ids_de_idea(I)))


    def clave_idea(I):
        """Texto que identifica una idea sin su id (para encontrar duplicados, V27)."""
        return json.dumps({k: v for k, v in I.items() if k != "id"}, sort_keys=True, ensure_ascii=False)


    def completar_cobertura(ideas, lex, tabla, rng, minimo=MIN_COBERTURA):
        """Genera ideas extra para las entradas que aparecen menos de `minimo` veces.

        Para cada entrada que falta, el generador la elige siempre que cabe en una función (FORZAR),
        y se acepta la idea solo si es válida, la contiene y no repite otra idea (los duplicados se
        descartan después, V27, así que no cuentan para la cobertura). El resto se sortea como siempre.

        Parámetros:
            ideas: ideas ya generadas. lex: léxico. tabla: tabla de cuotas de generar_ideas(). rng: random.Random.
        Devuelve:
            (ideas extra, {id: cuántas faltaron}).
        Reglas: CH-404, CH-406 (las ideas extra pasan el validador).
        """
        vistas = {}
        for I in ideas:
            vistas.setdefault(clave_idea(I), I)
        cuenta = cobertura(vistas.values())                    # solo ideas únicas
        T = tabla[tabla["posible"]]
        combos = [tuple(x) for x in T[["predicado", "modalidad", "tiempo", "persona"]].itertuples(index=False)]
        extra, faltan = [], {}
        try:
            for i in sorted(lex):
                falta, intentos = minimo - cuenta[i], 0
                FORZAR["id"] = i
                FORZAR["verbos"] = VERBOS_PARA.get(i)                 # R23: solo cabe con ciertos verbos
                cs = [c for c in combos if c[0] == "verbo"] if FORZAR["verbos"] else combos
                ws = [w for c, w in zip(combos, T["peso"].tolist()) if c in cs]
                while falta > 0 and intentos < MAX_INTENTOS_COBERTURA * minimo:
                    intentos += 1
                    I = idea_al_azar(rng, lex, rng.choices(cs, weights=ws)[0])
                    if I is not None and i in ids_de_idea(I) and clave_idea(I) not in vistas and not validar_idea(I, lex):
                        vistas[clave_idea(I)] = I
                        extra.append(I)
                        falta -= 1
                if falta > 0:
                    faltan[i] = falta
        finally:
            FORZAR["id"], FORZAR["verbos"] = None, None
        return extra, faltan


    t0 = time.perf_counter()
    IDEAS, TABLA_CUOTAS = generar_ideas(N_PARES, LEX, SEMILLA)
    TIEMPOS["ideas"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    rng_cob = random.Random(SEMILLA + 404)
    EXTRA, SIN_COBERTURA = completar_cobertura(IDEAS, LEX, TABLA_CUOTAS, rng_cob)
    IDEAS = [{k: v for k, v in I.items() if k != "id"} for I in IDEAS + EXTRA]
    rng_cob.shuffle(IDEAS)                                          # CH-404: las ideas extra se mezclan con el resto
    IDEAS = [{"id": f"I{k:07d}", **I} for k, I in enumerate(IDEAS, 1)]
    TIEMPOS["cobertura"] = time.perf_counter() - t0
    cuenta = cobertura(IDEAS)
    print(f"{N_PARES:,} ideas por cuotas + {len(EXTRA):,} ideas de cobertura (CH-404) = {len(IDEAS):,} ideas.")
    print(f"Apariciones por entrada: mínimo {min(cuenta[i] for i in LEX):,}, máximo {max(cuenta.values()):,}.")
    if SIN_COBERTURA:
        print(f"AVISO CH-404: {len(SIN_COBERTURA)} entradas no llegan a {MIN_COBERTURA} apariciones: {SIN_COBERTURA}")


In [ ]:
# 10.3 · Pares en el formato de CH-1301.
if not GENERAR:
    print("GENERAR = False en la celda de parámetros: esta celda no se ejecuta.")
else:
    def hacer_par(k, I, lex):
        """Construye un par (CH-1301) y su detalle interno para la verificación.

        Parámetros:
            k: número de par (1, 2…). I: la idea. lex: léxico.
        Devuelve:
            (par, detalle). par: el diccionario que se guarda. detalle: la salida completa de generar().
        Reglas: CH-1301.
        """
        r = generar(I, lex)
        par = {"id": f"P{k:08d}", "version_spec": VERSION_SPEC,
               "version_lexico": "ejemplo-§4.1" if LEXICO_DE_EJEMPLO else VERSION_LEXICO,
               "idea": I, "b": r["b"], "a": r["a"], "es": r["es"],
               "palabras_b": [[p["base"], p["ids"]] for p in r["palabras"]], "dv": r["dv"]}
        if LEXICO_DE_EJEMPLO:
            par["uso"] = AVISO_EJEMPLO
        return par, r


    t0 = time.perf_counter()
    PARES = [hacer_par(k, I, LEX)[0] for k, I in enumerate(IDEAS, 1)]   # el detalle se recalcula en 11.1 (ahorra memoria)
    TIEMPOS["pares"] = time.perf_counter() - t0
    print(f"{len(PARES):,} pares construidos.")
    print("Tiempos (s):", {k: round(v, 1) for k, v in TIEMPOS.items()})


## 11. Verificación y archivos

**Qué hace esta sección.** Revisa **cada par** con la lista de §13 (V01 a V31) y escribe los archivos.

**Acciones** (CH-1302):
- **E** (error del programa): el notebook se detiene y no escribe los pares. Hay un fallo en el código que hay que corregir.
- **D** (descartar): el par se quita y se cuenta en el informe.
- **A** (aviso): el par se queda, y el informe lo explica.

**Grupos de verificaciones:** formato (V01–V03), ortografía (V04–V07), morfología (V08–V10), modo A (V11–V13), sintaxis (V14–V18), idea (V19–V21), español (V22–V25) y conjunto completo (V26–V31).

**Archivos** (en `salida/prueba_lexico_ejemplo/` con el léxico de ejemplo, o en `salida/pares/` con el de producción):
- `pares.jsonl`: un par por línea (CH-1301).
- `informe.md`: resultado de cada verificación, distribuciones, cobertura, tiempos y avisos pendientes.
- `LEEME.txt`: solo con el léxico de ejemplo. Dice que estos pares no se usan para entrenar.

**Ejemplo de fallo** (V15): `Yaruqhiwa wirkharuwa tisqani.` tiene dos `-wa` → error E.

In [ ]:
# 11.1 · Verificaciones de cada par (CH-1302, V01–V25).
if not GENERAR:
    print("GENERAR = False en la celda de parámetros: esta celda no se ejecuta.")
else:
    CAMPOS_1301 = ["id", "version_spec", "version_lexico", "idea", "b", "a", "es", "palabras_b", "dv"]
    RE_TOKEN = re.compile(r"[^\s¿?.]+")
    RAIZ_DE = {e["raiz"]: i for i, e in LEX.items()}
    FORMAS_GRAM_B = {g["b"] for g in GRAMATICALES.values()}
    PROHIBIDAS_B = {"maya", "paya", "khaya", "mä", "pä", "khä"}                     # CH-518, CH-513
    PRONOMBRES_ES = {"yo", "tú", "él", "ella", "nosotros", "ustedes", "ellos", "ellas"}
    ACCION = {**{f"V{k:02d}": "E" for k in range(1, 32)}, "V19": "D", "V20": "D", "V27": "D", "V28": "A", "V29": "A", "V30": "A"}

    # V25: género y número de cada forma española (nombres, adjetivos y determinantes).
    def _gn(lex, cats):
        out = {}
        for e in lex.values():
            if e["cat"] in cats:
                g_fijo = e["genero"] if e["genero"] in ("m", "f") else None
                for col, g, pl in (("es_m", "m", False), ("es_f", "f", False), ("es_pl_m", "m", True), ("es_pl_f", "f", True)):
                    if e.get(col):
                        out.setdefault(e[col], set()).add((g_fijo or g, pl))
        return out


    GN_NOMBRE, GN_ADJ = _gn(LEX, NOMBRES), _gn(LEX, ["ADJ"])
    GN_DET = {}
    for formas, gs in [(("el", "un", "este", "ese", "aquel", "nuestro", "al", "del"), [("m", False)]),
                       (("la", "una", "esta", "esa", "aquella", "nuestra"), [("f", False)]),
                       (("los", "estos", "esos", "aquellos", "nuestros"), [("m", True)]),
                       (("las", "estas", "esas", "aquellas", "nuestras"), [("f", True)]),
                       (("mi", "tu", "su"), [("m", False), ("f", False)]),
                       (("mis", "tus", "sus", *NUM_ES.values()), [("m", True), ("f", True)])]:
        for f in formas:
            GN_DET[f] = set(gs)


    ART_EL_FORMAS = {e["es_m"] for e in LEX.values() if "ART_EL" in (e.get("notas") or "")}
    ADJ_RAICES = {e["raiz"] for e in LEX.values() if e["cat"] == "ADJ"}
    MODIFICADORES_B = set(DEM_B.values()) | set(NUM_B.values())


    def constituyentes(P, I, lex):
        """Constituyente de cada palabra, deducido de sus ids y de su base (sin la anotación del constructor).

        Parámetros:
            P: palabras de la oración B (base, ids). I: la idea. lex: léxico.
        Devuelve:
            Lista con S, JANI, T, L, COMCAUSE, OI, OD o V para cada palabra.
        Reglas: CH-901 (lo usan V14 y V15).
        """
        pred, S = I["predicado"], I["sujeto"]
        raiz_v = lex[I["verbo"]]["raiz"] if pred == "verbo" else None
        adv = lex[I["adv_t"]]["raiz"] if I.get("adv_t") else None
        com = [k for k, p in enumerate(P) if "COM" in p["ids"]]
        coord_s = set(com[:2]) if S.get("expreso") == "coordinado" else set()
        coord_od = set(com[-2:]) if I.get("od_coord") else set()
        out = [None] * len(P)
        for k, p in enumerate(P):
            ids, b = set(p["ids"]), p["base"]
            if k == len(P) - 1 and (pred != "verbo" or b == raiz_v):
                out[k] = "V"
            elif b == B_DE("NEG.PAL"):
                out[k] = "JANI"
            elif b == adv and ids <= {"TOP"}:
                out[k] = "T"
            elif k in coord_s:
                out[k] = "S"
            elif k in coord_od:
                out[k] = "OD"
            elif "GEN" in ids or (not ids and (b in ADJ_RAICES or b in MODIFICADORES_B)):
                continue                                              # modificador: es del núcleo que sigue
            elif ids & {"ALL", "ABL", "LOC", "UNTIL"}:
                out[k] = "L"
            elif ids & {"COM", "CAUSE"}:
                out[k] = "COMCAUSE"
            elif ids & {"COMPL.RU", "BEN"}:
                out[k] = "OI"
            elif ids & {"TOP", "AFF"} or (b == B_DE("INT.QUIEN") and "Q.INF" in ids):
                out[k] = "S"
            else:
                out[k] = "OD"                                         # nombre sin caso (o kuna-sa)
        for k in range(len(P) - 2, -1, -1):
            if out[k] is None:
                out[k] = out[k + 1]
        return out


    def wa_esperado(I):
        """Dónde debe ir -wa según CH-902, calculado aparte del constructor. Devuelve un conjunto de constituyentes."""
        mod, S = I["modalidad"], I["sujeto"]
        if mod == "negativa":
            return {"JANI"}
        if mod != "afirmativa":
            return set()
        if I["predicado"] != "verbo":
            return {"V"}
        pw = I.get("posicion_wa", "verbo")
        if pw == "ninguna":
            return set()
        if pw == "sujeto":
            return {"S"} if S.get("expreso", "ninguno") != "ninguno" and I.get("interrogativo") != "quien_suj" else {"V"}
        if pw == "complemento":
            Lg = I.get("lugar") or {}
            hay = [("L", Lg.get("caso") in ("ALL", "UNTIL") and bool(Lg.get("np"))), ("COMCAUSE", bool(I.get("comcausa"))),
                   ("OI", bool(I.get("oi")))]
            cerca = [c for c, si in hay if si]
            return {cerca[-1]} if cerca else {"V"}               # el más cercano al verbo (CH-901: L, COMCAUSE, OI)
        return {"V"}


    def tokens(texto):
        """Palabras de una oración, en minúsculas y sin ¿ ? ."""
        return RE_TOKEN.findall(texto.lower())


    def verificar_par(par, r):
        """Revisa un par con V01 a V25 (las de cada par).

        Parámetros:
            par: el par (CH-1301). r: la salida completa de generar() para su idea.
        Devuelve:
            Lista de (V, mensaje). Vacía si el par está bien.
        Reglas: CH-1302 (V01–V25). V10 se hizo antes de generar (sección 10.1).
        """
        F = []
        mal = lambda v, msg: F.append((v, msg))
        I, b, a, es = par.get("idea"), par.get("b"), par.get("a"), par.get("es")
        if any(c not in par for c in CAMPOS_1301):                                            # V01
            return [("V01", f"faltan campos {[c for c in CAMPOS_1301 if c not in par]}")]
        pregunta = I["modalidad"] in ("pregunta_si_no", "pregunta_info")
        for nombre, s in (("b", b), ("a", a), ("es", es)):
            if not unicodedata.is_normalized("NFC", s) or "  " in s or any(x in s for x in ALF["normalizacion"]["apostrofos_a_convertir"]):
                mal("V02", f"{nombre}: no está en NFC, tiene dobles espacios o un apóstrofo tipográfico")
            if not (s.startswith("¿") and s.endswith("?") if pregunta else (s.endswith(".") and not s.startswith("¿"))) \
                    or not s.lstrip("¿")[:1].isupper():
                mal("V03", f"{nombre}: mayúscula o puntuación incorrecta: {s!r}")
        for w in tokens(b) + tokens(a):                                                       # V04, V05
            for e in validar_palabra(w):
                mal("V04" if "CH-104" in e else "V05", f"{w}: {e}")
        P, PA = r["palabras"], r["palabras_a"]
        for p in P:
            if p["base"] in RAIZ_DE:                                                          # V06
                if validar_raiz(p["base"]):
                    mal("V06", f"raíz {p['base']}: {validar_raiz(p['base'])}")
            elif p["base"] not in FORMAS_GRAM_B:
                mal("V06", f"raíz desconocida {p['base']!r}")
            es_verbo = p["const"] == "V" and I["predicado"] == "verbo"
            cola = p["forma"][len(p["base"]) - 1:]
            if p["forma"] in PROHIBIDAS_B or "ŋ" in p["forma"] or (es_verbo and (p["forma"].endswith("tayna") or "taya" in cola)):
                mal("V07", f"forma prohibida {p['forma']!r}")                                 # V07
            err = errores_ids_verbo(p["ids"]) if es_verbo else errores_ids_nombre(p["ids"], p["base"])
            if err:
                mal("V08", f"{p['base']} + {p['ids']}: {err}")                                # V08
            if construir_palabra(p["base"], p["ids"], "B") != p["forma"]:
                mal("V09", f"{p['base']} + {p['ids']} no da {p['forma']}")                    # V09
        if " ".join(p["forma"] for p in P) != " ".join(tokens(b)):
            mal("V09", "las palabras no reconstruyen la oración B")
        if [[p["base"], p["ids"]] for p in PA] != par["palabras_b"]:                         # V11
            mal("V11", "A no tiene las mismas bases e ids que B")
        if " ".join(construir_palabra(x, ids, "A") for x, ids in par["palabras_b"]) != " ".join(tokens(a)):
            mal("V12", "A no es la reconstrucción en modo A de palabras_b")                   # V12
        for pb, pa in zip(P, PA):                                                             # V13, morfema a morfema
            cambia = pb["base"] in FORMAS_GRAM_B or any(MORFEMAS[i]["b"] != MORFEMAS[i]["a"] for i in pb["ids"])
            if cambia and pb["forma"] == pa["forma"]:                 # FUT.1 y VBLZ.LONG son "¨" en los dos modos (CH-513)
                mal("V13", f"{pa['forma']}: la forma A es igual a la B")
            if pb["base"] in FORMAS_GRAM_B and pb["base"] != GRAM_B_A[pb["base"]] and pa["forma"].startswith(pb["base"]):
                mal("V13", f"{pa['forma']}: la palabra gramatical sigue en su forma B ({pb['base']})")
            if pb["ids"]:
                m = MORFEMAS[pb["ids"][-1]]                           # el último sufijo se escribe entero (CH-706)
                fb, fa = m["b"].lstrip("¨"), m["a"].lstrip("¨")
                if fb and fb != fa and not fa.endswith(fb) and pa["forma"].endswith(fb):
                    mal("V13", f"{pa['forma']}: termina en la forma B -{fb} de {pb['ids'][-1]}")
        const = constituyentes(P, I, LEX)                                                    # V14 (deducido de los ids)
        orden = [ORDEN_CONSTITUYENTES.index(c) for c in const]
        if orden != sorted(orden):
            mal("V14", f"orden de constituyentes {const}")
        if const != [p["const"] for p in P]:
            mal("V14", f"constituyentes {const} distintos de los del constructor {[p['const'] for p in P]}")
        S = I["sujeto"]
        expreso, inter = S.get("expreso", "ninguno"), I.get("interrogativo")
        suj_expreso = expreso != "ninguno" and inter != "quien_suj"
        aff = [k for k, p in enumerate(P) if "AFF" in p["ids"]]                                # V15 (CH-902, aparte del constructor)
        esperado = wa_esperado(I)
        if len(aff) > 1 or {const[k] for k in aff} != esperado:
            mal("V15", f"-wa en {[const[k] for k in aff]}; se esperaba {sorted(esperado)}")
        if any(const[k] in ("OD", "T") or (const[k] == "L" and ({"LOC", "ABL"} & set(P[k]["ids"]))) for k in aff):
            mal("V15", "-wa en OD, ABL, LOC, adverbio o deíctico")
        hay_s = "S" in const                                                                   # V16 (CH-903)
        top = [k for k, p in enumerate(P) if "TOP" in p["ids"]]
        if any(not (const[k] == "S" or (const[k] == "T" and not hay_s)) for k in top):
            mal("V16", "-xa fuera del sujeto (o del adverbio sin sujeto)")
        if expreso in ("pronombre", "nombre", "demostrativo") and inter != "quien_suj" and \
                not any(const[k] == "S" for k in top) and not any(const[k] == "S" for k in aff):
            mal("V16", "el sujeto expreso no lleva -xa (ni -wa)")
        if I.get("adv_t") and not hay_s and not any(const[k] == "T" for k in top):
            mal("V16", "sin sujeto expreso, el adverbio de tiempo debe llevar -xa (CH-903.2)")
        verbo = [p for p in P if p["const"] == "V"]
        neg = I["modalidad"] == "negativa"                                                    # V17
        jani = any(p["base"] == B_DE("NEG.PAL") and "AFF" in p["ids"] for p in P)
        if neg != jani or (I["predicado"] == "verbo" and neg != ("NEG" in verbo[-1]["ids"] and "NEG.TI" in verbo[-1]["ids"])):
            mal("V17", "negativa ⇔ jani-wa + NEG + NEG.TI")
        if any("Q.YN" in p["ids"] and p["const"] != "V" for p in P):
            mal("V17", "Q.YN fuera del verbo o del predicado")
        qinf = [p for p in P if "Q.INF" in p["ids"]]
        if len(qinf) != (1 if I["modalidad"] == "pregunta_info" else 0) or any(p["base"] not in INTERROGATIVOS_B for p in qinf):
            mal("V17", "Q.INF: uno solo, en un interrogativo, solo en pregunta_info")
        for k, campo in (("S", S.get("coord")), ("OD", I.get("od_coord"))):                   # V18
            if campo and sum("COM" in p["ids"] for p in P if p["const"] == k) != 2:
                mal("V18", f"coordinación en {k} sin 2 miembros con COM")
        if (S.get("coord") or I.get("od_coord")) and (I.get("comcausa") or {}).get("caso") == "COM":
            mal("V18", "coordinación con un COM aparte")
        err = validar_idea(I, LEX)                                                            # V19, V20
        if err:
            mal("V20" if all("R20" in e for e in err) else "V19", "; ".join(err))
        pl = bool(S.get("plural"))                                                            # V21
        # marca de plural en el sujeto: PL (no la del poseedor con GEN), nänaka, un numeral o la coordinación
        marca = any(p["const"] == "S" and "GEN" not in p["ids"] and ("PL" in p["ids"] or p["base"] in (B_DE("PRON.1.PL"), *NUM_B.values()))
                    for p in P) or expreso == "coordinado"
        plv = I["predicado"] == "verbo" and "PL.V" in verbo[-1]["ids"]
        if plv != (pl and I["predicado"] == "verbo" and not neg) or marca != (pl and expreso != "ninguno"):
            mal("V21", "plural del sujeto mal marcado (CH-913)")
        if es != oracion_espanol(I, LEX):                                                     # V22
            mal("V22", "el español no es el reconstruido desde la idea")
        te = tokens(es)
        if I["predicado"] == "verbo":                                                         # V23
            sp, spl = (3, False) if inter == "quien_suj" else (S["persona"], pl or expreso == "coordinado")
            h, c, t = hueco(sp, spl), CONJUGACIONES[LEX[I["verbo"]]["es_m"]], I.get("tiempo", "SIMP")
            adv = I.get("adv_t")
            if I.get("aspecto") == "progresivo":
                vf = f"{ESTAR[h]} {c['ger']}"
            elif t == "SIMP":
                vf = c["pret" if adv and LEX[adv]["subcat"].startswith("pasado") else "pres"].split()[h]
            elif t == "FUT":
                vf = c["fut"].split()[h]
            else:
                vf = f"{HABER['pres' if t == 'REM.C' else 'imp'][h]} {c['part']}"
            if f" {vf} " not in f" {' '.join(te)} ":
                mal("V23", f"se esperaba la forma verbal '{vf}'")
        if (expreso == "pronombre" and inter != "quien_suj") != any(w in PRONOMBRES_ES for w in te):
            mal("V24", "pronombre español ⇔ pronombre aymara")                                # V24
        # V25 solo mira los nombres y adjetivos de la idea: "pruebas" puede ser verbo y "café" adjetivo;
        # en "té de coca dorado" el núcleo es "té", no "coca" (nombres de varias palabras).
        usados = set(ids_de_idea(I))
        formas = lambda cats: {LEX[x][c] for x in usados if LEX[x]["cat"] in cats
                               for c in ("es_m", "es_f", "es_pl_m", "es_pl_f") if LEX[x].get(c)}
        nombres_idea, adjs_idea = formas(NOMBRES), formas(["ADJ"])
        for w1, w2 in zip(te, te[1:]):                                                        # V25
            if w1 in GN_DET and w2 in nombres_idea and not GN_DET[w1] & GN_NOMBRE[w2] \
                    and not (w1 in ("el", "un", "al", "del") and w2 in ART_EL_FORMAS):       # CH-1008.9: el agua
                mal("V25", f"'{w1} {w2}' no concuerda")
            if w1 in nombres_idea and w2 in adjs_idea and not GN_NOMBRE[w1] & GN_ADJ[w2]:
                mal("V25", f"'{w1} {w2}' no concuerda")
        OI = I.get("oi") or {}
        if I["predicado"] == "verbo" and I.get("objeto_persona", 3) == 3 and (OI.get("funcion") == "COMPL.RU" or inter == "quien_oi"):
            n_oi = OI.get("np") or {}
            cl, otro = ("les", "le") if n_oi.get("plural") or n_oi.get("num") else ("le", "les")
            if cl not in te or otro in te:
                mal("V25", f"COMPL.RU: se esperaba el clítico '{cl}' (CH-1009)")
        P_ = I.get("pnom") or {}
        if I["predicado"] == "ser" and P_.get("adj") and expreso != "demostrativo":
            e = LEX[P_["adj"]]
            if expreso == "nombre":
                fila = LEX[S["np"]["n"]]
                g = (S["np"].get("genero") or "m") if fila["genero"] == "var" else fila["genero"]
            else:
                g = S.get("genero") or "m"
            pl_es = pl or S.get("persona") == 4                     # "nosotros" es plural en español (dev/Camilo)
            forma = e[{("m", False): "es_m", ("f", False): "es_f", ("m", True): "es_pl_m", ("f", True): "es_pl_f"}[(g, pl_es)]]
            if forma not in te:
                mal("V25", f"el adjetivo de 'ser' debe ser '{forma}' (CH-405)")
        return F


    t0 = time.perf_counter()
    FALLOS = {}                                         # id del par → [(V, mensaje)]
    for par in PARES:
        r = generar(par["idea"], LEX)                  # detalle completo (no se guarda en 10.3: memoria)
        f = verificar_par(par, r)
        if f:
            FALLOS[par["id"]] = f
    TIEMPOS["verificación por par"] = time.perf_counter() - t0
    cuenta_v = Counter(v for f in FALLOS.values() for v, _ in f)
    print(f"Verificación por par: {len(PARES):,} pares, {len(FALLOS):,} con algún fallo, en {TIEMPOS['verificación por par']:.1f} s.")
    for v, n in sorted(cuenta_v.items()):
        ejemplo = next(f"{pid}: {m}" for pid, f in FALLOS.items() for vv, m in f if vv == v)
        print(f"  {v} ({ACCION[v]}): {n:,} · p. ej. {ejemplo[:200]}")


In [ ]:
# 11.2 · Verificaciones del conjunto (V26–V31) y decisión.
if not GENERAR:
    print("GENERAR = False en la celda de parámetros: esta celda no se ejecuta.")
else:
    t0 = time.perf_counter()
    DESCARTADOS = {pid for pid, f in FALLOS.items() if all(ACCION[v] == "D" for v, _ in f)}
    ERRORES = {pid: f for pid, f in FALLOS.items() if any(ACCION[v] == "E" for v, _ in f)}
    CONJUNTO = {}                                        # V → (n, detalle)


    PARES_GENERO = {frozenset(p) for p in [("el", "la"), ("los", "las"), ("un", "una"), ("unos", "unas"), ("este", "esta"),
                                            ("estos", "estas"), ("ese", "esa"), ("esos", "esas"), ("aquel", "aquella"),
                                            ("aquellos", "aquellas"), ("nuestro", "nuestra"), ("nuestros", "nuestras"),
                                            ("él", "ella"), ("ellos", "ellas")]}
    for e in list(LEX.values()) + list(LEX_PRUEBA.values()):
        for m_, f_ in (("es_m", "es_f"), ("es_pl_m", "es_pl_f")):
            if e.get(m_) and e.get(f_) and e[m_] != e[f_]:
                PARES_GENERO.add(frozenset((e[m_].lower(), e[f_].lower())))


    def solo_genero(x, y):
        """CH-1013: dos españoles solo difieren en palabras de género (artículo, pronombre, nombre o adjetivo en m/f).

        Parámetros:
            x, y: dos oraciones españolas.
        Devuelve:
            True si cada palabra distinta forma un par m/f conocido (el/la, niño/niña, rojos/rojas…), con el mismo número.
        Reglas: CH-1013.
        Ejemplo:
            >>> solo_genero("La casa del niño.", "La casa de la niña.")
            True
        """
        partir = lambda s: (" " + s.lower()).replace("¿", " ").replace("?", " ").replace(".", " ") \
            .replace(" del ", " de el ").replace(" al ", " a el ").split()
        tx, ty = partir(x), partir(y)
        return len(tx) == len(ty) and all(p == q or frozenset((p, q)) in PARES_GENERO for p, q in zip(tx, ty))


    assert solo_genero("Él es niño.", "Ella es niña.") and solo_genero("¿Ustedes son hermanos?", "¿Ustedes son hermanas?")
    assert solo_genero("Veo la casa del niño.", "Veo la casa de la niña.") and not solo_genero("Él duerme.", "Él dormía.")
    assert not solo_genero("Él corta pan.", "Él cortó pan.") and not solo_genero("Él ve el perro.", "Él ve los perros.")
    assert not solo_genero("Él fue.", "Él fui.")
    grupos = {}                                                                              # V26
    for par in PARES:
        for modo in ("b", "a"):
            grupos.setdefault((modo, par[modo]), set()).add(par["es"])
    malos = [(k, sorted(v)) for k, v in grupos.items() if any(not solo_genero(x, y) for x in v for y in v)]
    CONJUNTO["V26"] = (len(malos), malos[:3])

    vistos, dup = set(), []                                                                  # V27
    for par in PARES:
        clave = clave_idea(par["idea"])
        if clave in vistos:
            dup.append(par["id"])
        vistos.add(clave)
    CONJUNTO["V27"] = (len(dup), dup[:3])
    DESCARTADOS |= set(dup)

    FINALES = [p for p in PARES if p["id"] not in DESCARTADOS and p["id"] not in ERRORES]
    ID_FINAL = {p["id"] for p in FINALES}


    def frecuencias(valores):
        s = pd.Series(list(valores))
        return s.value_counts(normalize=True).to_dict() if len(s) else {}


    def comparar(nombre, valores, pesos):
        """V28: compara una distribución con sus pesos. Devuelve filas para la tabla del informe."""
        obs, total = frecuencias(valores), sum(pesos.values())
        return [{"marginal": nombre, "valor": str(k), "peso": w / total, "observado": obs.get(k, 0.0),
                 "desvío relativo": abs(obs.get(k, 0.0) - w / total) / (w / total)} for k, w in pesos.items() if w > 0]


    ID = [p["idea"] for p in FINALES]                                                       # V28
    verb = [I for I in ID if I["predicado"] == "verbo"]
    T = TABLA_CUOTAS[TABLA_CUOTAS["posible"]]
    filas = comparar("predicado.modalidad", (f"{I['predicado']}.{I['modalidad']}" for I in ID),
                     T.assign(pm=T["predicado"] + "." + T["modalidad"]).groupby("pm")["peso"].sum().to_dict())
    filas += comparar("tiempo (general)", (I["tiempo"] for I in verb if I["modalidad"] != "negativa"), PESOS["tiempo"]["general"])
    filas += comparar("tiempo (negativa)", (I["tiempo"] for I in verb if I["modalidad"] == "negativa"), PESOS["tiempo"]["negativa"])
    for t, pesos in PESOS["persona_sujeto"].items():
        sel = [I for I in ID if (I.get("tiempo", "SIMP") == t if t != "general" else I.get("tiempo", "SIMP") in ("SIMP", "FUT"))]
        filas += comparar(f"persona ({t})", (I["sujeto"]["persona"] for I in sel), {int(k): w for k, w in pesos.items()})
    filas += comparar("posicion_wa", (I["posicion_wa"] for I in verb if I["modalidad"] == "afirmativa"), PESOS["posicion_wa"])
    filas += comparar("categoria_verbo", (LEX[I["verbo"]]["cat"] for I in verb), PESOS["categoria_verbo"])
    filas += comparar("sujeto plural", (bool(I["sujeto"].get("plural")) for I in ID),
                      {True: PESOS["sujeto"]["plural"], False: 1 - PESOS["sujeto"]["plural"]})
    DISTRIBUCION = pd.DataFrame(filas)
    fuera = DISTRIBUCION[DISTRIBUCION["desvío relativo"] > 0.10]
    CONJUNTO["V28"] = (len(fuera), fuera[["marginal", "valor"]].values.tolist()[:10])

    cuenta = cobertura(ID)                                                                   # V29
    pocas = {i: cuenta[i] for i in LEX if cuenta[i] < MIN_COBERTURA}
    CONJUNTO["V29"] = (len(pocas), pocas)

    dv = [p["dv"] for p in FINALES]                                                          # V30
    DV_PROP = {"alguna DV": sum(bool(x) for x in dv) / len(dv),
               **{k: v / len(dv) for k, v in sorted(Counter(d for x in dv for d in x).items())}}
    CONJUNTO["V30"] = (int(DV_PROP["alguna DV"] > 0.50) + sum(v > 0.15 for k, v in DV_PROP.items() if k != "alguna DV"),
                       {k: round(v, 3) for k, v in DV_PROP.items() if (k == "alguna DV" and v > 0.5) or (k != "alguna DV" and v > 0.15)})

    choques = {e["raiz"]: motivo_real(e["raiz"]) for e in LEX.values() if FILTRO_REAL_ACTIVO and motivo_real(e["raiz"])}   # V31
    CONJUNTO["V31"] = (len(choques), choques)
    TIEMPOS["verificación del conjunto"] = time.perf_counter() - t0

    print(f"Pares: {len(PARES):,} · descartados (D): {len(DESCARTADOS):,} · con error (E): {len(ERRORES):,} · finales: {len(FINALES):,}")
    for v, (n, det) in CONJUNTO.items():
        print(f"  {v} ({ACCION[v]}): {'bien' if n == 0 else f'{n} casos · {str(det)[:300]}'}")
    if ERRORES or CONJUNTO["V26"][0] or CONJUNTO["V31"][0]:
        raise RuntimeError("CH-1302: hay errores E (ver arriba). No se escriben los pares: hay que corregir el programa.")
    print("Sin errores E: se pueden escribir los archivos.")


In [ ]:
# 11.3 · Archivos: pares.jsonl, informe.md y (con el léxico de ejemplo) LEEME.txt.
if not GENERAR:
    print("GENERAR = False en la celda de parámetros: esta celda no se ejecuta.")
else:
    t0 = time.perf_counter()
    CARPETA_PARES = CARPETA_SALIDA / ("prueba_lexico_ejemplo" if LEXICO_DE_EJEMPLO else "pares")
    CARPETA_PARES.mkdir(parents=True, exist_ok=True)
    with open(CARPETA_PARES / "pares.jsonl", "w", encoding="utf-8") as f:
        for par in FINALES:
            f.write(json.dumps(par, ensure_ascii=False) + "\n")
    if LEXICO_DE_EJEMPLO:
        (CARPETA_PARES / "LEEME.txt").write_text(
            AVISO_EJEMPLO + "\n\nEstos pares prueban que el proceso completo funciona (generación, cobertura y verificación).\n"
            "El léxico de ejemplo solo tiene 26 palabras: los pares se repiten mucho y no sirven para pre-entrenar.\n"
            "Para datos de entrenamiento, pon lexico_sin_raices.csv en datos/ y vuelve a ejecutar el notebook.\n", encoding="utf-8")
    TIEMPOS["escritura"] = time.perf_counter() - t0

    estado = lambda v: "bien" if v not in cuenta_v and v not in CONJUNTO else \
        (f"{cuenta_v[v]:,} pares" if v in cuenta_v else ("bien" if CONJUNTO[v][0] == 0 else f"{CONJUNTO[v][0]} casos"))
    QUE = {"V01": "campos de CH-1301", "V02": "NFC, apóstrofo, espacios", "V03": "mayúscula y puntuación",
           "V04": "solo letras del alfabeto", "V05": "fonotáctica de cada palabra", "V06": "raíces válidas y del léxico",
           "V07": "formas prohibidas", "V08": "orden de sufijos (§6)", "V09": "reconstrucción en B", "V10": "pruebas antes de generar",
           "V11": "A con las mismas bases e ids", "V12": "reconstrucción en A", "V13": "A sin formas B",
           "V14": "orden de constituyentes", "V15": "-wa", "V16": "-xa", "V17": "negación y preguntas", "V18": "coordinación",
           "V19": "R1–R22", "V20": "compatibilidad semántica", "V21": "plural del sujeto", "V22": "español desde la idea",
           "V23": "tiempo español", "V24": "pronombre español", "V25": "concordancia", "V26": "una traducción por oración",
           "V27": "sin duplicados", "V28": "distribución ±10 %", "V29": "cobertura ≥ 20", "V30": "reglas DV", "V31": "raíces frente a palabras reales"}
    lineas = [f"# Informe de generación", ""]
    if LEXICO_DE_EJEMPLO:
        lineas += [f"**{AVISO_EJEMPLO}**", ""]
    lineas += [f"- Especificación: {VERSION_SPEC} · léxico: {PARES[0]['version_lexico']} ({len(LEX)} entradas) · semilla: {SEMILLA}",
               f"- Pedidos: {N_PARES:,} · ideas de cobertura (CH-404): {len(EXTRA):,} · generados: {len(PARES):,}",
               f"- Descartados (D): {len(DESCARTADOS):,} · finales en `pares.jsonl`: {len(FINALES):,}",
               f"- Tiempo total: {sum(TIEMPOS.values()):.1f} s ({', '.join(f'{k} {v:.1f} s' for k, v in TIEMPOS.items())})", "",
               "## Verificaciones (CH-1302)", "",
               "*Tipo*: **independiente** = se calcula aparte del código que generó el par; **coherencia** = vuelve a ejecutar "
               "ese mismo código (lo pide CH-1302): solo detecta que el resultado no se pueda reproducir.", "",
               "| V | Qué | Acción | Tipo | Resultado |", "|---|---|---|---|---|"]
    COHERENCIA = {"V09", "V10", "V12", "V19", "V20", "V22"}
    lineas += [f"| {v} | {QUE[v]} | {ACCION[v]} | {'coherencia' if v in COHERENCIA else 'independiente'} | "
               f"{'bien (sección 10.1)' if v == 'V10' else estado(v)} |" for v in QUE]
    lineas += ["", "## Distribuciones (V28)", "", "| Marginal | Valor | Peso | Observado | Desvío relativo |", "|---|---|---|---|---|"]
    lineas += [f"| {f.marginal} | {f.valor} | {f.peso:.3f} | {f.observado:.3f} | {f[5]:.0%}{' ⚠' if f[5] > 0.10 else ''} |"
               for f in DISTRIBUCION.itertuples()]
    lineas += ["", "## Reglas derivadas (V30)", "", "| DV | Proporción de pares |", "|---|---|"]
    lineas += [f"| {k} | {v:.1%} |" for k, v in DV_PROP.items()]
    lineas += ["", "## Cobertura (V29, CH-404)", "",
               f"Apariciones por entrada: mínimo {min(cuenta[i] for i in LEX):,}, máximo {max(cuenta[i] for i in LEX):,}. "
               f"Entradas por debajo de {MIN_COBERTURA}: {len(pocas)}.", "",
               "## Pendientes", "",
               f"- Conjugaciones sin revisar (P-07): {N_CONJ_SIN_REVISAR} filas de {len(TABLA_CONJ)}.",
               f"- Filtro de palabras reales (CH-306): {'activo' if FILTRO_REAL_ACTIVO else 'NO activo (falta la lista)'}, "
               f"modo '{FR['modo']}' (P-06 abierta).",
               f"- Léxico: {'de ejemplo (§4.1); falta lexico_sin_raices.csv' if LEXICO_DE_EJEMPLO else 'de producción'}."]
    (CARPETA_PARES / "informe.md").write_text("\n".join(lineas) + "\n", encoding="utf-8")
    print(f"Escritos en {CARPETA_PARES}: pares.jsonl ({len(FINALES):,} pares), informe.md"
          f"{', LEEME.txt' if LEXICO_DE_EJEMPLO else ''}.")
    print("Tiempos (s):", {k: round(v, 1) for k, v in TIEMPOS.items()}, "· total", round(sum(TIEMPOS.values()), 1))
    display(Markdown("\n".join(lineas)))


## 12. Cómo usar los resultados

**Qué hay en la carpeta de salida** (`salida/pares/`, o `salida/prueba_lexico_ejemplo/` con el léxico de ejemplo):

| Archivo | Qué es | Para qué sirve |
|---|---|---|
| `pares.jsonl` | Un par por línea, con la idea, B, A, español, las palabras y las reglas DV (CH-1301) | Es el archivo completo. Sirve para revisar o filtrar pares. |
| `chulupi_modo_B.tsv` | Tres columnas separadas por tabulador: `src` (modo B), `tgt` (español) e `id` del par | Pre-entrenar el traductor aymara → español con sufijos reales del aymara |
| `chulupi_modo_A.tsv` | Las mismas columnas, con el modo A | La versión de control, con sufijos inventados |
| `informe.md` | Resultado de las verificaciones, distribuciones, cobertura y tiempos | Revisar que todo salió bien antes de entrenar |
| `LEEME.txt` | Solo con el léxico de ejemplo | Recuerda que esos pares **no** se usan para entrenar |

Además, el diccionario con sus raíces y su versión (CH-309): `salida/lexico.csv` y `salida/lexico_version.json` en la carpeta simple, o `data/lexicon/lexico_con_raices_vX.csv` y `.json` en el repositorio del equipo.

**En el repositorio del equipo**, guarda cada corrida en su propia carpeta de `data/synthetic/` (por ejemplo `data/synthetic/v1.1.2_100k/`) y no sobrescribas una versión ya publicada.

**Antes de entrenar, revisa:**
1. En `informe.md`, que no haya errores (todas las verificaciones E en "bien").
2. Que `datos/conjugaciones.csv` esté revisado (columna `revisado = s`, P-07).
3. Que los pares no lleven el campo `uso` = "NO USAR PARA ENTRENAR".

**Si algo falla:**

| Mensaje | Qué significa | Qué hacer |
|---|---|---|
| `Faltan archivos en …` | No encuentra la especificación o los casos de prueba | Revisa la carpeta `datos/` (sección 0) |
| `El léxico tiene N errores` | `lexico_sin_raices.csv` no cumple CH-402 | Corrige las filas que se listan |
| `Fallaron N pruebas` | Una regla no da lo que dice la especificación | No generes: avisa al equipo con el detalle |
| `la raíz guardada … ya no pasa los filtros` | Cambió `palabras_reales.txt` o una regla de raíces | Sube `VERSION_LEXICO` para regenerar el léxico |
| `el léxico 1.0 se hizo con modo …` | Cambiaste `MODO_FILTRO` sin cambiar la versión | Vuelve al modo anterior, o sube `VERSION_LEXICO` |
| `hay errores E` | La verificación encontró un fallo del programa | No uses los pares: avisa al equipo con el ejemplo |
| `AVISO P-07` | Hay conjugaciones sin revisar | Revisa `datos/conjugaciones.csv` |

**Para generar otra vez:** cambia los parámetros de la sección 0 (por ejemplo `N_PARES` o `SEMILLA`) y usa *Ejecutar todo*. Las raíces no cambian mientras no cambie `VERSION_LEXICO`.

In [ ]:
# 12.1 · Archivos paralelos para entrenar: chulupi_modo_B.tsv y chulupi_modo_A.tsv (src = aymara, tgt = español, id).
if not GENERAR:
    print("GENERAR = False en la celda de parámetros: no hay pares que exportar.")
else:
    for modo in ("B", "A"):
        ruta_tsv = CARPETA_PARES / f"chulupi_modo_{modo}.tsv"
        with open(ruta_tsv, "w", encoding="utf-8") as f:
            f.write("src\ttgt\tid\n")
            for par in FINALES:
                assert "\t" not in par["es"] + par[modo.lower()], f"{par['id']}: una oración tiene un tabulador"
                f.write(f"{par[modo.lower()]}\t{par['es']}\t{par['id']}\n")
        n = sum(1 for _ in open(ruta_tsv, encoding="utf-8")) - 1
        assert n == len(FINALES), f"{ruta_tsv.name}: {n} pares, se esperaban {len(FINALES)}"
        print(f"Escrito {ruta_tsv} ({n:,} pares).")
    if LEXICO_DE_EJEMPLO:
        print(f"AVISO: {AVISO_EJEMPLO}")
    with pd.option_context("display.max_colwidth", None):
        display(pd.read_csv(CARPETA_PARES / "chulupi_modo_B.tsv", sep="\t", nrows=5, quoting=3))